# eBOSS-DAP Paper 1 — Master Notebook

*Run **Section 0** first every session — it sets up everything else. Sections 2–4 are self-contained and can be run independently after Section 0. Section 5 needs Section 1. Sections 12–22 all depend on the master DataFrame built in Section 11, which itself requires Section 6. The dependency chain for most science figures is: **0 → 1 → 6 → 11 → figure section**.*

*External figures (Figs 12, A1, A2) are generated by the spectrum-plotter tool; see https://github.com/owenmatthewsa/ebossdap/*

# Section 0 — Imports, Style, and Utility Functions
### *The boilerplate nobody reads but everyone needs.*
*All the machinery that has to exist before anything interesting can happen: standard library imports, numerical and astropy tools, plotting configuration, and the target-class colour palette used consistently through every figure. Also defines `plot_prettier` and the `StopExecution` debug hook. Run this every session before anything else — every other section depends on it.*

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
# Path manipulation, file system ops, warnings suppression, and regex.
# zip_longest is used in the line-list table builder (Section 25).
import os, sys, glob, shutil, warnings, json, re
from itertools import zip_longest
from pathlib import Path

# ── Numerical ─────────────────────────────────────────────────────────────
# numpy.ma handles masked arrays in the spectral index and bowtie cells.
# scipy.optimize and curve_fit are used for the Gaussian fits in Sections 6 and 15.
# sp_norm (scipy.stats.norm) provides the theoretical trumpet-plot curves.
# tqdm gives progress bars on the slow KDE loops.
import numpy as np
import numpy.ma as ma
import pandas as pd
import scipy
import scipy.ndimage
import scipy.optimize as opt
from scipy.optimize import curve_fit
from scipy.stats import norm as sp_norm
from tqdm import tqdm

# ── Astropy ───────────────────────────────────────────────────────────────
# SkyCoord + search_around_sky finds repeat-observation pairs in Section 6.
# fits.getdata is the primary interface to the eBOSS-DAP FITS catalogs.
# Cosmology is FlatLambdaCDM with the same H0/Om0 used in the pipeline itself.
import astropy.units as u
import astropy.constants as const
import astropy.coordinates as acoords
from astropy.coordinates import SkyCoord
from astropy.io import fits
from astropy import cosmology as csm
from astropy.table import Table
cosmo = csm.FlatLambdaCDM(H0=69.6, Om0=0.286)

# ── Plotting ──────────────────────────────────────────────────────────────
# mlines and mpatches build the custom legend handles used in Figs 1 and 2,
# where the legend entries are a mix of line and patch styles.
import matplotlib as mpl
import matplotlib.ticker as ticker
import matplotlib.lines as mlines
import matplotlib.patches as mpatches
from scipy.interpolate import interp1d
import matplotlib.patheffects as pe
import matplotlib.colors as mcolors
from matplotlib import pyplot as plt
from matplotlib.cm import ScalarMappable
from matplotlib.colors import LogNorm
import seaborn as sns

# ── Domain-specific ───────────────────────────────────────────────────────
# GCC09_MWAvg is the Gordon et al. 2009 Milky Way average extinction curve,
# used to correct spectra for foreground dust before any flux measurements.
from dust_extinction.averages import GCC09_MWAvg as dext

# ── Settings ──────────────────────────────────────────────────────────────
# Suppress scientific notation in numpy prints and silence routine warnings
# from the KDE and FITS-reading steps that are known and harmless.
np.set_printoptions(suppress=True)
warnings.filterwarnings('ignore')

In [ ]:
def plot_prettier(dpi=300, fontsize=9, figsize=(11, 8.5)):
    """Set global matplotlib rcParams to match the paper figure style.
    Called at the top of each figure section with section-specific dpi/fontsize.
    """
    plt.rc('savefig', dpi=dpi)
    plt.rc('font', size=fontsize)
    plt.rc('xtick', direction='in')
    plt.rc('ytick', direction='in')
    plt.rc('xtick.major', pad=10)
    plt.rc('xtick.minor', pad=5)
    plt.rc('ytick.major', pad=10)
    plt.rc('ytick.minor', pad=5)
    plt.rc('lines', dotted_pattern=[0.5, 1.1])
    plt.rc('figure', figsize=figsize, facecolor='white')

plot_prettier(150, fontsize=18)

# Raising StopExecution instead of a standard exception halts cell execution
# cleanly without printing a traceback — useful for bailing out mid-cell
# during debugging without polluting the output.
class StopExecution(Exception):
    def _render_traceback_(self):
        return []

# ── Target-class colour palette ───────────────────────────────────────────
# Six colours, one per targeting category. Used consistently across every
# figure in the paper so the same class always maps to the same colour.
c_lrg4 = 'magenta'
c_lrg3 = 'xkcd:dark red'
c_lrgl = 'xkcd:burnt orange'
c_elg  = 'green'
c_tdsp = 'xkcd:dark yellow'
c_qso  = 'blue'
c_tot  = 'grey'

# Section 1 — File Paths
### *Tell the notebook where things live before asking it to find them.*
*Sets all catalog paths relative to `WORKING_DIR` — edit that if you're not running from the paper directory. `HAS_SUMMARY` gates any section that depends on the Drake stellar-mass catalog (D. Miller et al. in prep.); if the file isn't present, those sections skip gracefully. Also defines the PMF tag helper functions used throughout, and does a first read of AUXDATA to build `df_downfiber` (kept in memory for Section 5). **Required before:** Sections 5, 6, and anything downstream of Section 11.*

In [ ]:
# ── Directory layout ──────────────────────────────────────────────────────
# All paths below are relative to WORKING_DIR. The expected layout is:
#   data/               — eBOSS-DAP FITS catalogs (emlines, spind, auxdata, summary)
#   linelists/          — .par files used by the emission-line table in Section 25
#   eBOSSDAP/c3k/       — C3K SSP template FITS files for Fig 10
#   templates/ssps/     — d4hda_ckc_noneb*.fit files for the Dn4000/HδA tracks (Fig 24)
#   spectra/plotting/   — the single clipped-spectrum FITS file for Fig 9
#   spectra/test_cneb/  — nebular-continuum DAP test outputs for Fig 11
#   spectra/test_nneb/  — no-nebular-continuum test outputs for Fig 11
#                         (may not yet be present on all machines — see Section 10)
#   outputs/            — created fresh on each run; all figures/CSVs go here
WORKING_DIR = '.'

# Path to the directory containing individual spectrum FITS files for
# the bowtie/pair-histogram figures (Figs 5 & 6). Only needed if
# RUN_SPECTRA_PLOTS = True. Set to None or a non-existent path to skip.
MATCH_SPECS_DIR = '/path/to/your/spectra'

# The bowtie figure (Fig 6) loads ~2600 individual spectrum FITS files from
# MATCH_SPECS_DIR, which lives on a separate machine. This only
# works if that path is mounted locally. Set to False to skip Figs 5-7
# and print a placeholder instead.
RUN_SPECTRA_PLOTS = True

# Lower KDE_GRIDSIZE (e.g. 25-50) for faster test runs; set to 200
# (seaborn's default) for the final paper-quality output.
KDE_GRIDSIZE = 50
DATA_DIR      = os.path.join(WORKING_DIR, 'data')
C3K_DIR       = os.path.join(WORKING_DIR, 'eBOSSDAP', 'c3k')
PLOT_FITS_DIR = os.path.join(WORKING_DIR, 'templates', 'ssps')

# ── eBOSS-DAP catalog paths ───────────────────────────────────────────────
# All four DR2 catalog files are available at:
#   https://datalab.noirlab.edu/data/sdss
# under the heading "SDSS DR17 Database Value Added Catalogs (VAC)".
PATH_EMLINES_FULL   = os.path.join(DATA_DIR, 'eboss_dap_dr2_emlines.fit')
PATH_EMLINES_HIGHEW = os.path.join(DATA_DIR, 'eboss_dap_dr2_emlines_high_ew.fit')
PATH_SPIND          = os.path.join(DATA_DIR, 'eboss_dap_dr2_spind.fit')
PATH_AUXDATA        = os.path.join(DATA_DIR, 'eboss_dap_dr2_auxdata.fit')

# Drake stellar-mass catalog (D. Miller et al. in prep.) — sections that need
# it check HAS_SUMMARY before running and skip gracefully if it's absent.
# Will be released alongside the Drake et al. catalog paper.
#PATH_SUMMARY = os.path.join(DATA_DIR, 'drake_catalog_name')
HAS_SUMMARY  = os.path.exists(PATH_SUMMARY)

# SDSS-I comparison sample — used in Sections 2 and 3 (Figs 1 and 2) to
# overlay the SDSS main galaxy redshift and S/N distributions. Available at:
#   https://data.sdss.org/sas/dr17/sdss/spectro/redux/specObj-dr17.fits
PATH_SDSS_SPEC = os.path.join(DATA_DIR, 'specObj-SDSS-dr17.fits')

# gas_template.csv contains the Balmer-series emission template for Fig 11.
# Its original location was outside the project directory, so we search a few
# likely spots rather than hardcoding a path that won't generalise.
# Will be available on the eBOSS-DAP GitHub repository.
for _gtdir in (os.path.join(WORKING_DIR, 'spectra'), os.path.join(WORKING_DIR, 'eBOSSDAP'), WORKING_DIR):
    if os.path.exists(os.path.join(_gtdir, 'gas_template.csv')):
        PATH_GAS_TEMPLATE = os.path.join(_gtdir, 'gas_template.csv')
        break
else:
    PATH_GAS_TEMPLATE = os.path.join(WORKING_DIR, 'spectra', 'gas_template.csv')
del _gtdir

# ── Dn4000/HδA template paths (Fig 24) ───────────────────────────────────
# These files will be available on the eBOSS-DAP GitHub repository.
PATH_SSPS_D4HDA = os.path.join(PLOT_FITS_DIR, 'd4hda_ckc_noneb_ssp.fit')
PATH_CONT_D4HDA = os.path.join(PLOT_FITS_DIR, 'd4hda_ckc_noneb_cont2.fit')
PATH_OUR_SPIND  = os.path.join(PLOT_FITS_DIR, 'd4hda_ckc_noneb.fit')

In [ ]:
# ── Output directory ──────────────────────────────────────────────────────
# Wiped and recreated on every run so stale figures don't accumulate.
# Everything this notebook produces — PDFs, PNGs, CSVs, table text files —
# lands here.
OUTPUT_DIR = os.path.join(os.path.dirname(DATA_DIR), 'outputs')
if os.path.exists(OUTPUT_DIR):
    shutil.rmtree(OUTPUT_DIR)
os.makedirs(OUTPUT_DIR)
print(f'Output directory ready: {OUTPUT_DIR}')

# mzr_measured.csv caches the per-galaxy metallicity measurements computed in
# Section 20 so they don't have to be recomputed if that section is re-run.
PATH_MZR_MEASURED = os.path.join(OUTPUT_DIR, 'mzr_measured.csv')

In [ ]:
# ── PMF tag helpers ───────────────────────────────────────────────────────
# A "tag" is the canonical string identifier for a spectrum: zero-padded
# plate (5 digits) + MJD + zero-padded fiber (4 digits), dash-separated.
# This format is used as the join key throughout the notebook.

def make_tag_from_pmf(plates, mjds, fibers):
    """Build dash-separated tag strings from plate, MJD, and fiber arrays."""
    p = [str(int(x)).zfill(5) for x in plates]
    m = [str(int(x))           for x in mjds]
    f = [str(int(x)).zfill(4)  for x in fibers]
    return np.array([f'{a}-{b}-{c}' for a,b,c in zip(p,m,f)])

def tag_from_pmf_str(pmf_arr):
    """Convert PMF_String (underscore-separated, as stored in the FITS files)
    to the dash-separated tag format used internally."""
    return np.array([s.replace('_','-') for s in pmf_arr])

def tag_from_pmf_string(pmf_strings):
    """Convert PMF_String to a 16-character zero-padded dash-separated tag.
    The zero-padding matches the output of make_tag_from_pmf, so these two
    formats can be compared directly after this conversion."""
    out = []
    for s in pmf_strings:
        s = s.strip().lstrip('0')
        out.append(s.replace('_', '-').zfill(16))
    return np.array(out)

In [ ]:
# ── Build df_downfiber for Section 5 (Fig 4) ─────────────────────────────
# Reads the infiber fraction and a handful of other per-spectrum quantities
# from AUXDATA now so Section 5 doesn't need to re-open the full catalog.
# Everything except df_downfiber is deleted at the end of this cell.
aux = fits.getdata(PATH_AUXDATA, 1)

sp_plate  = aux['PLATE'].byteswap().newbyteorder()
sp_mjd    = aux['MJD'].byteswap().newbyteorder()
sp_fiber  = aux['FIBER'].byteswap().newbyteorder()
sp_snr    = aux['SN_MEDIAN_G'].byteswap().newbyteorder()
# Redshift comes from SPIND rather than AUXDATA — the two files have the same
# rows in the same order, so this is a safe column-level join.
sp_z      = fits.getdata(PATH_SPIND, 1)['Z'].byteswap().newbyteorder()
sp_R90    = aux['PETROTH90_I'].byteswap().newbyteorder()
sp_R90_e  = aux['PETROTH90ERR_I'].byteswap().newbyteorder()
sp_infiber= aux['INFIBER'].byteswap().newbyteorder()
target_class = aux['TARGET_CLASS'].byteswap().newbyteorder()
ebv       = aux['E(B-V)'].byteswap().newbyteorder()
z         = sp_z
del aux

cal_sample_raw = pd.DataFrame({
    'tag':          make_tag_from_pmf(sp_plate, sp_mjd, sp_fiber),
    'snr':          sp_snr,
    'SpallZ':       sp_z,
    'R90':          sp_R90,
    'R90_err':      sp_R90_e,
    'infiber':      sp_infiber,
    'target_class': target_class,
    'ebv_mw':       ebv,
    'z':            z,
})
del sp_plate, sp_mjd, sp_fiber, sp_snr, sp_z
del sp_R90, sp_R90_e, sp_infiber, target_class, ebv, z

# Drop the small number of effectively zero-redshift entries — these are
# Milky Way star contaminants that slipped through the GALAXY classification.
cal_sample = cal_sample_raw[cal_sample_raw['z'] > 0.001].reset_index(drop=True)
del cal_sample_raw

# Keep only tag + infiber in memory; the rest isn't needed until Section 11.
df_downfiber = cal_sample[['tag','infiber']].copy()
del cal_sample

# Section 2 — Figure 1: Redshift Distribution
### *1.9 million galaxies walk into a histogram.*
*Loads target class and redshift from AUXDATA + SPIND, splits by the six targeting categories (LRG4, LRG3, LRGL, ELG, TDSP, QSO), and overlays the SDSS-I main galaxy sample for comparison. This is Figure 1 of the paper — it establishes the scale and heterogeneity of eBOSS and shows just how much redshift space SDSS-I left on the table. **Self-contained:** needs only Section 0.*

**Output:** `sample_makeup_unified.pdf` / `.png`

In [ ]:
# ── Load redshift and target class for Fig 1 ─────────────────────────────
# AUXDATA and SPIND are row-matched, so we can pull target class from one
# and redshift from the other without a merge.
aux = fits.getdata(PATH_AUXDATA, 1)
tag      = [make_tag_from_pmf([p],[m],[f])[0]
            for p,m,f in zip(aux['PLATE'].byteswap().newbyteorder(),
                              aux['MJD'].byteswap().newbyteorder(),
                              aux['FIBER'].byteswap().newbyteorder())]
target_class = aux['TARGET_CLASS'].byteswap().newbyteorder()
zd           = fits.getdata(PATH_SPIND, 1)['Z'].byteswap().newbyteorder()
del aux

dist_sample_raw  = pd.DataFrame({'tag': tag, 'target_class': target_class, 'z': zd})
# Apply the same upper redshift cut used throughout the paper (Hβ redshifts
# out of the spectrograph window above z ~ 1.12).
dist_sample = dist_sample_raw[dist_sample_raw['z'] <= 1.12]
del tag, target_class, zd, dist_sample_raw

# Split by targeting category for the stacked histogram.
lrg3s = dist_sample[dist_sample['target_class'].str.contains('LRG3')]
lrg4s = dist_sample[dist_sample['target_class'].str.contains('LRG4')]
lrgls = dist_sample[dist_sample['target_class'].str.contains('LRGL')]
elgs  = dist_sample[dist_sample['target_class'].str.contains('ELG')]
tdsp  = dist_sample[dist_sample['target_class'].str.contains('TDSP')]
qso   = dist_sample[dist_sample['target_class'].str.contains('QSO')]

# SDSS-I comparison sample. specObj-dr17.fits covers all SDSS I-IV spectra;
# we filter to the SDSS-I main galaxy survey here.
# Download: https://data.sdss.org/sas/dr17/sdss/spectro/redux/specObj-dr17.fits
sdss_hdu = fits.open(PATH_SDSS_SPEC)
survey = sdss_hdu[1].data['survey'].byteswap().newbyteorder()
sdss_z = sdss_hdu[1].data['z'].byteswap().newbyteorder()
cl     = sdss_hdu[1].data['CLASS'].byteswap().newbyteorder()
sdss_hdu.close()
df_sdss = pd.DataFrame({'survey': survey, 'z': sdss_z, 'class': cl})
sdss_gals = df_sdss[df_sdss['class'] == 'GALAXY']
sdss_main = sdss_gals[sdss_gals['survey'].str.contains('sdss')]
del survey, sdss_z, cl, df_sdss, sdss_gals

In [ ]:
plot_prettier(150, fontsize=18)
plt.figure(figsize=[11, 8.5])
binwidth = 0.01
bini = np.arange(0.0, 1.13 + binwidth, binwidth)

# eBOSS targeting-class histograms (unfilled step outlines)
plt.hist(lrg4s['z'], bins=bini, histtype='step', color=c_lrg4, fill=False, label='eBOSS LRG')
plt.hist(lrg3s['z'], bins=bini, histtype='step', color=c_lrg3, fill=False, label='BOSS LRG')
plt.hist(lrgls['z'], bins=bini, histtype='step', color=c_lrgl, fill=False, label='BOSS LOWZ LRG')
plt.hist(elgs['z'],  bins=bini, histtype='step', color=c_elg,  fill=False, label='ELG')
plt.hist(tdsp['z'],  bins=bini, histtype='step', color=c_tdsp, fill=False, label='TDSS/SPIDERS')
plt.hist(qso['z'],   bins=bini, histtype='step', color=c_qso,  fill=False, label='QSO')

# eBOSS total: faint grey fill + black outline, inline label placed at the
# histogram peak so it doesn't compete with the legend.
plt.hist(dist_sample['z'], bins=bini, histtype='step', color=c_tot, alpha=0.10, fill=True, zorder=-1)
counts, edges, _ = plt.hist(dist_sample['z'], bins=bini, histtype='step', color='black',
                             alpha=1.0, fill=False, zorder=-1, label='eBOSS Total')
peak_index = np.argmax(counts)
plt.text(0.535, counts[peak_index]*0.46, 'eBOSS', fontsize=12,
         bbox=dict(facecolor='white', alpha=0.0), ha='center')

# SDSS-I: same treatment, drawn last so it sits on top.
plt.hist(sdss_main['z'], bins=bini, histtype='step', color='xkcd:light blue',
         alpha=0.15, fill=True, zorder=-1)
counts_s, edges_s, _ = plt.hist(sdss_main['z'], bins=bini, histtype='step',
                                 color='xkcd:sky blue', alpha=1.0, fill=False, zorder=-1)
peak_index_s = np.argmax(counts_s)
peak_x_s = (edges_s[peak_index_s] + edges_s[peak_index_s+1]) / 2
plt.text(peak_x_s + 0.02, counts_s[peak_index_s]*0.4, 'SDSS-I', fontsize=12,
         color='xkcd:bright blue', bbox=dict(facecolor=None, edgecolor=None, alpha=0.0), ha='center')
del counts, edges, peak_index, counts_s, edges_s, peak_index_s, peak_x_s

# The legend mixes Line2D handles (for the 6 targeting classes) and Patch
# handles (for the filled eBOSS Total and SDSS-I). Building them explicitly
# lets us control both the style and order.
handles = []
handles.append(mlines.Line2D([], [], color=c_lrg4, label='eBOSS LRG'))
handles.append(mlines.Line2D([], [], color=c_lrg3, label='BOSS LRG'))
handles.append(mlines.Line2D([], [], color=c_lrgl, label='BOSS LOWZ LRG'))
handles.append(mlines.Line2D([], [], color=c_elg,  label='ELG'))
handles.append(mlines.Line2D([], [], color=c_tdsp, label='TDSS/SPIDERS'))
handles.append(mlines.Line2D([], [], color=c_qso,  label='QSO'))

faint_grey = (0.5019607843137255, 0.5019607843137255, 0.5019607843137255, 0.1)
handles.append(mpatches.Patch(label='eBOSS Total', facecolor=faint_grey, edgecolor='black'))

faint_blue = (0.5843137254901961, 0.8156862745098039, 0.9882352941176471, 0.15)
handles.append(mpatches.Patch(label='SDSS-I', facecolor=faint_blue, edgecolor='xkcd:sky blue'))

leg = plt.legend(handles=handles, loc=1, fontsize=20)
leg.set_zorder(100)

plt.xlabel('Redshift', fontsize=36)
plt.ylabel('Number of Galaxies', fontsize=36)
plt.xlim(0.0, 1.12)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'sample_makeup_unified.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'sample_makeup_unified.png'), dpi=150, bbox_inches='tight')
plt.show()
del bini

In [ ]:
# ── Section 2 cleanup ────────────────────────────────────────────────────
# Separated from the plotting cell so Figs 1 and 2 can each be re-run
# independently without hitting a NameError on sdss_main.
del sdss_main

# Section 3 — Figure 2: S/N Distribution
### *Bigger survey, noisier spectra. A trade we made with open eyes.*
*Same structure as Figure 1 but with median spectral S/N on the x-axis. The SDSS-I comparison is doing real work here: eBOSS is roughly twice the size at systematically lower S/N per pixel, which sets the context for every data quality discussion that follows. **Self-contained:** needs only Section 0.*

**Output:** `sample_SNR.pdf` / `.png`

In [ ]:
# ── Load S/N and target class for Fig 2 ──────────────────────────────────
# SNR comes from the emlines catalog; target class is merged in from AUXDATA.
# The two catalogs share PMF identifiers but aren't row-matched, so we join
# on the tag string.
hdu_eml_snr = fits.open(PATH_EMLINES_FULL)
e_pl  = hdu_eml_snr[1].data['PLATE'].byteswap().newbyteorder()
e_mj  = hdu_eml_snr[1].data['MJD'].byteswap().newbyteorder()
e_fb  = hdu_eml_snr[1].data['FIBER'].byteswap().newbyteorder()
e_snr = hdu_eml_snr[1].data['SNR'].byteswap().newbyteorder()
hdu_eml_snr.close()

snr_sample = pd.DataFrame({'tag': make_tag_from_pmf(e_pl, e_mj, e_fb), 'SNR': e_snr})
del e_pl, e_mj, e_fb, e_snr

aux = fits.getdata(PATH_AUXDATA, 1)
sm_tag  = [make_tag_from_pmf([p],[m],[f])[0]
           for p,m,f in zip(aux['PLATE'].byteswap().newbyteorder(),
                             aux['MJD'].byteswap().newbyteorder(),
                             aux['FIBER'].byteswap().newbyteorder())]
sm_target_class = aux['TARGET_CLASS'].byteswap().newbyteorder()
del aux

df_target_class2 = pd.DataFrame({'tag': sm_tag, 'TARGET_CLASS': sm_target_class}).drop_duplicates(subset='tag')
del sm_tag, sm_target_class

snr_sample = snr_sample.merge(df_target_class2, on='tag')
del df_target_class2
# Drop entries with non-positive S/N — these are spectra with pathological
# noise estimates that would clutter the low end of the histogram.
snr_sample = snr_sample[snr_sample['SNR'] > 0]

lrg3s = snr_sample[snr_sample['TARGET_CLASS'].str.contains('LRG3')]
lrg4s = snr_sample[snr_sample['TARGET_CLASS'].str.contains('LRG4')]
lrgls = snr_sample[snr_sample['TARGET_CLASS'].str.contains('LRGL')]
elgs  = snr_sample[snr_sample['TARGET_CLASS'].str.contains('ELG')]
tdsp  = snr_sample[snr_sample['TARGET_CLASS'].str.contains('TDSP')]
qso   = snr_sample[snr_sample['TARGET_CLASS'].str.contains('QSO')]

# SDSS-I comparison sample — same file as Fig 1, re-opened here so Section 3
# can run independently without Section 2 in memory.
# Download: https://data.sdss.org/sas/dr17/sdss/spectro/redux/specObj-dr17.fits
sdss_hdu2  = fits.open(PATH_SDSS_SPEC)
snr_all_sdss = sdss_hdu2[1].data['SN_MEDIAN_ALL'].byteswap().newbyteorder()
class_sdss   = sdss_hdu2[1].data['CLASS'].byteswap().newbyteorder()
qual_sdss    = sdss_hdu2[1].data['PLATEQUALITY'].byteswap().newbyteorder()
zwarn_sdss   = sdss_hdu2[1].data['ZWARNING'].byteswap().newbyteorder()
z_sdss       = sdss_hdu2[1].data['Z'].byteswap().newbyteorder()
sdss_hdu2.close()

# Apply the same quality cuts used to define the eBOSS sample, for a fair
# apples-to-apples comparison.
class_sdss = np.char.strip(class_sdss)
qual_sdss  = np.char.strip(qual_sdss)
sdss_mask = ((class_sdss == 'GALAXY') & (qual_sdss == 'good')
              & (zwarn_sdss == 0) & (z_sdss >= 0.0005))
SNR_sdss = snr_all_sdss[sdss_mask]
SNR_sdss = SNR_sdss[SNR_sdss > 0.0]
del snr_all_sdss, class_sdss, qual_sdss, zwarn_sdss, z_sdss, sdss_mask

In [ ]:
plot_prettier(150, fontsize=18)
plt.figure(figsize=[11, 8.5])
binwidth = 0.1
bini = np.arange(min(snr_sample['SNR']), 20, binwidth)

plt.hist(lrg4s['SNR'], bins=bini, histtype='step', color='magenta',          alpha=1, fill=False)
plt.hist(lrg3s['SNR'], bins=bini, histtype='step', color='xkcd:dark red',     alpha=1, fill=False)
plt.hist(lrgls['SNR'], bins=bini, histtype='step', color='xkcd:burnt orange', alpha=1, fill=False)
plt.hist(elgs['SNR'],  bins=bini, histtype='step', color='green',             alpha=1, fill=False)
plt.hist(tdsp['SNR'],  bins=bini, histtype='step', color='xkcd:dark yellow',  alpha=1, fill=False)
plt.hist(qso['SNR'],   bins=bini, histtype='step', color='blue',              alpha=1, fill=False)

# eBOSS total with inline label, matching the style of Fig 1.
plt.hist(snr_sample['SNR'], bins=bini, histtype='step', color='grey', alpha=0.10, fill=True, zorder=-1)
counts, bins, _ = plt.hist(snr_sample['SNR'], bins=bini, histtype='step', color='black',
                            alpha=1, fill=False, zorder=-1)
peak_index = np.argmax(counts)
peak_x = (bins[peak_index] + bins[peak_index+1]) / 2
plt.text(peak_x + 0.2, counts[peak_index]*0.35, 'eBOSS', fontsize=12,
         bbox=dict(facecolor='white', alpha=0.0), ha='center')

# SDSS-I comparison.
plt.hist(SNR_sdss, bins=bini, histtype='step', color='xkcd:light blue', alpha=0.15, fill=True, zorder=-1)
counts, bins, _ = plt.hist(SNR_sdss, bins=bini, histtype='step', color='xkcd:sky blue',
                            alpha=1, fill=False, zorder=-2)
peak_index = np.argmax(counts)
peak_x = (bins[peak_index] + bins[peak_index+1]) / 2
plt.text(peak_x + 0.8, counts[peak_index]*0.5, 'SDSS-I', fontsize=12, color='xkcd:bright blue',
         bbox=dict(facecolor=None, edgecolor=None, alpha=0.0), ha='center')
del counts, bins, peak_index, peak_x

# Legend with explicit handles — same approach as Fig 1.
handles = []
handles.append(mlines.Line2D([], [], color='magenta',          label='eBOSS LRG'))
handles.append(mlines.Line2D([], [], color='xkcd:dark red',     label='BOSS LRG'))
handles.append(mlines.Line2D([], [], color='xkcd:burnt orange', label='BOSS LOWZ LRG'))
handles.append(mlines.Line2D([], [], color='green',             label='ELG'))
handles.append(mlines.Line2D([], [], color='xkcd:dark yellow',  label='TDSS/SPIDERS'))
handles.append(mlines.Line2D([], [], color='blue',              label='QSO'))

faint_grey = (0.5019607843137255, 0.5019607843137255, 0.5019607843137255, 0.1)
handles.append(mpatches.Patch(label='eBOSS Total', facecolor=faint_grey, edgecolor='black'))

faint_blue = (0.5843137254901961, 0.8156862745098039, 0.9882352941176471, 0.15)
handles.append(mpatches.Patch(label='SDSS-I', facecolor=faint_blue, edgecolor='xkcd:sky blue'))

leg = plt.legend(handles=handles, loc=1, fontsize=20)
leg.set_zorder(100)

plt.xlabel('Median Spectral S/N', fontsize=36)
plt.ylabel('Number of Galaxies', fontsize=36)
plt.xlim(0.2, 20.0)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'sample_SNR.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'sample_SNR.png'), dpi=150, bbox_inches='tight')
plt.show()
del bini

In [ ]:
# ── Section 3 cleanup ────────────────────────────────────────────────────
# Separated so the plotting cell can be re-run without re-running the data
# load, and so the cleanup doesn't silently break a second pass.
del snr_sample, lrg4s, lrg3s, lrgls, elgs, tdsp, qso, SNR_sdss

# Section 4 — Figure 3: Hβ EW vs Redshift
### *A first look at who's forming stars and who's just sitting there.*
*Hβ equivalent width against redshift for all six target classes, shown as KDE contours over a full-sample 2D histogram. ELGs and QSO-targeted galaxies cluster at high EW; LRGs sit near zero. The mild uptick in LRG EW with redshift is real. This figure also motivates the high-EW / low-EW catalog split that runs through the whole pipeline — not every spectrum needs the full line list fitted. **Self-contained:** needs only Section 0.*

**Output:** `HB_EW_vs_z_kde.pdf` / `.png`

In [ ]:
# ── Load Hβ EW and redshift for Fig 3 ────────────────────────────────────
# EW values come from the emlines catalog; target class is merged in from
# AUXDATA on the tag string.
hdu_eml_ew = fits.open(PATH_EMLINES_FULL)
e_pl  = hdu_eml_ew[1].data['PLATE'].byteswap().newbyteorder()
e_mj  = hdu_eml_ew[1].data['MJD'].byteswap().newbyteorder()
e_fb  = hdu_eml_ew[1].data['FIBER'].byteswap().newbyteorder()
e_z   = hdu_eml_ew[1].data['Z'].byteswap().newbyteorder()
e_hbew = hdu_eml_ew[1].data['H_beta_EW'].byteswap().newbyteorder()
hdu_eml_ew.close()

ew_sample_raw = pd.DataFrame({'tag': make_tag_from_pmf(e_pl, e_mj, e_fb), 'z': e_z, 'Hbeta_ew': e_hbew})
del e_pl, e_mj, e_fb, e_z, e_hbew

aux = fits.getdata(PATH_AUXDATA, 1)
tag4  = [make_tag_from_pmf([p],[m],[f])[0]
         for p,m,f in zip(aux['PLATE'].byteswap().newbyteorder(),
                           aux['MJD'].byteswap().newbyteorder(),
                           aux['FIBER'].byteswap().newbyteorder())]
target_class4 = aux['TARGET_CLASS'].byteswap().newbyteorder()
del aux

df_target_class3 = pd.DataFrame({'tag': tag4, 'target_class': target_class4}).drop_duplicates(subset='tag')
del tag4, target_class4

ew_sample = df_target_class3.merge(ew_sample_raw, on='tag')
del df_target_class3, ew_sample_raw

# Quality cuts: exclude unphysical EW values and non-finite entries. The
# EW floor of 0.01 Å removes non-detections stored as near-zero.
cut1 = ew_sample[(ew_sample['Hbeta_ew'] > 10**(-2)) & (ew_sample['Hbeta_ew'] < 300)]
cut2 = cut1[np.isfinite(cut1['z'])]
df_targ_ew = cut2[np.isfinite(cut2['Hbeta_ew'])].copy()
del ew_sample, cut1, cut2
# Log-transform for plotting; the KDE runs in log space.
df_targ_ew['LOG_HBEW'] = np.log10(df_targ_ew['Hbeta_ew'])

lrg3s_ew = df_targ_ew[df_targ_ew['target_class'].str.contains('LRG3')]
lrg4s_ew = df_targ_ew[df_targ_ew['target_class'].str.contains('LRG4')]
lrgls_ew = df_targ_ew[df_targ_ew['target_class'].str.contains('LRGL')]
elgs_ew  = df_targ_ew[df_targ_ew['target_class'].str.contains('ELG')]
tdsp_ew  = df_targ_ew[df_targ_ew['target_class'].str.contains('TDSP')]
qso_ew   = df_targ_ew[df_targ_ew['target_class'].str.contains('QSO')]

In [ ]:
plot_prettier(150, fontsize=18)
plt.figure(figsize=[11, 8.5])

# 2D histogram of the full sample as a grey background. cmin=100 suppresses
# sparsely populated bins so the KDE contours read cleanly on top.
plt.hist2d(df_targ_ew['z'], df_targ_ew['LOG_HBEW'],
           bins=50, range=[[0,1.2],[-2,2.4]], cmin=100, cmap='gray_r',
           vmin=0.0, zorder=0)

# KDE contours per target class. thresh=0.5 means contours start at the
# 50th percentile of each class's density, so only the densest regions are
# shown — avoids contours spreading into sparsely sampled corners.
handles = []
_target_classes = [
    (lrg4s_ew, c_lrg4, 'eBOSS LRG'),
    (lrg3s_ew, c_lrg3, 'BOSS LRG'),
    (lrgls_ew, c_lrgl, 'BOSS LOWZ LRG'),
    (elgs_ew,  c_elg,  'ELG'),
    (tdsp_ew,  c_tdsp, 'TDSS/SPIDERS'),
    (qso_ew,   c_qso,  'QSO'),
]
_ew_total = sum(len(g) for g,_,_ in _target_classes)
for _idx, (grp, color, label) in enumerate(_target_classes):
    n_done = sum(len(g) for g,_,_ in _target_classes[:_idx+1])
    print(f'Starting [{n_done:,}/{_ew_total:,} = {100*n_done//_ew_total}%] {label}...')
    sns.kdeplot(x=grp['z'], y=grp['LOG_HBEW'],
                fill=False, thresh=0.5, levels=4, cut=3, gridsize=KDE_GRIDSIZE,
                color=color, linewidths=2, clip=((0,1.2),(-2,2.4)))
    print('Finished.')
    handles.append(mlines.Line2D([], [], color=color, label=label))

plt.ylabel(r'log EW(H$\beta$) [Å]', fontsize=28)
plt.xlabel('Redshift', fontsize=28)
plt.xlim(0, 1.2)
plt.ylim(-2, 2.4)
leg = plt.legend(handles=handles, loc=4, fontsize=12)
leg.set_zorder(100)
for lh in getattr(leg, 'legend_handles', getattr(leg, 'legendHandles', [])): lh.set_alpha(1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'HB_EW_vs_z_kde.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'HB_EW_vs_z_kde.png'), dpi=150, bbox_inches='tight')
plt.show()
del handles

In [ ]:
# ── Section 4 cleanup ────────────────────────────────────────────────────
del df_targ_ew, lrg3s_ew, lrg4s_ew, lrgls_ew, elgs_ew, tdsp_ew, qso_ew

# Section 5 — Figure 4: Fraction of Light in Fiber
### *How much of the galaxy actually made it down the tube?*
*Takes `df_downfiber` built in Section 1 and merges in target class and redshift from AUXDATA + SPIND to produce a 2D KDE of fiber light fraction versus redshift, split by targeting category. The figure shows that fiber coverage varies substantially by target class and redshift — close enough to a global spectrum for most science cases, far enough to matter for anything aperture-sensitive. **Requires:** Sections 0 and 1.*

**Output:** `fluxdownfiberratio_kde.pdf` / `.png`

In [ ]:
# ── Merge df_downfiber with target class and redshift for Fig 4 ──────────
# df_downfiber was built in Section 1 and contains only tag + infiber.
# We merge in target class and redshift from AUXDATA + SPIND here, and
# apply loose quality cuts to keep the KDE well-behaved.
df_infiber_full = df_downfiber
aux = fits.getdata(PATH_AUXDATA, 1)
tag_in  = [make_tag_from_pmf([p],[m],[f])[0]
           for p,m,f in zip(aux['PLATE'].byteswap().newbyteorder(),
                             aux['MJD'].byteswap().newbyteorder(),
                             aux['FIBER'].byteswap().newbyteorder())]
target_class_in = aux['TARGET_CLASS'].byteswap().newbyteorder()
z_in    = fits.getdata(PATH_SPIND, 1)['Z'].byteswap().newbyteorder()
del aux

df_target_class = pd.DataFrame({'tag':tag_in, 'target_class':target_class_in, 'z':z_in})
del tag_in, target_class_in, z_in

df_if = df_infiber_full.merge(df_target_class, on='tag')
del df_infiber_full, df_target_class, df_downfiber

# Clip infiber > 1.2: a small fraction of spectra have synthetic photometry
# brighter than the imaging, usually due to flux calibration errors or
# photometric deblending failures. Including them distorts the KDE tail.
df_if = df_if[(df_if['z'] > 0) & (df_if['z'] < 1.2) &
              (df_if['infiber'] > 0) & (df_if['infiber'] < 1.2)]

lrg3s_if = df_if[df_if['target_class'].str.contains('LRG3')]
lrg4s_if = df_if[df_if['target_class'].str.contains('LRG4')]
lrgls_if = df_if[df_if['target_class'].str.contains('LRGL')]
elgs_if  = df_if[df_if['target_class'].str.contains('ELG')]
tdsp_if  = df_if[df_if['target_class'].str.contains('TDSP')]
qso_if   = df_if[df_if['target_class'].str.contains('QSO')]

In [ ]:
plt.figure(figsize=[11, 8.5])
handles = []
_target_classes = [
    (lrg4s_if, c_lrg4, 'eBOSS LRG'),
    (lrg3s_if, c_lrg3, 'BOSS LRG'),
    (lrgls_if, c_lrgl, 'BOSS LOWZ LRG'),
    (elgs_if,  c_elg,  'ELG'),
    (tdsp_if,  c_tdsp, 'TDSS/SPIDERS'),
    (qso_if,   c_qso,  'QSO'),
]
# thresh=0.1 and cut=20 produce more extended contours than Figs 1-3,
# deliberately showing the full tail of each class's infiber distribution.
_if_total = sum(len(g) for g,_,_ in _target_classes)
for _idx, (grp, color, label) in enumerate(_target_classes):
    n_done = sum(len(g) for g,_,_ in _target_classes[:_idx+1])
    print(f'Starting [{n_done:,}/{_if_total:,} = {100*n_done//_if_total}%] {label}...')
    sns.kdeplot(x=grp['z'], y=grp['infiber'],
                fill=False, thresh=0.1, levels=7, cut=20, gridsize=KDE_GRIDSIZE,
                color=color, linewidths=3, zorder=5)
    print('Finished.')
    handles.append(mlines.Line2D([], [], color=color, label=label))

plt.ylim(0, 1.2)
plt.xlim(0, 1.2)
plt.ylabel('Fraction of light in fiber', fontsize=28)
plt.xlabel('Redshift', fontsize=28)
leg = plt.legend(handles=handles, loc=4, fontsize=12)
leg.set_zorder(100)
for lh in getattr(leg, 'legend_handles', getattr(leg, 'legendHandles', [])): lh.set_alpha(1)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'fluxdownfiberratio_kde.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'fluxdownfiberratio_kde.png'), dpi=150, bbox_inches='tight')
plt.show()
del handles

In [ ]:
# ── Section 5 cleanup ────────────────────────────────────────────────────
del df_if, lrg3s_if, lrg4s_if, lrgls_if, elgs_if, tdsp_if, qso_if

# Section 6 — Figures 5 & 6: Spectrophotometric Calibration
### *How well did the flux calibration actually work? Let's ask the repeat spectra.*
*The heavy infrastructure section. Reads AUXDATA to build `pair_sample`, then cross-matches all spectra by sky position to find repeat observations of the same object. Those repeat pairs are the calibration testbed: Fig. 5 compares total flux in a narrow window around each `LAMBDA_EFF` between epochs (the absolute calibration check), and Fig. 6 divides normalised repeat spectra across the full wavelength range to map how calibration error varies with wavelength (the "bowtie" plot). Fig. 6 requires `MATCH_SPECS_DIR` to be mounted — set `RUN_SPECTRA_PLOTS = False` in Section 1 to skip it. This section also writes `df_uniques` (the full repeat-pair index) to memory, which Section 11 needs to build `savedf`. **Requires:** Sections 0 and 1. **Required before:** Section 11 (for `df_uniques`).*

**Fig. 5** `pair_hist_all_gauss.pdf` — flux ratio distributions at `LAMBDA_EFF`, split by 4000/5400/7500 Å.

**Fig. 6** `bowtie_plot_all.pdf` — spectral-shape calibration error vs. wavelength.

In [ ]:
# ── Build pair_sample and find repeat-observation pairs (Figs 5, 6, 7) ───
# This is the busiest cell in the notebook. It does three distinct things:
#
#   1. Reads AUXDATA + SPIND to build pair_sample — the working DataFrame
#      for all spectrophotometric calibration figures.
#   2. Computes photo/spectro synthetic colours needed for Fig 7 (absolute
#      calibration), and applies S/N + brightness cuts to produce df5400_gr
#      and df5400_ri.
#   3. Cross-matches all spectra within 1 arcsec to find repeat observations
#      of the same galaxy, groups them into unique pairs, and writes
#      df_uniques — the repeat-pair index that Section 11 uses to build
#      savedf. Also produces match_4000/5400/7500: the S/N-filtered pair
#      subsets used by the bowtie plot (Fig 6).
#
# LAMBDA_EFF determines which wavelength the flux calibration was optimised
# for (4000/5400/7500 Å depending on target type). The S/N cuts for each
# LAMBDA_EFF group are intentionally different to yield a workable number
# of pairs per group despite the different survey depths.
aux = fits.getdata(PATH_AUXDATA, 1)

pc_plate = aux['PLATE'].byteswap().newbyteorder()
pc_mjd   = aux['MJD'].byteswap().newbyteorder()
pc_fiber = aux['FIBER'].byteswap().newbyteorder()

pc_snr     = aux['SN_MEDIAN_R'].byteswap().newbyteorder()
pc_snr_i   = aux['SN_MEDIAN_I'].byteswap().newbyteorder()
pc_snr_all = aux['SN_MEDIAN_ALL'].byteswap().newbyteorder()
pc_lameff  = aux['LAMBDA_EFF'].byteswap().newbyteorder()
pc_ra      = aux['RA'].byteswap().newbyteorder()
pc_dec     = aux['DEC'].byteswap().newbyteorder()
pc_z       = fits.getdata(PATH_SPIND, 1)['Z'].byteswap().newbyteorder()

# ── Synthetic photo/spectro colours for Fig 7 ────────────────────────────
# Model fluxes are from the imaging; spectroflux is synthesised from the
# spectra. The difference in colour is the absolute calibration error.
mg = aux['MODELFLUX_G'].byteswap().newbyteorder()
mr = aux['MODELFLUX_R'].byteswap().newbyteorder()
mi = aux['MODELFLUX_I'].byteswap().newbyteorder()
mg_ivar = aux['MODELFLUX_IVAR_G'].byteswap().newbyteorder()
mr_ivar = aux['MODELFLUX_IVAR_R'].byteswap().newbyteorder()
mi_ivar = aux['MODELFLUX_IVAR_I'].byteswap().newbyteorder()
sg = aux['SPECTROFLUX_G'].byteswap().newbyteorder()
sr = aux['SPECTROFLUX_R'].byteswap().newbyteorder()
si = aux['SPECTROFLUX_I'].byteswap().newbyteorder()
del aux

mg_e = np.power(mg_ivar, -0.5)
mr_e = np.power(mr_ivar, -0.5)
mi_e = np.power(mi_ivar, -0.5)
del mg_ivar, mr_ivar, mi_ivar

# AB magnitudes from fluxes; errors propagated via d(m)/d(f) = 2.5/ln(10)/f.
photo_color_gr = -2.5*np.log10(mg/mr)
photo_color_ri = -2.5*np.log10(mr/mi)
model_mag_r    = 22.5 - 2.5*np.log10(mr)
mag_g_e = (2.5/np.log(10)) * mg_e/mg
mag_r_e = (2.5/np.log(10)) * mr_e/mr
mag_i_e = (2.5/np.log(10)) * mi_e/mi
photo_color_gr_err = np.sqrt(mag_g_e**2 + mag_r_e**2)
photo_color_ri_err = np.sqrt(mag_r_e**2 + mag_i_e**2)
del mg, mr, mi, mg_e, mr_e, mi_e, mag_g_e, mag_r_e, mag_i_e

spectro_color_gr = -2.5*np.log10(sg/sr)
spectro_color_ri = -2.5*np.log10(sr/si)
del sg, sr, si

pair_sample_raw = pd.DataFrame({
    'plate':pc_plate, 'mjd':pc_mjd, 'fiber':pc_fiber, 'snr':pc_snr,
    'lambda_eff':pc_lameff, 'snr_all':pc_snr_all, 'snr_i':pc_snr_i,
    'RA':pc_ra, 'DEC':pc_dec, 'z':pc_z,
    'photo_color_gr':photo_color_gr, 'photo_color_gr_err':photo_color_gr_err,
    'spectro_color_gr':spectro_color_gr,
    'photo_color_ri':photo_color_ri, 'photo_color_ri_err':photo_color_ri_err,
    'spectro_color_ri':spectro_color_ri, 'model_mag_r':model_mag_r,
})
del (pc_plate, pc_mjd, pc_fiber, pc_snr, pc_lameff, pc_snr_all, pc_snr_i,
     pc_ra, pc_dec, pc_z,
     photo_color_gr, photo_color_gr_err, photo_color_ri, photo_color_ri_err,
     spectro_color_gr, spectro_color_ri, model_mag_r)

# Drop spectra with non-positive S/N (pathological noise estimates) and the
# very small number of effectively-zero redshift entries.
pair_sample_snrcut = pair_sample_raw[pair_sample_raw['snr'] > 0].reset_index(drop=True)
del pair_sample_raw
pair_sample_snrcut['tag'] = make_tag_from_pmf(pair_sample_snrcut['plate'], pair_sample_snrcut['mjd'], pair_sample_snrcut['fiber'])

pair_sample = pair_sample_snrcut[pair_sample_snrcut['z'] > 0.001].reset_index(drop=True).copy()
del pair_sample_snrcut

# ── Fig 7 inputs: bright LAMBDA_EFF=5400 galaxies ────────────────────────
# The r < 17.7 cut ensures photometric errors are small enough that any
# colour discrepancy between imaging and spectroscopy is a calibration issue,
# not a photometry issue. S/N >= 3 removes noisy spectra.
pair_sample['color_comp_gr'] = pair_sample['photo_color_gr'] - pair_sample['spectro_color_gr']
pair_sample['color_comp_ri'] = pair_sample['photo_color_ri'] - pair_sample['spectro_color_ri']
df5400_ini = pair_sample[pair_sample['lambda_eff'] == 5400.0]
_cut1 = df5400_ini[df5400_ini['snr'] >= 3.0]
_cut2 = _cut1[_cut1['model_mag_r'] < 17.7]
df5400_gr = _cut2[_cut2['photo_color_gr'] > 5*_cut2['photo_color_gr_err']]
df5400_ri = _cut2[_cut2['photo_color_ri'] > 5*_cut2['photo_color_ri_err']]
del df5400_ini, _cut1, _cut2
print(f'abs_cal 5400 bright sample: gr={len(df5400_gr):,}  ri={len(df5400_ri):,}')

# ── Find repeat-observation pairs within 1 arcsec ────────────────────────
# search_around_sky returns every (i, j) pair within the search radius,
# including self-matches (i == j). We filter self-matches in the loop below.
catalog = SkyCoord(ra=pair_sample['RA'].to_numpy()*u.degree,
                    dec=pair_sample['DEC'].to_numpy()*u.degree)
idxc, idxcatalog, _, _ = catalog.search_around_sky(catalog, 1*u.arcsec)
del catalog

tag_arr = pair_sample['tag'].to_numpy()
tag1_all = tag_arr[idxc]
tag2_all = tag_arr[idxcatalog]
del idxc, idxcatalog

init_tags, matched_tags = [], []
for t1, t2 in zip(tag1_all, tag2_all):
    if t1 != t2:
        init_tags.append(t1)
        matched_tags.append(t2)
del tag1_all, tag2_all, tag_arr

# Deduplicate: (A,B) and (B,A) represent the same pair. Using frozenset
# as the key collapses both orderings into one row.
all_pairs_raw = pd.DataFrame({'init_tag': init_tags, 'matches': matched_tags})
del init_tags, matched_tags
out = all_pairs_raw[~all_pairs_raw[['init_tag','matches']].apply(frozenset, axis=1).duplicated()]
del all_pairs_raw
grp = out.groupby('init_tag').agg(lambda x: list(x)).reset_index()
del out

# Group repeat observations of the same object into clusters (some objects
# were observed more than twice), then expand clusters into all unique pairs.
repeat_groups = [grp['matches'][i] + [grp['init_tag'][i]] for i in range(grp.shape[0])]
del grp

import itertools
def _unique_combinations(elements):
    return list(set(itertools.combinations(elements, 2)))

uniques1, uniques2 = [], []
for group in repeat_groups:
    for a, b in _unique_combinations(group):
        uniques1.append(a)
        uniques2.append(b)
del repeat_groups

# df_uniques is kept in memory and consumed by Section 11 to build savedf.
df_uniques = pd.DataFrame({'tag1': uniques1, 'tag2': uniques2})
del uniques1, uniques2
print(f'singlematch pairs: {len(df_uniques):,}')

# ── Fig 6 inputs: S/N-filtered pair subsets per LAMBDA_EFF ───────────────
# S/N thresholds differ per LAMBDA_EFF because the populations have different
# typical S/N distributions. The thresholds were chosen empirically to
# yield a usable number of pairs while suppressing spectra with near-zero flux.
pairs_snr6   = pair_sample[pair_sample['snr'] > 6]
pairs_snr3   = pair_sample[pair_sample['snr'] > 3]
pairs_snr1_5 = pair_sample[pair_sample['snr'] > 1.5]

# Keep only spectra that appear in at least one repeat pair.
unispecs = np.unique(np.concatenate([df_uniques['tag1'].to_numpy(),
                                      df_uniques['tag2'].to_numpy()]))
paired_spectra = pair_sample[pair_sample['tag'].isin(unispecs)]
del unispecs, pair_sample

# Filter down to pairs where both members pass the LAMBDA_EFF-specific S/N cut.
df_4000 = paired_spectra[(paired_spectra['lambda_eff'] == 4000.0) & (paired_spectra['tag'].isin(pairs_snr3['tag']))]
df_5400 = paired_spectra[(paired_spectra['lambda_eff'] == 5400.0) & (paired_spectra['tag'].isin(pairs_snr6['tag']))]
df_7500 = paired_spectra[(paired_spectra['lambda_eff'] == 7500.0) & (paired_spectra['tag'].isin(pairs_snr1_5['tag']))]
del paired_spectra, pairs_snr3, pairs_snr6, pairs_snr1_5

# Cap at 2000 pairs per LAMBDA_EFF group for the bowtie (Fig 6). More pairs
# don't change the envelope shape but add significant runtime.
np.random.seed(42)

match_4000_b = df_uniques[df_uniques['tag1'].isin(df_4000['tag']) &
                           df_uniques['tag2'].isin(df_4000['tag'])]
match_4000 = (match_4000_b.sample(2000).reset_index()[['tag1','tag2']]
               if len(match_4000_b) > 2000 else match_4000_b[['tag1','tag2']])
del df_4000, match_4000_b

match_5400_b = df_uniques[df_uniques['tag1'].isin(df_5400['tag']) &
                           df_uniques['tag2'].isin(df_5400['tag'])]
match_5400 = (match_5400_b.sample(2000).reset_index()[['tag1','tag2']]
               if len(match_5400_b) > 2000 else match_5400_b[['tag1','tag2']])
del df_5400, match_5400_b

match_7500_b = df_uniques[df_uniques['tag1'].isin(df_7500['tag']) &
                           df_uniques['tag2'].isin(df_7500['tag'])]
match_7500 = (match_7500_b.sample(2000).reset_index()[['tag1','tag2']]
               if len(match_7500_b) > 2000 else match_7500_b[['tag1','tag2']])
del df_7500, match_7500_b

print(f'bowtie pairs: 4000={len(match_4000):,}  5400={len(match_5400):,} (Fig5 only)  7500={len(match_7500):,}')

In [ ]:
# ── Load spectrum pairs and compute bowtie + flux ratio statistics ────────
# Reads individual spectrum FITS files from MATCH_SPECS_DIR and produces
# two outputs:
#
#   avg_divs_{4000,5400,7500}: single-value flux ratios in a narrow window
#     around each LAMBDA_EFF, used by Fig 5 (pair_hist_all_gauss).
#
#   div_{4000,5400,7500} / wl_{4000,5400,7500}: per-pair full-spectrum
#     smoothed ratio arrays, used by Fig 6 (bowtie_plot_all).
#
# Each spectrum is located by searching all four EW/z subdirectories
# (highew_highz, highew_lowz, lowew_highz, lowew_lowz) under MATCH_SPECS_DIR.
# The plate directory and filename use UNPADDED plate numbers even though
# our internal tags are zero-padded — find_spec_file handles the conversion.
#
# The _build_bowtie helper normalises each spectrum at its LAMBDA_EFF window,
# smooths with a median then uniform filter, divides one spectrum by the
# other (order randomised to avoid systematic sign bias), then bins the
# resulting ratio arrays onto a master wavelength grid. Each pair retains
# its own overlap range; the master grid spans the union of all pairs.
MATCH_SPECS_DIR = '/path/to/your/spectra'  # set this in Section 1
_SPEC_SUBDIRS = ('highew_highz', 'highew_lowz', 'lowew_highz', 'lowew_lowz')

def find_spec_file(tag, spec_dir):
    # Tag is zero-padded internally (e.g. '04650-55648-0737') but the
    # on-disk directory and filename use unpadded plate numbers.
    parts = tag.split('-')
    if len(parts) != 3:
        return None
    plate_unpadded = str(int(parts[0]))
    tag_unpadded = f'{plate_unpadded}-{parts[1]}-{parts[2]}'
    for sub in _SPEC_SUBDIRS:
        p = os.path.join(spec_dir, sub, plate_unpadded, f'spec-{tag_unpadded}.fits')
        if os.path.exists(p):
            return p
    return None

# Wavelength windows (Å) for the avg_divs flux ratio computation (Fig 5).
_AVG_WINDOWS = {4000: (3900, 4100), 5400: (5300, 5500), 7500: (7400, 7500)}

if RUN_SPECTRA_PLOTS and os.path.isdir(MATCH_SPECS_DIR):
    def _build_bowtie(wave1s, wave2s, flux1s, flux2s, norm_centre,
                      norm_window=None, ivar1s=None, ivar2s=None,
                      snr_window=None, snr_cut=0):
        """Build a bowtie (per-pair divided spectra + binned percentile envelope).

        Each spectrum's loglam array is a uniform grid (fixed dex/pixel step
        of 1e-4 dex, per eBOSS-DAP.py), but spectra from different
        plates/runs can have different START values and LENGTHS. The overlap
        range is computed PER PAIR (intersection of that pair's two arms
        only) -- NOT across the whole sample -- so each pair retains its own
        full wavelength coverage. The percentile/median envelope is then
        computed in fixed-width wavelength bins on a master grid spanning
        the union of all pairs' ranges, aggregating only over the pairs that
        actually cover each bin.

        Returns:
            divs    : list of n_pairs 1D arrays, each pair's divided spectrum
                      over its own overlap range
            wave_uni_list : list of n_pairs 1D log10(wavelength) arrays,
                      one per pair, matching divs[i]'s length
            waves, std_top, std_bot, medians : 1D arrays over the master
                      binned grid (for the percentile envelope / vlines)
        """
        if not flux1s:
            return None, None, None, None, None, None, None

        n_pairs = len(flux1s)
        dloglam = 1e-4  # per eBOSS-DAP.py: lam2 = 10**(arange(...)*1E-4 + loglam[0])

        wl = norm_centre
        if norm_window is not None:
            lo, hi = norm_window
        else:
            lo, hi = norm_centre*0.975, norm_centre*1.025

        divs, wave_uni_list = [], []
        for i in range(n_pairs):
            w1, w2 = wave1s[i], wave2s[i]
            f1, f2 = np.asarray(flux1s[i]), np.asarray(flux2s[i])

            pair_start = max(w1[0], w2[0])
            pair_end   = min(w1[-1], w2[-1])
            n_pair = int(round((pair_end - pair_start) / dloglam)) + 1
            if n_pair <= 1:
                continue

            i1 = int(round((pair_start - w1[0]) / dloglam))
            i2 = int(round((pair_start - w2[0]) / dloglam))
            n_pair = min(n_pair, len(f1) - i1, len(f2) - i2)
            if n_pair <= 1:
                continue

            wave_uni_i = pair_start + dloglam * np.arange(n_pair)
            f1r = f1[i1:i1+n_pair]
            f2r = f2[i2:i2+n_pair]

            wl_i = 10**wave_uni_i
            nwl = np.where((wl_i > lo) & (wl_i < hi))[0]
            if nwl.size == 0:
                continue

            if snr_window is not None and ivar1s is not None and ivar2s is not None:
                iv1 = np.asarray(ivar1s[i])[i1:i1+n_pair]
                iv3 = np.asarray(ivar2s[i])[i2:i2+n_pair]
                slo, shi = snr_window
                swl = np.where((wl_i > slo) & (wl_i < shi))[0]
                if swl.size > 0:
                    snr1 = np.nanmean(f1r[swl] * np.sqrt(iv1[swl]))
                    snr2 = np.nanmean(f2r[swl] * np.sqrt(iv3[swl]))
                    if not (snr1 >= snr_cut and snr2 >= snr_cut):
                        continue

            n1 = np.nanmedian(f1r[nwl]); n2 = np.nanmedian(f2r[nwl])
            if not (n1 > 0 and n2 > 0):
                continue

            # Smooth each spectrum with a median then uniform filter before
            # dividing. This suppresses pixel-level noise that would otherwise
            # dominate the bowtie at blue/red wavelengths where S/N is low.
            avg1 = scipy.ndimage.uniform_filter(
                scipy.ndimage.median_filter(f1r/n1, size=100, mode='constant', cval=1.0),
                size=40, mode='constant', cval=1.0)
            avg2 = scipy.ndimage.uniform_filter(
                scipy.ndimage.median_filter(f2r/n2, size=100, mode='constant', cval=1.0),
                size=40, mode='constant', cval=1.0)
            # Randomise the division order so the distribution of log(F1/F2)
            # is symmetric around zero rather than biased toward > 1.
            div_i = avg1/avg2 if np.random.randint(0,2)==1 else avg2/avg1

            divs.append(div_i)
            wave_uni_list.append(wave_uni_i)

        if not divs:
            return None, None, None, None, None, None, None

        master_start = min(w[0] for w in wave_uni_list)
        master_end   = max(w[-1] for w in wave_uni_list)
        n_master = int(round((master_end - master_start) / dloglam)) + 1
        master_grid = master_start + dloglam * np.arange(n_master)

        binning = 15*3  # 45-pixel bins, per eBOSS_error_analysis.ipynb
        x = np.arange(0, n_master, binning)
        if len(x) < 2:
            return None, None, None, None, None, None, None

        waves, std_top, std_bot, medians, n_pairs_bin = [], [], [], [], []
        for j, ix in enumerate(x[1:]):
            bin_lo = master_grid[x[j]]
            bin_hi = master_grid[ix]
            vals = []
            for div_i, wave_uni_i in zip(divs, wave_uni_list):
                mask = (wave_uni_i >= bin_lo) & (wave_uni_i < bin_hi)
                if mask.any():
                    vals.append(div_i[mask])
            waves.append((bin_lo + bin_hi) / 2)
            n_pairs_bin.append(len(vals))
            if vals:
                slc = np.concatenate(vals)
                std_top.append(np.nanpercentile(slc, 68.27+15.7))
                std_bot.append(np.nanpercentile(slc, 15.7))
                medians.append(np.nanmedian(slc))
            else:
                std_top.append(np.nan)
                std_bot.append(np.nan)
                medians.append(np.nan)

        return (divs, wave_uni_list, np.array(waves), np.array(std_top),
                np.array(std_bot), np.array(medians), np.array(n_pairs_bin))

    # Pool all unique pairs across the three LAMBDA_EFF groups. Each spectrum
    # is loaded only once regardless of how many groups it appears in.
    all_pairs = pd.concat([match_4000, match_5400, match_7500]).drop_duplicates().reset_index(drop=True)
    match_4000_set = set(zip(match_4000['tag1'], match_4000['tag2']))
    match_5400_set = set(zip(match_5400['tag1'], match_5400['tag2']))
    match_7500_set = set(zip(match_7500['tag1'], match_7500['tag2']))
    del match_4000, match_5400, match_7500

    raw_wave1s_4000,raw_wave2s_4000,raw_flux1s_4000,raw_flux2s_4000 = [],[],[],[]
    raw_wave1s_5400,raw_wave2s_5400,raw_flux1s_5400,raw_flux2s_5400 = [],[],[],[]
    raw_ivar1s_5400,raw_ivar2s_5400 = [],[]
    raw_wave1s_7500,raw_wave2s_7500,raw_flux1s_7500,raw_flux2s_7500 = [],[],[],[]
    avg_divs_4000, avg_divs_5400, avg_divs_7500 = [], [], []

    for _, row in all_pairs.iterrows():
        t1 = str(row['tag1']).strip()
        t2 = str(row['tag2']).strip()
        f1path = find_spec_file(t1, MATCH_SPECS_DIR)
        f2path = find_spec_file(t2, MATCH_SPECS_DIR)
        if f1path is None or f2path is None:
            continue
        try:
            d1 = fits.getdata(f1path, 1)
            d2 = fits.getdata(f2path, 1)
        except Exception:
            continue

        wl1 = 10**d1['loglam']
        wl2 = 10**d2['loglam']
        # Floor negative flux at a small positive value to prevent division
        # by zero or sign changes from swamping the ratio distributions.
        f1 = np.where(d1['flux'] > 1e-3, d1['flux'], 1e-3)
        f2 = np.where(d2['flux'] > 1e-3, d2['flux'], 1e-3)

        # ── avg_divs: single flux ratio near this pair's LAMBDA_EFF ─────────
        # Each pair's avg_div is computed at its OWN LAMBDA_EFF window only.
        # Cross-contaminating (e.g. a 7500-Å pair into avg_divs_4000) gives
        # spurious outliers because blue flux is near-zero for red-optimised
        # spectra.
        pair_key = (t1, t2)
        in_4000 = pair_key in match_4000_set
        in_5400 = pair_key in match_5400_set
        in_7500 = pair_key in match_7500_set
        centre = 4000 if in_4000 else (5400 if in_5400 else (7500 if in_7500 else None))

        if centre is not None:
            lo, hi = _AVG_WINDOWS[centre]
            nwl1 = (wl1 > lo) & (wl1 < hi)
            nwl2 = (wl2 > lo) & (wl2 < hi)
            if nwl1.any() and nwl2.any():
                m1, m2 = np.median(f1[nwl1]), np.median(f2[nwl2])
                if m1 > 0 and m2 > 0:
                    avg_div = m1/m2 if np.random.randint(0, 2) == 1 else m2/m1
                    {4000: avg_divs_4000, 5400: avg_divs_5400, 7500: avg_divs_7500}[centre].append(avg_div)

        # ── Collect raw arrays for the bowtie builder ─────────────────────
        # Stored as lists of per-spectrum arrays; _build_bowtie handles the
        # variable-length alignment internally.
        if in_4000:
            raw_wave1s_4000.append(np.log10(wl1).tolist())
            raw_wave2s_4000.append(np.log10(wl2).tolist())
            raw_flux1s_4000.append(f1.tolist())
            raw_flux2s_4000.append(f2.tolist())
        if in_5400:
            raw_wave1s_5400.append(np.log10(wl1).tolist())
            raw_wave2s_5400.append(np.log10(wl2).tolist())
            raw_flux1s_5400.append(f1.tolist())
            raw_flux2s_5400.append(f2.tolist())
            # ivars needed for the S/N >= 5 pre-filter on the 5400 bowtie.
            raw_ivar1s_5400.append(d1['ivar'].tolist())
            raw_ivar2s_5400.append(d2['ivar'].tolist())
        if in_7500:
            raw_wave1s_7500.append(np.log10(wl1).tolist())
            raw_wave2s_7500.append(np.log10(wl2).tolist())
            raw_flux1s_7500.append(f1.tolist())
            raw_flux2s_7500.append(f2.tolist())

    del all_pairs, match_4000_set, match_5400_set, match_7500_set

    div_4000, wl_4000, waves_4000, std_top_4000, std_bot_4000, medians_4000, npairs_4000 = _build_bowtie(
        raw_wave1s_4000, raw_wave2s_4000, raw_flux1s_4000, raw_flux2s_4000, 4000,
        norm_window=(3900, 4100))
    div_5400, wl_5400, waves_5400, std_top_5400, std_bot_5400, medians_5400, npairs_5400 = _build_bowtie(
        raw_wave1s_5400, raw_wave2s_5400, raw_flux1s_5400, raw_flux2s_5400, 5400,
        norm_window=(5300, 5500), ivar1s=raw_ivar1s_5400, ivar2s=raw_ivar2s_5400,
        snr_window=(4000, 6000), snr_cut=5)
    div_7500, wl_7500, waves_7500, std_top_7500, std_bot_7500, medians_7500, npairs_7500 = _build_bowtie(
        raw_wave1s_7500, raw_wave2s_7500, raw_flux1s_7500, raw_flux2s_7500, 7500,
        norm_window=(7400, 7600))
    del raw_wave1s_4000,raw_wave2s_4000,raw_flux1s_4000,raw_flux2s_4000
    del raw_wave1s_5400,raw_wave2s_5400,raw_flux1s_5400,raw_flux2s_5400
    del raw_ivar1s_5400, raw_ivar2s_5400
    del raw_wave1s_7500,raw_wave2s_7500,raw_flux1s_7500,raw_flux2s_7500

    avg_divs_4000 = np.array(avg_divs_4000)
    avg_divs_5400 = np.array(avg_divs_5400)
    avg_divs_7500 = np.array(avg_divs_7500)

    print('Pairs loaded  4000:', 0 if div_4000 is None else len(div_4000),
          '  5400:', 0 if div_5400 is None else len(div_5400),
          '  7500:', 0 if div_7500 is None else len(div_7500))
    print('  (5400 panel applies an SNR>=5 cut over 4000-6000 A, per eBOSS_error_analysis.ipynb)')
    print('avg_divs counts  4000:', len(avg_divs_4000),
          '  5400:', len(avg_divs_5400), '  7500:', len(avg_divs_7500))
else:
    print('Skipping Figs 5 & 6 spectrum loading: RUN_SPECTRA_PLOTS is False or the '
          f'spectra library was not found at {MATCH_SPECS_DIR!r}.')
    div_4000 = div_5400 = div_7500 = None
    wl_4000 = wl_5400 = wl_7500 = None
    waves_4000 = std_top_4000 = std_bot_4000 = medians_4000 = None
    waves_5400 = std_top_5400 = std_bot_5400 = medians_5400 = None
    waves_7500 = std_top_7500 = std_bot_7500 = medians_7500 = None
    avg_divs_4000 = np.array([])
    avg_divs_5400 = np.array([])
    avg_divs_7500 = np.array([])

In [ ]:
if RUN_SPECTRA_PLOTS and os.path.isdir(MATCH_SPECS_DIR):
    # ── Figure 5: pair_hist_all_gauss ────────────────────────────────────────
    # Three-panel histogram of flux ratios at each LAMBDA_EFF, with a Gaussian
    # fit overlaid. The fitted sigma is the minimum spectrophotometric floor
    # on any line flux measurement at that wavelength.
    plot_prettier(150, fontsize=24)
    fig, axes = plt.subplot_mosaic("AAA;BBB;CCC", figsize=(11, 8.5*1.5))
    fig.tight_layout(pad=-1.0)

    fitfunc = lambda p, x: p[0]*np.exp(-0.5*((x-p[1])/p[2])**2) + p[3]
    errfunc = lambda p, x, y: (y - fitfunc(p, x))

    xmin, xmax = -0.1, 2.1

    panels = [('A', avg_divs_4000, 100), ('B', avg_divs_5400, 100), ('C', avg_divs_7500, 100)]
    sigs = {}
    for label, data, nbin in panels:
        counts, bins = np.histogram(data, bins=nbin)
        # Use a robust (16th/84th percentile) width estimate for the initial
        # sigma guess. np.nanstd is inflated by outlier tails and can push
        # leastsq to a degenerate negative-amplitude fit.
        sigma_init = max((np.nanpercentile(data, 84) - np.nanpercentile(data, 16)) / 2, 1e-3)
        init = [np.nanmax(counts), np.nanmedian(data), sigma_init, 0.0]
        out = opt.leastsq(errfunc, init, args=(bins[:-1], counts))
        c = out[0]
        x_fit = np.arange(xmin, xmax, 0.01)
        y_fit = c[0]*np.exp(-0.5*((x_fit-c[1])/c[2])**2) + c[3]
        ax = axes[label]
        ax.hist(data, bins=nbin)
        ax.plot(x_fit, y_fit, lw=10)
        ax.set_xlim(xmin, xmax)
        ax.vlines(1, 0, np.nanmax(counts)+5, alpha=1.0, lw=3, zorder=50, color='k', ls='-.')
        ax.set_ylim(0, np.nanmax(counts)+5)
        sigs[label] = abs(c[2])

    axes['A'].set_xticklabels([])
    axes['B'].set_xticklabels([])
    axes['C'].set_xlabel('Flux 1 / Flux 2', fontsize=36)
    axes['B'].set_ylabel('Number of galaxies', fontsize=50)

    linelabels = [
        f'Lambda_eff = 4000;\n sigma = {sigs["A"]:.2f}',
        f'Lambda_eff = 5400;\n sigma = {sigs["B"]:.2f}',
        f'Lambda_eff = 7500;\n sigma = {sigs["C"]:.2f}',
    ]
    for i, (label, ax) in enumerate(axes.items()):
        ax.annotate(
            label + ': ' + linelabels[i],
            xy=(1,1), xycoords='axes fraction',
            xytext=(-1.25,-2.75), textcoords='offset fontsize',
            fontsize=24, va='bottom', ha='right', fontfamily='serif',
            bbox=dict(pad=8.0, facecolor='white', edgecolor='black'))

    plt.subplots_adjust(wspace=0.07, hspace=0.02)
    plt.savefig(os.path.join(OUTPUT_DIR,'pair_hist_all_gauss.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(OUTPUT_DIR,'pair_hist_all_gauss.png'), dpi=150, bbox_inches='tight')
    plt.show()
    if 'sigs' in dir():
        del sigs
    del avg_divs_4000, avg_divs_5400, avg_divs_7500
else:
    print('Skipping Figure 5 (pair_hist_all_gauss): RUN_SPECTRA_PLOTS is False or the '
          f'spectra library was not found at {MATCH_SPECS_DIR!r}.')

In [ ]:
if RUN_SPECTRA_PLOTS and os.path.isdir(MATCH_SPECS_DIR):
    # ── Figure 6: bowtie_plot_all ────────────────────────────────────────────
    # Three-panel plot of smoothed spectral ratios as a function of observed
    # wavelength. Each grey line is one pair's divided spectrum. The orange
    # percentile dots show the 1-sigma envelope, and the coloured vertical
    # lines mark where the calibration error first exceeds 10%, 25%, and 50%
    # scanning outward from LAMBDA_EFF. Each panel has its OWN xlim (not
    # shared), since the covered wavelength range differs by LAMBDA_EFF.
    plot_prettier(150, fontsize=24)
    scale = 3
    plt.rcParams.update({'xtick.major.size': 5*scale, 'xtick.major.width': 1.25*scale,
                          'xtick.minor.size': 2.5*scale, 'xtick.minor.width': 1.25*scale,
                          'ytick.direction': 'in',
                          'ytick.major.size': 5*scale, 'ytick.major.width': 1.25*scale,
                          'ytick.minor.size': 2.5*scale, 'ytick.minor.width': 1.25*scale})

    pointsize = 18*scale
    dotcolor = 'C1'

    fig, axes = plt.subplot_mosaic("AAA;BBB;CCC", figsize=(11*scale, 8.5*scale*1))
    fig.tight_layout(pad=-1.0)

    panel_data = {
        'A': (div_4000, wl_4000, waves_4000, std_top_4000, std_bot_4000, medians_4000, npairs_4000, 4000, 'Lambda_eff = 4000'),
        'B': (div_5400, wl_5400, waves_5400, std_top_5400, std_bot_5400, medians_5400, npairs_5400, 5400, 'Lambda_eff = 5400'),
        'C': (div_7500, wl_7500, waves_7500, std_top_7500, std_bot_7500, medians_7500, npairs_7500, 7500, 'Lambda_eff = 7500'),
    }

    for panel, (div_arr, wave_uni, waves, std_top, std_bot, medians, n_pairs_bin, lam_eff, label) in panel_data.items():
        ax = axes[panel]
        ax.set_ylim(0, 2)

        if div_arr is None:
            ax.text(0.5, 0.5, 'No spectra passed the SNR cut for this lambda_eff',
                    transform=ax.transAxes, ha='center', va='center', fontsize=18, color='grey')
        else:
            for div_i, wave_uni_i in zip(div_arr, wave_uni):
                ax.plot(10**wave_uni_i, div_i, alpha=0.25, c='grey', lw=1, zorder=50, rasterized=True)
            ax.scatter(10**waves, std_top, c=dotcolor, s=pointsize,
                       label='16th percentile / 84th percentile', zorder=75)
            ax.scatter(10**waves, std_bot, c=dotcolor, s=pointsize, zorder=75)
            ax.hlines(1, 10**waves[1], 10**waves[-3], color='C3', zorder=100)
            ax.set_xlim(10**waves[1], 10**waves[-2])

            # Scan outward from LAMBDA_EFF to find where calibration error
            # first exceeds each floor. Scanning outward (rather than a
            # global np.where) avoids picking up spurious noise spikes in
            # edge bins that can fall below the floor by chance.
            noise = ((std_top - medians) + (medians - std_bot)) / 2
            centre_idx = np.argmin(np.abs((10**waves) - lam_eff))

            for floor, color in [(10, 'g'), (25, 'y'), (50, 'r')]:
                left_idx = centre_idx
                for j in range(centre_idx, -1, -1):
                    if noise[j] * 100 < floor:
                        left_idx = j
                    else:
                        break
                right_idx = centre_idx
                for j in range(centre_idx, len(noise)):
                    if noise[j] * 100 < floor:
                        right_idx = j
                    else:
                        break
                ax.vlines(np.round((10**waves)[left_idx]),  0, 2, color=color, ls='-.', lw=4, zorder=10)
                ax.vlines(np.round((10**waves)[right_idx]), 0, 2, color=color, ls='-.', lw=4, zorder=10)

        ax.annotate(label, xy=(1, 0), xycoords='axes fraction',
                    xytext=(-0.5, 0.5), textcoords='offset fontsize',
                    fontsize=22, va='bottom', ha='right', fontfamily='serif',
                    bbox=dict(facecolor='white', edgecolor='black', pad=6.0), zorder=100)
        ax.set_ylabel('Flux 1 / Flux 2', fontsize=48)

    axes['A'].set_xticklabels([])
    axes['B'].set_xticklabels([])
    axes['C'].set_xlabel('Observed Wavelength (\u00c5)', fontsize=48)
    axes['C'].legend(loc=1, fontsize=36)

    plt.savefig(os.path.join(OUTPUT_DIR,'bowtie_plot_all.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(OUTPUT_DIR,'bowtie_plot_all.png'), dpi=150, bbox_inches='tight')
    plt.show()
else:
    print('Skipping Figure 6 (bowtie_plot_all): RUN_SPECTRA_PLOTS is False or the '
          f'spectra library was not found at {MATCH_SPECS_DIR!r}.')

In [ ]:
if RUN_SPECTRA_PLOTS and os.path.isdir(MATCH_SPECS_DIR):
    # ── Bowtie summary tables (released alongside the paper, per Data Avail.) ─
    # Bins the per-pair ratio arrays onto the same master wavelength grid used
    # for the bowtie plot and writes per-bin percentile statistics to CSV.
    def make_wavelength_table(div_arr, wave_uni_list, binning=45, dloglam=1e-4):
        """div_arr/wave_uni_list: lists of per-pair 1D arrays (variable length,
        each pair's own overlap range). Bins on a master grid spanning the
        union of all pairs' ranges, aggregating only the pairs that cover
        each bin."""
        master_start = min(w[0] for w in wave_uni_list)
        master_end   = max(w[-1] for w in wave_uni_list)
        n_master = int(round((master_end - master_start) / dloglam)) + 1
        master_grid = master_start + dloglam * np.arange(n_master)

        x = np.arange(0, n_master, binning)
        rows = []
        for i, ix in enumerate(x[1:]):
            bin_lo = master_grid[x[i]]
            bin_hi = master_grid[ix]
            wcen = (bin_lo + bin_hi) / 2
            vals = []
            for div_i, wave_uni_i in zip(div_arr, wave_uni_list):
                mask = (wave_uni_i >= bin_lo) & (wave_uni_i < bin_hi)
                if mask.any():
                    vals.append(div_i[mask])
            slc = np.concatenate(vals) if vals else np.array([np.nan])
            rows.append({
                'wave_bin_log10': wcen,
                'wave_bin_Ang':   10**wcen,
                'median':         np.nanmedian(slc),
                'scatter':        (abs(np.nanpercentile(slc,83.97)-np.nanmedian(slc)) +
                                   abs(np.nanmedian(slc)-np.nanpercentile(slc,15.7)))/2,
                'p16':   np.nanpercentile(slc, 15.7),
                'p84':   np.nanpercentile(slc, 83.97),
                'p023':  np.nanpercentile(slc, 2.3),
                'p977':  np.nanpercentile(slc, 97.7),
            })
        return pd.DataFrame(rows)

    for lam, div_arr, wu in [(4000,div_4000,wl_4000),(5400,div_5400,wl_5400),(7500,div_7500,wl_7500)]:
        if div_arr is None: continue
        tbl = make_wavelength_table(div_arr, wu)
        tbl.to_csv(os.path.join(OUTPUT_DIR, f'lameff_{lam}_spectrophotometric_error_table.csv'), index=False)
        print(f'lambda_eff={lam} table:')
        print(tbl.to_latex(index=False, float_format='%.4f'))
    print('Bowtie summary tables saved.')
    del div_4000, div_5400, div_7500, wl_4000, wl_5400, wl_7500
else:
    print('Skipping bowtie summary tables: RUN_SPECTRA_PLOTS is False or the '
          f'spectra library was not found at {MATCH_SPECS_DIR!r}.')

*Cleanup cell for Section 7 -- run after Figs 5 & 6 are finalized. Cells 23-26 can be re-run freely before this without NameErrors.*

In [ ]:
# ── Section 6 cleanup ────────────────────────────────────────────────────
# Consolidated cleanup for all state produced by the Section 6 cells.
# Run this once Figs 5 & 6 are finalised. It's separated so the plotting
# cells can be re-run individually without hitting NameErrors.
_section6_vars = [
    'match_4000_set', 'match_5400_set', 'match_7500_set',
    'avg_divs_4000', 'avg_divs_5400', 'avg_divs_7500',
    'div_4000', 'div_5400', 'div_7500',
    'wl_4000', 'wl_5400', 'wl_7500',
    'waves_4000', 'waves_5400', 'waves_7500',
    'std_top_4000', 'std_top_5400', 'std_top_7500',
    'std_bot_4000', 'std_bot_5400', 'std_bot_7500',
    'medians_4000', 'medians_5400', 'medians_7500',
    'panel_data', 'sigs',
]
for _v in _section6_vars:
    if _v in dir():
        exec(f'del {_v}')
del _section6_vars, _v

# Section 7 — Figure 7: Absolute Spectrophotometric Calibration
### *Checking the colours: do the spectra agree with the photometry?*
*Compares synthetic g−r and r−i colours derived from the spectra against photometric colours for bright (r < 17.7) galaxies with `LAMBDA_EFF` = 5400 Å. Uses `df5400_gr` and `df5400_ri` built in Section 6, so that section must have been run first. The Gaussian fit parameters (σ, μ) are what go into the paper. **Requires:** Sections 0, 1, and 6.*

**Output:** `abs_cal_plot_5400_colorcut.pdf` / `.png`

In [ ]:
# ── Figure 7: abs_cal_plot_5400_colorcut ─────────────────────────────────
# Two-panel comparison of photometric vs spectroscopic g-r (top) and r-i
# (bottom) colours for bright LAMBDA_EFF=5400 galaxies. df5400_gr and
# df5400_ri were built in Section 6. The Gaussian is fitted to the
# colour-difference distribution; the fitted mu and sigma are the paper's
# absolute calibration accuracy numbers. A data-driven sigma init (robust
# 16th/84th percentile width) is used to prevent leastsq converging to a
# degenerate negative-amplitude solution on narrow distributions.
plot_prettier(150, fontsize=24)
fig, axes = plt.subplot_mosaic("A;B", figsize=(11*2.2/2, 8.5*2.2*2/3))
fig.tight_layout(pad=-1.0)

fitfunc = lambda p, x: p[0]*np.exp(-0.5*((x-p[1])/p[2])**2) + p[3]
errfunc = lambda p, x, y: (y - fitfunc(p, x))

xmin, xmax = -0.4, 0.4
nbin = 100

panels = [('A', df5400_gr, 'color_comp_gr'), ('B', df5400_ri, 'color_comp_ri')]
sigs, mus = {}, {}
for label, dframe, col in panels:
    data_init = dframe[dframe[col].astype(float) > xmin]
    data = data_init[data_init[col].astype(float) < xmax][col]
    counts, bins = np.histogram(data, bins=nbin)
    sigma_init = max((np.nanpercentile(data, 84) - np.nanpercentile(data, 16)) / 2, 1e-3)
    init = [np.nanmax(counts), np.nanmedian(data), sigma_init, 0.0]
    out = opt.leastsq(errfunc, init, args=(bins[:-1], counts))
    c = out[0]
    x_fit = np.arange(xmin, xmax, 0.01)
    y_fit = c[0]*np.exp(-0.5*((x_fit-c[1])/c[2])**2) + c[3]
    ax = axes[label]
    ax.hist(data, bins=nbin)
    ax.plot(x_fit, y_fit, lw=10)
    ax.set_xlim(xmin, xmax)
    # Clip the y-axis to the larger of the histogram peak and the fitted
    # curve's peak, so the fit is never visually truncated.
    ymax = max(np.nanmax(counts), np.nanmax(y_fit)) + 5
    ax.vlines(0, 0, ymax, alpha=1.0, lw=3, zorder=50, color='k', ls='-.')
    ax.set_ylim(0, ymax)
    sigs[label] = abs(c[2])
    mu_r = np.round(c[1], decimals=2)
    mus[label] = 0.0 if mu_r == 0 else mu_r

axes['A'].set_xticklabels([])

linelabels = [
    f'(g - r) color ;\n mu = {mus["A"]} ; sigma = {sigs["A"]:.2f}',
    f'(r - i) color ;\n mu = {mus["B"]} ; sigma = {sigs["B"]:.2f}',
]
for i, (label, ax) in enumerate(axes.items()):
    ax.annotate(
        linelabels[i],
        xy=(1, 1), xycoords='axes fraction',
        xytext=(-0.5, -2.75), textcoords='offset fontsize',
        fontsize=18, va='bottom', ha='right', fontfamily='serif',
        bbox=dict(facecolor='white', edgecolor='black', pad=6.0))

plt.subplots_adjust(wspace=0.11, hspace=0.01)
fig.supxlabel('Photometric Color - Spectroscopic Color', y=-0.09, fontsize=36)
fig.supylabel('Number of Galaxies in Sub-Sample', x=-0.12, fontsize=36)

plt.savefig(os.path.join(OUTPUT_DIR,'abs_cal_plot_5400_colorcut.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'abs_cal_plot_5400_colorcut.png'), dpi=150, bbox_inches='tight')
plt.show()
del df5400_gr, df5400_ri, sigs, mus

# Section 8 — Figure 9: Clipped Emission-Line Spectrum
### *An example of the pipeline eating its own homework.*
*Plots a single pathological spectrum — a high-EW, low-dispersion galaxy where the strongest emission lines have been clipped by the `idlspec2D` reduction pipeline and flagged as bad data. The figure shows flux, error, and inverse variance, with zoomed insets on [O III] 5008 and Hα to make the clipping visible. The spectrum is read directly from disk; no prior section needed beyond Section 0. Figure 8 (the photometric cutout) is a FITS-viewer screenshot and is not generated here. **Self-contained:** needs only Section 0.*

**Output:** `cutoff_spec.pdf` / `.png`

In [ ]:
# ── Figure 9: cutoff_spec ────────────────────────────────────────────────
# Plots a single eBOSS spectrum with clipped emission lines, with zoomed
# insets at [O III] 5008 and Hα to make the clipping (ivar=0 regions)
# visible. The spectrum is spec-11294-58451-0164.fits — a high-EW,
# low-dispersion galaxy where idlspec2D masked the line peaks as bad data.
# The figure is shown in rest-frame wavelength using the pipeline redshift.
from mpl_toolkits.axes_grid1.inset_locator import zoomed_inset_axes, mark_inset

cutoff_path = os.path.join(WORKING_DIR, 'spectra', 'plotting', 'spec-11294-58451-0164.fits')

if os.path.exists(cutoff_path):
    data_c, err_c = 'C0', 'lightgrey'
    cutoff_hdu = fits.open(cutoff_path)
    flux = cutoff_hdu[1].data['flux']
    var  = cutoff_hdu[1].data['ivar']
    loglam = cutoff_hdu[1].data['loglam']
    obs_wave = 10**loglam
    try:
        z_spec = float(cutoff_hdu[2].data['Z'][0])
    except Exception:
        z_spec = 0.094  # Plate 11294, MJD 58451, Fiber 164
    cutoff_hdu.close()
    wave = obs_wave / (1 + z_spec)
    window_width = 100

    oiii_obs, halpha_obs = 5007, 6562.8

    plot_prettier(150, fontsize=18)
    fig, ax = plt.subplots(figsize=(11, 8.5/2))
    ax.plot(wave, flux, c=data_c)
    ax.fill_between(wave, flux - np.sqrt(1/var), flux + np.sqrt(1/var), color=err_c, alpha=0.25)
    ax.set_xlim(wave[0], wave[-1])
    ax.set_xlabel('Rest Wavelength (\u00c5)')
    ax.set_ylabel(r'Flux [$10^{-17}$ erg/s/cm$^2$/\u00c5]')

    # Zoomed inset centred on [O III] 5008. The ivar=0 region (clipped line)
    # is visible as an orange spike (ivar=0 → 1/ivar → inf; orange line
    # plotted on top of flux to show the mask coincides with the line centre).
    axins1 = zoomed_inset_axes(ax, 8, loc='upper center',
                                bbox_to_anchor=(0.25, 0.65, 0.3, 0.3), bbox_transform=ax.transAxes)
    axins1.plot(wave, flux, c=data_c)
    axins1.plot(wave, var, c='orange')
    axins1.fill_between(wave, flux - np.sqrt(1/var), flux + np.sqrt(1/var), color=err_c, alpha=0.25)
    axins1.grid(False)
    axins1.set_xlim(oiii_obs - window_width, oiii_obs + window_width)
    axins1.set_ylim(0, 25)

    # Zoomed inset centred on Hα.
    axins2 = zoomed_inset_axes(ax, 8, loc='upper right',
                                bbox_to_anchor=(0.7, 0.7, 0.3, 0.3), bbox_transform=ax.transAxes)
    axins2.plot(wave, flux, c=data_c)
    axins2.plot(wave, var, c='orange')
    axins2.fill_between(wave, flux - np.sqrt(1/var), flux + np.sqrt(1/var), color=err_c, alpha=0.25)
    axins2.grid(False)
    axins2.set_xlim(halpha_obs - window_width, halpha_obs + window_width)
    axins2.set_ylim(0, 25)

    axins1.tick_params(axis='both', labelsize=10)
    axins2.tick_params(axis='both', labelsize=10)

    mark_inset(ax, axins1, loc1=2, loc2=4, fc='none', ec='0.5', alpha=0.5)
    mark_inset(ax, axins2, loc1=2, loc2=4, fc='none', ec='0.5', alpha=0.5)

    plt.savefig(os.path.join(OUTPUT_DIR,'cutoff_spec.pdf'), dpi=300, bbox_inches='tight')
    plt.savefig(os.path.join(OUTPUT_DIR,'cutoff_spec.png'), dpi=150, bbox_inches='tight')
    plt.show()
    del flux, var, loglam, obs_wave, wave
else:
    print(f'WARNING: spectrum not found at {cutoff_path!r}')

# Section 9 — Figure 10: C3K SSP Templates
### *Meet the models doing the continuum fitting.*
*Plots the C3K simple stellar population template library used by the eBOSS-DAP: solar-metallicity SSPs at 20 ages (top panel) and four metallicities at a fixed age of 653 Myr (bottom panel), all normalised at ~3500 Å. Also overplots two continuous star-formation models. The templates are read directly from `C3K_DIR`; no catalog data needed. This is a methods figure — its purpose is to show what the fitter actually has to work with. **Self-contained:** needs only Section 0.*

**Output:** `ssps_plots.pdf` / `.png`

In [ ]:
# ── Figure 10: C3K SSP templates ─────────────────────────────────────────
# Top panel: solar-metallicity SSPs at 20 ages, colour-coded by log(age).
# Bottom panel: four metallicities at a fixed age of 653 Myr.
# All spectra are normalised at NORM_IDX (~3500 Å) so differences in
# spectral shape rather than normalisation are what the reader sees.
# The two youngest entries (5 and 10 Myr) are continuous star-formation
# models with nebular continuum added; the rest are single-burst SSPs.
# File naming convention: SpecAge{AA}-{BBBB}Met{m}-{f}.fits for SSPs,
# SpecAgeC-{age_Myr*10:04d}Met1-0+nebcont.fits for continuous models.
# The on-disk wavelength grid is log10(wavelength); x = 10**wavearr.

NORM_IDX = 6029  # index corresponding to ~3500 Å

ssp_ages = [
    (0.0050, '5.0 Myr Continuous'), (0.0100, '10.0 Myr Continuous'),
    (0.0121, '12.1 Myr'), (0.0138, '13.8 Myr'), (0.0150, '15.0 Myr'),
    (0.0199, '19.9 Myr'), (0.0250, '25.0 Myr'), (0.0328, '32.8 Myr'),
    (0.0540, '54.0 Myr'), (0.0889, '88.9 Myr'), (0.1463, '146 Myr'),
    (0.2409, '241 Myr'),  (0.3965, '397 Myr'),  (0.6528, '653 Myr'),
    (1.0748, '1.08 Gyr'), (1.7695, '1.77 Gyr'), (2.9132, '2.91 Gyr'),
    (4.7962, '4.80 Gyr'), (7.8962, '7.90 Gyr'), (13.0000, '13.0 Gyr'),
]

mets       = [0.1, 0.5, 1.0, 2.5]
met_labels = ['0.1 * solar metalicity', '0.5 * solar metalicity',
               '1.0 * solar metalicity', '2.5 * solar metalicity']
age_mid = 0.6528  # Gyr (653 Myr) — the fixed age for the metallicity panel

def _ssp_filename(age_gyr, met):
    if age_gyr < 0.0105 and age_gyr in (0.0050, 0.0100):
        age_str = f'C-{int(round(age_gyr*1000*10)):04d}'
        suffix = '+nebcont'
    else:
        age_str = f'{int(age_gyr):02d}-{round((age_gyr - int(age_gyr))*10000):04d}'
        suffix = ''
    met_str = f'{int(met)}-{int(round((met - int(met))*10))}'
    return os.path.join(C3K_DIR, f'SpecAge{age_str}Met{met_str}{suffix}.fits')

def _load_spec(fname):
    if not os.path.exists(fname):
        return None, None
    hdu = fits.open(fname)
    nbins = hdu[0].header['NAXIS1']
    wavearr = hdu[0].header['CRVAL1'] + hdu[0].header['CDELT1'] * np.arange(nbins)
    spec = hdu[0].data
    hdu.close()
    return 10**wavearr, spec

# Check that C3K_DIR is set correctly before running the (slow) plot loop.
if not os.path.isdir(C3K_DIR):
    print(f'WARNING: C3K_DIR not found: {C3K_DIR!r}')
    print('  Adjust C3K_DIR in Section 1 to point at your c3k templates directory.')
else:
    _c3k_files = [f for f in os.listdir(C3K_DIR) if f.endswith('.fits')]
    print(f'C3K_DIR = {C3K_DIR!r}  ({len(_c3k_files)} .fits files found)')
    _test_path = _ssp_filename(age_mid, 1.0)
    if not os.path.exists(_test_path):
        print(f'WARNING: expected file not found: {_test_path!r}')
        print('  Sample files actually present:', sorted(_c3k_files)[:10])
    del _c3k_files, _test_path

plot_prettier(150, fontsize=24)
fig, axes = plt.subplot_mosaic("AAA;BBB", figsize=(33, 25.5))
fig.tight_layout(pad=-1.0)

# ── Top panel: solar metallicity, all ages, viridis colour-coded by log age ─
lages = np.log10(np.array([a for a, _ in ssp_ages]) * 1000)
d = (lages - lages.min()) / (lages.max() - lages.min())
cgrad = mpl.cm.get_cmap('viridis')(d)

for (age_gyr, label), col in zip(ssp_ages, cgrad):
    wave, spec = _load_spec(_ssp_filename(age_gyr, 1.0))
    if wave is None:
        continue
    axes['A'].plot(wave, spec/spec[NORM_IDX], label=label, c=col, rasterized=True)

axes['A'].set_ylim(1e-3, 1e2)
axes['A'].set_yscale('log')
axes['A'].set_xticklabels([])
axes['A'].legend(loc=1, ncol=7)
axes['A'].set_ylabel('Normalized Flux', fontsize=28)

# ── Bottom panel: fixed age (653 Myr), four metallicities, plasma colour ──
lmets = np.log10(np.array(mets))
d = (lmets - lmets.min()) / (lmets.max() - lmets.min())
cgrad = mpl.cm.get_cmap('plasma')(d)

for met, label, col in zip(mets, met_labels, cgrad):
    wave, spec = _load_spec(_ssp_filename(age_mid, met))
    if wave is None:
        continue
    axes['B'].plot(wave, spec/spec[NORM_IDX], label=label, c=col, rasterized=True)

axes['B'].set_ylim(10**-2.2, 10)
axes['B'].set_yscale('log')
axes['B'].legend(loc=1, ncol=1)
axes['B'].set_xlabel('Restframe Wavelength (\u00c5)', fontsize=28)
axes['B'].set_ylabel('Normalized Flux', fontsize=28)

linelabels = ['Solar Metalicities', f'Age = {age_mid*1000:.0f} Myr']
for i, (label, ax) in enumerate(axes.items()):
    ax.annotate(
        label + ': ' + linelabels[i],
        xy=(1, 0), xycoords='axes fraction',
        xytext=(-0.25, +1.25), textcoords='offset fontsize',
        fontsize='large', va='bottom', ha='right', fontfamily='serif',
        bbox=dict(facecolor='none', edgecolor='none', pad=3.0))

plt.subplots_adjust(wspace=0.03, hspace=0.02)
plt.savefig(os.path.join(OUTPUT_DIR,'ssps_plots.pdf'), bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'ssps_plots.png'), dpi=150, bbox_inches='tight')
plt.show()

# Section 10 — Figure 11: Nebular Continuum Comparison
### *What happens when you forget that ionised gas glows on its own.*
*Compares DAP continuum fits with and without nebular continuum emission for a stacked high-EW spectrum, including a second panel where high-order Balmer line flux is added back by scaling from H12. The figure makes the case that nebular continuum is not optional for star-forming galaxies: leaving it out misattributes flux both sides of the Balmer jump and pulls [O II] 3727 down with it. Reads test FITS outputs from `spectra/test_cneb/` and `spectra/test_nneb/` — note that `test_nneb/` may not yet be present on all machines (see in-cell comments). Also requires `gas_template.csv` for the Balmer-series template. **Self-contained:** needs only Section 0, but requires the test spectrum files on disk.*

**Output:** `stackfitmew1.pdf` / `.png`

In [ ]:
# ── Figure 11: nebular continuum comparison ───────────────────────────────
# Compares DAP continuum fits with and without nebular continuum for a
# stacked spectrum of medium-EW galaxies (fittag='mew1', plotbin='02').
#
# Two sets of DAP outputs are needed:
#   FIT_CNEB: fit result from templates WITH nebular continuum
#   FIT_NNEB: fit result from templates WITHOUT nebular continuum
#   DATA_SPEC: the stacked observed spectrum
#
# Panel A (top, 'B' in subplot_mosaic): full model flux vs data.
# Panel B (bottom, 'A' in subplot_mosaic): stellar continuum with high-order
#   Balmer lines added back by scaling the gas template to the measured H12
#   flux (Case B scaling). This panel makes the case that nebular continuum
#   is required to correctly recover the continuum near the Balmer jump.
#
# NOTE: subplot_mosaic("B;A") means axes['B'] is the TOP panel and axes['A']
# is the BOTTOM. The paper labels them A (top) and B (bottom) — the loop
# reverses the annotation order to match.
#
# spectra/test_nneb/ may not yet be present on all machines. If FIT_NNEB
# is missing, the cell prints a warning and skips rather than crashing.
plotbin = '02'
fittag  = {'00':'hew','01':'lew','02':'mew1','03':'mew2'}[plotbin]

BASE_DIR  = os.path.join(WORKING_DIR, 'spectra')
FIT_CNEB  = os.path.join(BASE_DIR, 'test_cneb', f'fits_{fittag}_cont2', 'test-00000-0000_fit.fits')
FIT_NNEB  = os.path.join(BASE_DIR, 'test_nneb', f'fits_{fittag}_cont2', 'test-00000-0000_fit.fits')
DATA_SPEC = os.path.join(BASE_DIR, 'test_cneb', f'bin_t{plotbin}', 'test', 'spec-test-00000-0000.fits')

if all(os.path.exists(p) for p in [FIT_CNEB, FIT_NNEB, DATA_SPEC, PATH_GAS_TEMPLATE]):
    fit_cneb_15 = fits.open(FIT_CNEB)
    fit_nneb_15 = fits.open(FIT_NNEB)
    data        = fits.open(DATA_SPEC)

    wave_cneb_15 = fit_cneb_15['RESTWAVE'].data
    flux_cneb_15 = fit_cneb_15[0].data
    # Continuum = total model - emission line model (extension 1)
    cont_cneb_15 = fit_cneb_15[0].data - fit_cneb_15[1].data

    wave_nneb_15 = fit_nneb_15['RESTWAVE'].data
    flux_nneb_15 = fit_nneb_15[0].data
    cont_nneb_15 = fit_nneb_15[0].data - fit_nneb_15[1].data

    wave_data = data[1].data['WAVE'][0]
    flux_data = data[1].data['FLUX'][0]

    fit_cneb_15.close(); fit_nneb_15.close(); data.close()

    # Interpolate the Balmer-series emission template onto the model grids,
    # then scale it to H12 using Case B line ratios. The hardcoded H12 fluxes
    # and gas_H12_flux are specific to the mew1 stack (plotbin='02').
    gas_df = pd.read_csv(PATH_GAS_TEMPLATE)
    f2 = interp1d(gas_df['wave'], gas_df['flux'], kind='cubic', bounds_error=False, fill_value=np.nan)
    flux2_interp_cneb = f2(wave_cneb_15)
    flux2_interp_nneb = f2(wave_nneb_15)
    del gas_df, f2

    H12_cneb = 0.64061325
    H12_nneb = 0.44886347
    gas_H12_flux = 0.315165
    gas_multi_cneb = H12_cneb / gas_H12_flux
    gas_multi_nneb = H12_nneb / gas_H12_flux

    plot_prettier(100, fontsize=24)
    scale = 3
    plt.rcParams.update({'xtick.major.size': 5*scale, 'xtick.major.width': 1.25*scale,
                          'xtick.minor.size': 2.5*scale, 'xtick.minor.width': 1.25*scale,
                          'ytick.direction': 'in',
                          'ytick.major.size': 5*scale, 'ytick.major.width': 1.25*scale,
                          'ytick.minor.size': 2.5*scale, 'ytick.minor.width': 1.25*scale})

    fig, axes = plt.subplot_mosaic("B;A", figsize=(22, 17), sharex=True)
    fig.tight_layout(pad=-1.0)
    plt.subplots_adjust(hspace=0.075)

    # Top panel (axes['B'], labelled 'A' in the paper): full best-fit model
    axes['B'].plot(wave_data, flux_data, c='k', label='Data')
    axes['B'].plot(wave_cneb_15, flux_cneb_15, c='xkcd:kelly green', ls='-.',
                   label='Model Flux, Nebular Continuum Added')
    axes['B'].plot(wave_nneb_15, flux_nneb_15, c='C1', ls='-.',
                   label='Model Flux, No Nebular Continuum')
    # Mask region (3662–3706 Å): high-order Balmer lines not fitted by the DAP
    axes['B'].axvspan(3662, 3706.3, color='gray', alpha=0.3)

    # Bottom panel (axes['A'], labelled 'B'): stellar continuum + Balmer template
    axes['A'].plot(wave_data, flux_data, c='k', label='Data')
    axes['A'].plot(wave_cneb_15, cont_cneb_15 + flux2_interp_cneb*gas_multi_cneb,
                   c='xkcd:kelly green', label='Model Continuum with Balmer Lines,\n Nebular Continuum Added ')
    axes['A'].plot(wave_nneb_15, cont_nneb_15 + flux2_interp_nneb*gas_multi_nneb,
                   c='C1', label='Model Continuum with Balmer Lines,\n No Nebular Continuum')
    axes['A'].axvspan(3662, 3706.3, color='gray', alpha=0.3)

    for key in ('A', 'B'):
        leg = axes[key].legend(fontsize=22, loc=3)
        leg.get_frame().set_alpha(1.0)
        leg.set_zorder(100)
        for line in leg.get_lines():
            line.set_linewidth(2.0)

    axes['A'].set_xlabel('Rest Wavelength (\u00c5)', fontsize=32)
    axes['A'].set_ylabel('Flux', fontsize=32)
    axes['B'].set_ylabel('Flux', fontsize=32)
    for key in ('A', 'B'):
        axes[key].set_xlim(3500.0, 4000.0)
        axes[key].set_ylim(0.7, 1.4)

    # subplot_mosaic("B;A") -> axes['B'] is top, axes['A'] is bottom.
    # Paper labels: top = 'A', bottom = 'B'. Reverse the annotation list.
    linelabels = ['A', 'B']
    for i, (label, ax) in enumerate(axes.items()):
        ax.annotate(
            linelabels[i],
            xy=(1, 0), xycoords='axes fraction',
            xytext=(-0.5, +0.5), textcoords='offset fontsize',
            fontsize='large', va='bottom', ha='right', fontfamily='serif', backgroundcolor='w',
            bbox=dict(facecolor='white', edgecolor='black', pad=3.0, alpha=1.0), zorder=100)

    plt.savefig(os.path.join(OUTPUT_DIR, f'stackfit{fittag}.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(OUTPUT_DIR, f'stackfit{fittag}.png'), dpi=150, bbox_inches='tight')
    plt.show()
    del (wave_cneb_15, flux_cneb_15, cont_cneb_15, wave_nneb_15, flux_nneb_15, cont_nneb_15,
         wave_data, flux_data, flux2_interp_cneb, flux2_interp_nneb)
else:
    for p in [FIT_CNEB, FIT_NNEB, DATA_SPEC, PATH_GAS_TEMPLATE]:
        if not os.path.exists(p):
            print(f'WARNING: nebular comparison input not found: {p}')
    print('  Skipping stackfit{} figure.'.format(fittag))

# Section 11 — Master DataFrame
### *The big one. Everything downstream lives here.*
*Builds `df` — the master emission-line DataFrame used by Sections 12–22. Reads flux, EW, and kinematics from the emlines FITS, merges target class, redshift, infiber fraction, and E(B-V) from AUXDATA, and (if `HAS_SUMMARY`) pulls in stellar masses, SFRs, and photometry from the Drake catalog. Also applies AGN/BPT/MEx/other classifications to produce `AGN`, builds `savedf` (the wide repeat-pair DataFrame used by Sections 15–16) from `df_uniques`, and builds `zach_df` (the MZR subset used by Section 20). Run this once per session before any science figure section. **Requires:** Sections 0, 1, and 6 (for `df_uniques`). **Required before:** Sections 12–22.*

**Defines:** `df`, `AGN`, `savedf`, `zach_df`

In [ ]:
# ── 2a. Build the core emission-line DataFrame from the emlines FITS ──────
# Reads flux and EW columns for all lines in _LINE_MAP from the full-sample
# emlines catalog. The mapping translates our internal shorthand (e.g. 'Hb',
# 'Ha') to the FITS column names (e.g. 'H_beta', 'H_alpha'). FITS column
# access is case-insensitive, so the uppercase column names in the file
# match regardless.
#
# EW columns for OII and SII are loaded as components and summed below,
# because those doublets share a single integration window.
#
# After this cell: df contains tag, PLATE, MJD, FIBER, z, per-line flux +
# err, and Gaussian/integrated EW columns.

hdu_eml0 = fits.open(PATH_EMLINES_FULL)
d0 = hdu_eml0[1].data

pl0 = d0['PLATE'].byteswap().newbyteorder()
mj0 = d0['MJD'].byteswap().newbyteorder()
fb0 = d0['FIBER'].byteswap().newbyteorder()
z0  = d0['Z'].byteswap().newbyteorder()

_LINE_MAP = {
    'NeV_3347':   'NeV_3347',
    'NeV_3427':   'NeV_3427',
    'OII_3727':   'OII_3727',
    'OII_3729':   'OII_3729',
    'NeIII_3869': 'NeIII_3870',
    'OIII_4364':  'OIII_4364',
    'OIII_4960':  'OIII_4960',
    'HeI_5877':   'HeI_5877',
    'OI_6302':    'OI_6302',
    'HeII_4687':  'HeII_4687',
    'Hb':         'H_beta',
    'OIII_5008':  'OIII_5008',
    'NII_6585':   'NII_6585',
    'Ha':         'H_alpha',
    'SII_6718':   'SII_6718',
    'SII_6732':   'SII_6733',
    'SIII_9071':  'SIII_9071',
    'SIII_9533':  'SIII_9533',
    'MGII_2796':  'MGII_2796',
    'MGII_2803':  'MGII_2803',
}

df_dict = {
    'tag':   make_tag_from_pmf(pl0, mj0, fb0),
    'PLATE': pl0,
    'MJD':   mj0,
    'FIBER': fb0,
    'z':     z0,
}

for our_name, lineid in _LINE_MAP.items():
    df_dict[our_name]          = d0[f'{lineid}_FLUX'].byteswap().newbyteorder()
    df_dict[our_name + '_err'] = d0[f'{lineid}_FLUX_ERR'].byteswap().newbyteorder()

# ── EW columns for Fig 15 (Gaussian vs integrated EW comparison) ──────────
# OII and SII are doublets — load components separately and sum below.
df_dict['OII_3727_ew']     = d0['OII_3727_EW'].byteswap().newbyteorder()
df_dict['OII_3729_ew']     = d0['OII_3729_EW'].byteswap().newbyteorder()
df_dict['OII_3727_ew_err'] = d0['OII_3727_EW_ERR'].byteswap().newbyteorder()
df_dict['OII_3729_ew_err'] = d0['OII_3729_EW_ERR'].byteswap().newbyteorder()
df_dict['OII_3727_int_ew'] = d0['OII_3727_INT_EW'].byteswap().newbyteorder()
df_dict['OII_3729_int_ew'] = d0['OII_3729_INT_EW'].byteswap().newbyteorder()

df_dict['Hb_ew']     = d0['H_beta_EW'].byteswap().newbyteorder()
df_dict['Hb_ew_err'] = d0['H_beta_EW_ERR'].byteswap().newbyteorder()
df_dict['Hb_int_ew'] = d0['H_beta_INT_EW'].byteswap().newbyteorder()

df_dict['OIII_5008_ew']     = d0['OIII_5008_EW'].byteswap().newbyteorder()
df_dict['OIII_5008_ew_err'] = d0['OIII_5008_EW_ERR'].byteswap().newbyteorder()
df_dict['OIII_5008_int_ew'] = d0['OIII_5008_INT_EW'].byteswap().newbyteorder()

df_dict['Ha_ew']     = d0['H_alpha_EW'].byteswap().newbyteorder()
df_dict['Ha_ew_err'] = d0['H_alpha_EW_ERR'].byteswap().newbyteorder()
df_dict['Ha_int_ew'] = d0['H_alpha_INT_EW'].byteswap().newbyteorder()

df_dict['NII_6585_ew']     = d0['NII_6585_EW'].byteswap().newbyteorder()
df_dict['NII_6585_ew_err'] = d0['NII_6585_EW_ERR'].byteswap().newbyteorder()
df_dict['NII_6585_int_ew'] = d0['NII_6585_INT_EW'].byteswap().newbyteorder()

df_dict['SII_6718_ew']     = d0['SII_6718_EW'].byteswap().newbyteorder()
df_dict['SII_6718_ew_err'] = d0['SII_6718_EW_ERR'].byteswap().newbyteorder()
df_dict['SII_6718_int_ew'] = d0['SII_6718_INT_EW'].byteswap().newbyteorder()

df_dict['SII_6732_ew']     = d0['SII_6733_EW'].byteswap().newbyteorder()
df_dict['SII_6732_ew_err'] = d0['SII_6733_EW_ERR'].byteswap().newbyteorder()
df_dict['SII_6732_int_ew'] = d0['SII_6733_INT_EW'].byteswap().newbyteorder()

hdu_eml0.close()
del d0, pl0, mj0, fb0, z0

df = pd.DataFrame(df_dict)
del df_dict

# Sum doublet EW components into combined columns used by Fig 15.
df['OII_ew']     = df['OII_3727_ew'] + df['OII_3729_ew']
df['OII_int_ew'] = df['OII_3727_int_ew'] + df['OII_3729_int_ew']
df['OII_ew_err'] = np.sqrt(df['OII_3727_ew_err']**2 + df['OII_3729_ew_err']**2)

df['SII_ew']     = df['SII_6718_ew'] + df['SII_6732_ew']
df['SII_int_ew'] = df['SII_6718_int_ew'] + df['SII_6732_int_ew']
df['SII_ew_err'] = np.sqrt(df['SII_6718_ew_err']**2 + df['SII_6732_ew_err']**2)

print(f'eBOSS DAP DR2 emlines rows: {len(df):,}')

In [ ]:
# ── 2b. Derived composite line quantities ─────────────────────────────────
# Sums doublet pairs and computes standard line ratios used throughout the
# science sections. All ratios are raw (no S/N cut); invalid values (from
# non-detections stored as -999 or 0) propagate as inf or nan and are
# filtered in each science section individually.
df['OII']     = df['OII_3727'] + df['OII_3729']
df['OII_err'] = np.sqrt(df['OII_3727_err']**2 + df['OII_3729_err']**2)
if 'OII_3727_ew' in df.columns and 'OII_3729_ew' in df.columns:
    df['OII_ew']     = df['OII_3727_ew'] + df['OII_3729_ew']
    df['OII_ew_err'] = np.sqrt(df['OII_3727_ew_err']**2 + df['OII_3729_ew_err']**2)
    df['OII_int_ew'] = df['OII_3727_int_ew'] + df['OII_3729_int_ew']

if 'MGII_2796' in df.columns:
    df['MGII']     = df['MGII_2796'] + df['MGII_2803']
    df['MGII_err'] = np.sqrt(df['MGII_2796_err']**2 + df['MGII_2803_err']**2)

if 'SII_6718' in df.columns:
    df['SII']     = df['SII_6718'] + df['SII_6732']
    df['SII_err'] = np.sqrt(df['SII_6718_err']**2 + df['SII_6732_err']**2)

if 'SIII_9071' in df.columns:
    df['SIII']     = df['SIII_9071'] + df['SIII_9533']
    df['SIII_err'] = np.sqrt(df['SIII_9071_err']**2 + df['SIII_9533_err']**2)

if 'NeV_3347' in df.columns and 'NeV_3427' in df.columns:
    df['NeV']     = df['NeV_3347'] + df['NeV_3427']
    df['NeV_err'] = np.sqrt(df['NeV_3347_err']**2 + df['NeV_3427_err']**2)

# Standard BPT/metallicity line ratios in log space.
df['N2']  = np.log10(np.divide(df['NII_6585'], df['Ha']))
df['R3']  = np.log10(np.divide(df['OIII_5008'], df['Hb']))
df['R2']  = np.log10(np.divide(df['OII'], df['Hb']))
df['O32'] = np.log10(np.divide(df['OIII_5008'], df['OII']))
df['Ne53']= np.log10(np.divide(df['NeV_3427'], df['NeIII_3869']))
df['Ha/Hb']    = np.divide(df['Ha'], df['Hb'])
df['Ha/Hb_err']= df['Ha/Hb'] * np.sqrt(
    (df['Ha_err']/df['Ha'])**2 + (df['Hb_err']/df['Hb'])**2)

In [ ]:
# ── 2c. Merge stellar kinematics from the emlines FITS ────────────────────
# SC_VELOCITY, SC_DISPERSION, and SC_CORRECTION are in the emlines catalog
# (not SPIND), so we re-open it here. The correction term is the instrumental
# resolution contribution that must be subtracted in quadrature from
# SC_DISPERSION to get the intrinsic stellar velocity dispersion.
hdu_eml = fits.open(PATH_EMLINES_FULL)
kin_pl = hdu_eml[1].data['PLATE'].byteswap().newbyteorder()
kin_mj = hdu_eml[1].data['MJD'].byteswap().newbyteorder()
kin_fb = hdu_eml[1].data['FIBER'].byteswap().newbyteorder()
kin_vel   = hdu_eml[1].data['SC_VELOCITY'].byteswap().newbyteorder()
kin_vel_e = hdu_eml[1].data['SC_VELOCITY_ERR'].byteswap().newbyteorder()
kin_disp  = hdu_eml[1].data['SC_DISPERSION'].byteswap().newbyteorder()
kin_disp_e= hdu_eml[1].data['SC_DISPERSION_ERR'].byteswap().newbyteorder()
kin_corr  = hdu_eml[1].data['SC_CORRECTION'].byteswap().newbyteorder()
kin_snr   = hdu_eml[1].data['SNR'].byteswap().newbyteorder()
hdu_eml.close()

df_kin = pd.DataFrame({
    'tag':           make_tag_from_pmf(kin_pl, kin_mj, kin_fb),
    'SC_vel':        kin_vel,
    'SC_vel_err':    kin_vel_e,
    'SC_disp':       kin_disp,
    'SC_disp_err':   kin_disp_e,
    'SC_correction': kin_corr,
    'SNR':           kin_snr,
})
del kin_pl, kin_mj, kin_fb, kin_vel, kin_vel_e, kin_disp, kin_disp_e, kin_corr, kin_snr
df = df.merge(df_kin, on='tag')
del df_kin
print(f'After kin merge: {len(df):,}')

In [ ]:
# ── 2d. Merge TARGET_CLASS, infiber, and (if available) Drake catalog ─────
# AUXDATA provides target class, infiber fraction, E(B-V), RA, and DEC.
# If the Drake catalog is present (HAS_SUMMARY), LOGM, LOGM_ERR, and LOGSFR
# are also merged in. Sections without stellar masses (17, 19, 20, 22)
# check HAS_SUMMARY and skip gracefully if the catalog is absent.
aux = fits.getdata(PATH_AUXDATA, 1)
s_taginit = aux['PMF_String'].byteswap().newbyteorder()
s_tag     = [s.strip().lstrip('0').replace('_','-').zfill(16) for s in s_taginit]
s_target_class = aux['TARGET_CLASS'].byteswap().newbyteorder()
s_z       = fits.getdata(PATH_SPIND, 1)['Z'].byteswap().newbyteorder()
s_infiber = aux['INFIBER'].byteswap().newbyteorder()
s_ebv     = aux['E(B-V)'].byteswap().newbyteorder()
s_ra      = aux['RA'].byteswap().newbyteorder()
s_dec     = aux['DEC'].byteswap().newbyteorder()
del aux, s_taginit

if HAS_SUMMARY:
    summary = fits.getdata(PATH_SUMMARY, 1)
    sum_tag    = tag_from_pmf_string(summary['PMF_String'].byteswap().newbyteorder())
    sum_logm   = summary['LOGMSTAR'].byteswap().newbyteorder()
    sum_logm_e = summary['LOGMSTAR_ERR'].byteswap().newbyteorder()
    sum_logsfr = summary['LOGSFR'].byteswap().newbyteorder()
    del summary
    df_drake = pd.DataFrame({
        'tag': sum_tag, 'LOGM': sum_logm, 'LOGM_ERR': sum_logm_e,
        'LOGSFR': sum_logsfr,
    }).drop_duplicates(subset='tag')
    del sum_tag, sum_logm, sum_logm_e, sum_logsfr

    df_sum = pd.DataFrame({
        'tag': s_tag, 'TARGET_CLASS': s_target_class,
        'z_sum': s_z, 'infiber': s_infiber, 'ebv_mw': s_ebv,
        'RA': s_ra, 'DEC': s_dec,
    })
    df_sum = df_sum.merge(df_drake, on='tag', how='left')
    del df_drake
    print(f'After summary merge: {len(df_sum):,}')
else:
    # Fill mass/SFR columns with NaN so downstream code doesn't need to
    # check column existence — it just checks HAS_SUMMARY before using them.
    df_sum = pd.DataFrame({
        'tag': s_tag, 'TARGET_CLASS': s_target_class,
        'z_sum': s_z, 'infiber': s_infiber, 'ebv_mw': s_ebv,
        'RA': s_ra, 'DEC': s_dec,
        'LOGM': np.nan, 'LOGM_ERR': np.nan, 'LOGSFR': np.nan,
    })
    print(f'No summary catalog -- LOGM/LOGSFR will be NaN.')

del s_tag, s_target_class, s_z, s_infiber, s_ebv, s_ra, s_dec

df = df.merge(df_sum, on='tag')
del df_sum
print(f'After merge: {len(df):,}')

In [ ]:
# ── 2e. Merge spectral indices from the spind FITS ────────────────────────
# Merges Dn4000, HδA, Mgb, and the custom BalmerBreak index. These are
# used in Sections 21 (Fig 24) and the mass-σ★ cuts (Sections 17, 22).
# The BalmerBreak index is an eBOSS-DAP addition not present in MaNGA-DAP.
hdu_sp = fits.open(PATH_SPIND)
sp_pl   = hdu_sp[1].data['PLATE'].byteswap().newbyteorder()
sp_mj   = hdu_sp[1].data['MJD'].byteswap().newbyteorder()
sp_fb   = hdu_sp[1].data['FIBER'].byteswap().newbyteorder()
sp_dn   = hdu_sp[1].data['Dn4000'].byteswap().newbyteorder()
sp_dn_e = hdu_sp[1].data['Dn4000_ERR'].byteswap().newbyteorder()
sp_hda  = hdu_sp[1].data['HDeltaA'].byteswap().newbyteorder()
sp_hda_e= hdu_sp[1].data['HDeltaA_ERR'].byteswap().newbyteorder()
sp_mgb  = hdu_sp[1].data['Mgb'].byteswap().newbyteorder()
sp_mgb_e= hdu_sp[1].data['Mgb_err'].byteswap().newbyteorder()
sp_bb   = hdu_sp[1].data['BalmerBreak'].byteswap().newbyteorder()
sp_bb_e = hdu_sp[1].data['BalmerBreak_ERR'].byteswap().newbyteorder()
hdu_sp.close()

df_spind = pd.DataFrame({
    'tag':             make_tag_from_pmf(sp_pl, sp_mj, sp_fb),
    'Dn4000':          sp_dn,
    'Dn4000_err':      sp_dn_e,
    'HDeltaA':         sp_hda,
    'HDeltaA_err':     sp_hda_e,
    'MGB':             sp_mgb,
    'MGB_err':         sp_mgb_e,
    'BalmerBreak':     sp_bb,
    'BalmerBreak_err': sp_bb_e,
})
del sp_pl, sp_mj, sp_fb, sp_dn, sp_dn_e, sp_hda, sp_hda_e
del sp_mgb, sp_mgb_e, sp_bb, sp_bb_e

df = df.merge(df_spind, on='tag')
del df_spind
print(f'After spind merge: {len(df):,}')

In [ ]:
# ── 2f. AGN/galaxy classifications ────────────────────────────────────────
# Applies BPT, MEx, Blue-diagram, TrEW, OHNO, Ne53, and NeV classifications
# directly to df. Class values: 0=AGN, 1=SFG, 2=composite/SF, 3=LIER,
# 4=confusion, 5=IMBH candidate, -1=line redshifted out of window.
#
# Classifications are later invalidated for spectra where the relevant lines
# fall outside the observed spectral range (redshift cuts at the end of
# this cell). This is why, e.g., BPT_class has entries with -1 at high z
# even though the classification code ran on the full sample.
#
# Note: MEx uses a redshift-dependent mass adjustment (MEx_adj) to account
# for the redshift evolution of the ionisation parameter, per Juneau+2014.

df['Tr2']   = np.log10(df['OII_ew'] + df['Hb_ew'])
df['Ne3O2'] = np.log10(np.divide(df['NeIII_3869'], df['OII']))

df['BPT_class']  = np.nan
df['NeV_class']  = np.nan
df['Blue_class'] = np.nan
df['MEx_class']  = np.nan
df['TrEW_class'] = np.nan
df['OHNO_class'] = np.nan
df['Ne53_class'] = np.nan

df['MEx_adj'] = 1.7 * (df['z'] - 0.35)
df['MEx_x']   = df['LOGM'] - df['MEx_adj']

# ── NeV: S/N > 6 on the summed [Ne V] doublet ────────────────────────────
NeVAGN = df[np.abs(df['NeV']) >= np.abs(df['NeV_err'] * 6)]
df.loc[df.tag.isin(NeVAGN.tag), 'NeV_class'] = 0

# ── BPT: Law et al. 2021 demarcation lines ───────────────────────────────
cut1   = df[(-0.39*df['R3']**4 - 0.582*df['R3']**3 - 0.637*df['R3']**2
             - 0.048*df['R3'] - 0.119) > df['N2']]
cut1_1 = cut1[(0.438/(cut1['N2'] + 0.023) + 1.222) < cut1['R3']]
BPT_composites = cut1_1[cut1_1['N2'] > -1.4]
cut2 = df[(0.438/(df['N2'] + 0.023) + 1.222) > df['R3']]
BPT_SFGs = cut2[cut2['N2'] < -0.1]
cut3 = df[np.invert(df.index.isin(pd.concat([BPT_composites, BPT_SFGs]).index))]
BPT_LIERs = cut3[cut3['R3'] < (0.95*cut3['N2'] + 0.56)]
BPT_AGNs  = cut3[cut3['R3'] > (0.95*cut3['N2'] + 0.56)]

df.loc[df.tag.isin(BPT_AGNs.tag),       'BPT_class'] = 0
df.loc[df.tag.isin(BPT_SFGs.tag),       'BPT_class'] = 1
df.loc[df.tag.isin(BPT_composites.tag), 'BPT_class'] = 2
df.loc[df.tag.isin(BPT_LIERs.tag),      'BPT_class'] = 3

# ── MEx: mass-excitation diagram (Juneau+2014) ────────────────────────────
cut1 = df[df['LOGM'] < (10 + df['MEx_adj'])]
cut1_1 = cut1[((0.375/(cut1['MEx_x']-10.5)) + 1.14) < cut1['R3']]
a0,a1,a2,a3 = 410.24, -109.333, 9.71731, -0.288244
cut2 = df[df['LOGM'] > (10 + df['MEx_adj'])]
cut2_1 = cut2[(a0 + a1*cut2['MEx_x'] + a2*cut2['MEx_x']**2 + a3*cut2['MEx_x']**3) < cut2['R3']]
MEx_AGN = pd.concat([cut1_1, cut2_1])

cut3 = df[df['LOGM'] < (9.59 + df['MEx_adj'])]
cut3_1 = cut3[((0.375/(cut3['MEx_x']-10.5)) + 1.14) > cut3['R3']]
a0,a1,a2,a3 = 352.066, -93.8249, 8.32651, -0.246416
cut4 = df[df['LOGM'] > (9.59 + df['MEx_adj'])]
cut4_1 = cut4[(a0 + a1*cut4['MEx_x'] + a2*cut4['MEx_x']**2 + a3*cut4['MEx_x']**3) > cut4['R3']]
MEx_SFG = pd.concat([cut3_1, cut4_1])
MEx_CMP = df[np.invert(df.index.isin(pd.concat([MEx_AGN, MEx_SFG]).index))]

df.loc[df.tag.isin(MEx_AGN.tag), 'MEx_class'] = 0
df.loc[df.tag.isin(MEx_SFG.tag), 'MEx_class'] = 1
df.loc[df.tag.isin(MEx_CMP.tag), 'MEx_class'] = 2

# ── Blue diagram (Lamareille+2010) ────────────────────────────────────────
cut1   = df[df['R2'] < 1.06]
cut1_1 = cut1[cut1['R2'] > 0.09]
cut1_2 = cut1_1[(-(cut1_1['R2']-1.0)**2 - 0.1*cut1_1['R2'] + 0.25) > cut1_1['R3']]
Blue_confusion1 = cut1_2[((cut1_2['R2']-0.2)**2 - 0.6) < cut1_2['R3']]

cut2   = df[df['R2'] < 0.72]
cut2_1 = cut2[((0.11/(cut2['R2']-0.92)) + 0.85) < cut2['R3']]
cut3   = df[df['R2'] > 0.72]
cut3_1 = cut3[(0.95*(cut3['R2']) - 0.4) < cut3['R3']]
Blue_AGN = pd.concat([cut2_1, cut3_1])

cut4 = cut2[((0.11/(cut2['R2']-0.92)) + 0.85) > cut2['R3']]
Blue_confusion2 = cut4[cut4['R3'] > 0.3]

cut5   = df[df['R2'] < 0.861]
cut5_1 = cut5[((0.11/(cut5['R2']-0.92)) + 0.85) > cut5['R3']]
cut6   = Blue_confusion1[Blue_confusion1['R2'] < 0.861]
Blue_SFG  = cut5_1[np.invert(cut5_1.index.isin(pd.concat([Blue_confusion2, cut6]).index))]
Blue_LIER = df[np.invert(df.index.isin(
    pd.concat([Blue_AGN, Blue_SFG, Blue_confusion1, Blue_confusion2]).index))]

df.loc[df.tag.isin(Blue_AGN.tag),         'Blue_class'] = 0
df.loc[df.tag.isin(Blue_SFG.tag),         'Blue_class'] = 1
df.loc[df.tag.isin(Blue_confusion1.tag),  'Blue_class'] = 4
df.loc[df.tag.isin(Blue_confusion2.tag),  'Blue_class'] = 4
df.loc[df.tag.isin(Blue_LIER.tag),        'Blue_class'] = 3

# ── TrEW (Lamareille+2010) ────────────────────────────────────────────────
TrEW_SFG = df[(-3.25 + 3.20*df['Tr2'] - 0.62*(df['Tr2']**2)) > df['R3']]
cut1 = df[np.invert(df.index.isin(TrEW_SFG.index))]
TrEW_AGN       = cut1[cut1['R3'] > 0.332]
TrEW_confusion = cut1[cut1['R3'] < 0.332]

df.loc[df.tag.isin(TrEW_AGN.tag),       'TrEW_class'] = 0
df.loc[df.tag.isin(TrEW_SFG.tag),       'TrEW_class'] = 1
df.loc[df.tag.isin(TrEW_confusion.tag), 'TrEW_class'] = 4

# ── OHNO diagram (Backhaus+2022) ──────────────────────────────────────────
OHNO_AGN = df[((0.35/(2.8*df['Ne3O2'] - 0.8)) + 0.64) < df['R3']]
OHNO_SFG = df[np.invert(df.index.isin(OHNO_AGN.index))]
df.loc[df.tag.isin(OHNO_AGN.tag), 'OHNO_class'] = 0
df.loc[df.tag.isin(OHNO_SFG.tag), 'OHNO_class'] = 1

# ── Ne53: [Ne V]/[Ne III] ratio diagnostic ────────────────────────────────
init = df[np.isfinite(df['Ne53'])]
cut1 = init[init['Ne53'] < -5]
Ne53_SFG = cut1
cut2 = init[np.invert(init.index.isin(Ne53_SFG.index))]
cut2_1 = cut2[cut2['Ne53'] < -0.3]
cut2_2 = cut2_1[cut2_1['R3'] < 1]
Ne53_COMP = cut2_2
cut3 = cut2[np.invert(cut2.index.isin(Ne53_COMP.index))]
cut3_1 = cut3[cut3['R3'] < 0.6]
Ne53_IMBH = cut3_1
Ne53_AGN  = cut3[np.invert(cut3.index.isin(Ne53_IMBH.index))]

df.loc[df.tag.isin(Ne53_AGN.tag),  'Ne53_class'] = 0
df.loc[df.tag.isin(Ne53_SFG.tag),  'Ne53_class'] = 1
df.loc[df.tag.isin(Ne53_COMP.tag), 'Ne53_class'] = 2
df.loc[df.tag.isin(Ne53_IMBH.tag), 'Ne53_class'] = 5

del (cut1, cut1_1, cut1_2, cut2, cut2_1, cut2_2, cut3, cut3_1, cut4, cut4_1, cut5, cut5_1,
     cut6, init, NeVAGN, BPT_composites, BPT_SFGs, BPT_AGNs, BPT_LIERs,
     MEx_AGN, MEx_SFG, MEx_CMP, Blue_AGN, Blue_SFG, Blue_LIER, Blue_confusion1, Blue_confusion2,
     TrEW_AGN, TrEW_SFG, TrEW_confusion, OHNO_AGN, OHNO_SFG,
     Ne53_AGN, Ne53_SFG, Ne53_COMP, Ne53_IMBH)

# ── Invalidate classifications for lines that have redshifted out ─────────
# Sets class to -1 where the required lines are no longer in the spectral
# window. This prevents Section 18, 19 etc. from using spurious -999
# flux entries in the classification logic.
BPT_zmax1 = np.max(df[df['Ha']  != -999.0]['z'])
BPT_zmax2 = np.max(df[df['NII_6585'] != -999.0]['z'])
BPT_zmax  = np.min([BPT_zmax1, BPT_zmax2])
OIII_zmax = np.max(df[df['OIII_5008'] != -999.0]['z'])
NeV_zmax  = np.max(df[df['NeV'] != -999.0]['z'])

df.loc[df['z'] >= BPT_zmax,  'BPT_class']  = -1
df.loc[df['z'] >= OIII_zmax, 'MEx_class']  = -1
df.loc[df['z'] >= OIII_zmax, 'Blue_class'] = -1
df.loc[df['z'] >= OIII_zmax, 'TrEW_class'] = -1
df.loc[df['z'] >= NeV_zmax,  'NeV_class']  = -1
# [Ne V] is only reliably in the window above z=0.147 (where it clears
# the spectrograph's blue edge), so also invalidate at low redshift.
df.loc[df['z'] <= 0.1,       'NeV_class']  = -1
del BPT_zmax1, BPT_zmax2, BPT_zmax, OIII_zmax, NeV_zmax

# AGN subsample: BPT AGN only. Used by Sections 18, 20, 22, 24, 25.
AGN = df[df['BPT_class'] == 0].copy()
print(f'df rows: {len(df):,}   AGN rows: {len(AGN):,}')

# imp_lines.csv and eboss_lines.csv are read by the NeV and Radio master
# notebooks — df now contains all columns either expects, so they can be
# written from here rather than maintained as separate files.

In [ ]:
# ── 2g. Build savedf: wide repeat-pair DataFrame ─────────────────────────
# Joins df_uniques (the pair index from Section 6) with the full df twice —
# once keyed on the first spectrum in each pair, once on the second — then
# merges the two halves into one wide row per pair. The result has _x and
# _y suffixes for every df column, representing the two observations of the
# same galaxy. savedf is kept in memory for Sections 15 and 16.
matches = df_uniques.drop_duplicates().copy()
del df_uniques
matches = matches[matches['tag1'] != matches['tag2']]
matches['tag1'] = matches['tag1'].str.zfill(16)
matches['tag2'] = matches['tag2'].str.zfill(16)
matches['match'] = matches['tag1'] + '|' + matches['tag2']

df['tag1'] = df['tag']
df['tag2'] = df['tag']

pairs_side1 = df.merge(matches, on='tag1').drop(columns=['tag1','tag2_x','tag2_y']).drop_duplicates()
pairs_side2 = df.merge(matches, on='tag2').drop(columns=['tag2','tag1_x','tag1_y']).drop_duplicates()
del matches

df = df.drop(columns=['tag1','tag2'])

final_matches = pairs_side1.merge(pairs_side2, on='match').drop_duplicates()
del pairs_side1, pairs_side2

savedf = final_matches.copy()
print(f'Matched pairs in memory (savedf): {len(savedf):,} repeat-observation pairs.')
del final_matches

In [ ]:
if not HAS_SUMMARY:
    print('This section requires data from D. Miller et al. (in prep.), '
          'which will be released alongside the Drake et al. catalog paper. Skipping.')
else:
    # ── 2h. Build zach_df: MZR working subset ────────────────────────────────
    # Applies S/N >= 3 cuts on the four key BPT lines and selects and renames
    # the columns expected by the MZR metallicity computation in Section 20.
    # zach_df is kept in memory and consumed by Section 20.
    mzr_cut = df[np.abs(df['OIII_5008']) / np.abs(df['OIII_5008_err']) >= 3]
    mzr_cut = mzr_cut[np.abs(mzr_cut['NII_6585']) / np.abs(mzr_cut['NII_6585_err']) >= 3]
    mzr_cut = mzr_cut[np.abs(mzr_cut['Hb']) / np.abs(mzr_cut['Hb_err']) >= 3]
    mzr_cut = mzr_cut[np.abs(mzr_cut['Ha']) / np.abs(mzr_cut['Ha_err']) >= 3]

    # PMF_String format: zero-padded plate_mjd_fiber (underscore-separated),
    # matching the format in the AUXDATA catalog.
    pmf_string = (mzr_cut['PLATE'].astype(int).astype(str).str.zfill(5) + '_'
                   + mzr_cut['MJD'].astype(int).astype(str)
                   + '_' + mzr_cut['FIBER'].astype(int).astype(str).str.zfill(4))

    zach_df = pd.DataFrame({
        'PMF_String': pmf_string,
        'OII':        mzr_cut['OII'],        'OII_err':    mzr_cut['OII_err'],
        'OIII':       mzr_cut['OIII_5008'],  'OIII_err':   mzr_cut['OIII_5008_err'],
        'Hbeta':      mzr_cut['Hb'],         'Hbeta_err':  mzr_cut['Hb_err'],
        'LOGM':       mzr_cut['LOGM'],       'LOGM_ERR':   mzr_cut['LOGM_ERR'],
        'NII':        mzr_cut['NII_6585'],   'NII_err':    mzr_cut['NII_6585_err'],
        'Halpha':     mzr_cut['Ha'],         'Halpha_err': mzr_cut['Ha_err'],
        'TARGET_CLASS':       mzr_cut['TARGET_CLASS'],
        'redshift':   mzr_cut['z'],
    })
    del pmf_string, mzr_cut

    print(f'mzr data in memory (zach_df): {len(zach_df):,} rows.')

# Section 12 — Figures 13 & 14: Emission-Line and Spectral-Index Histograms
### *A catalog's worth of S/N, summarised in two figures.*
*Fig. 13 shows S/N histograms for 12 commonly-used emission lines split by target class, annotated with detection counts above S/N > 5. Fig. 14 does the same for errors in four spectral indices. Together they give a quick sense of which measurements are actually usable for which populations. Both read directly from `df`. **Requires:** Sections 0 and 11.*

**Fig. 13** `eml_hists.pdf` — emission-line S/N histograms.

**Fig. 14** `spind_hists.pdf` — spectral index error histograms.

In [ ]:
# ── Figure 13: eml_hists ─────────────────────────────────────────────────
# S/N histograms for 12 commonly-used emission lines, split by target class.
# The annotation on each panel shows the number of spectra with S/N > 5 in
# that line. Lines use the same colour palette as Figs 1-4.
eml_lines  = ['MGII','NeV','OII','NeIII_3869','OIII_4364','HeII_4687',
              'Hb','OIII_5008','NII_6585','Ha','SII','SIII']
eml_labels = ['MG II', '[Ne V]', '[O II]', '[Ne III] 3869',
              '[O III] 4364', 'He II 4687',
              r'H$\beta$', '[O III] 5008', '[N II] 6585',
              r'H$\alpha$', '[S II]', '[S III]']
eml_ranges = [(0.001,11)]*4 + [(0.001,11)] + [(0.001,21)]*7

scale = 3
plot_prettier(150, fontsize=24)
plt.rcParams.update({'xtick.major.size':5*scale,'xtick.major.width':1.25*scale,
                     'xtick.minor.size':2.5*scale,'xtick.minor.width':1.25*scale,
                     'ytick.major.size':5*scale,'ytick.major.width':1.25*scale,
                     'ytick.minor.size':2.5*scale,'ytick.minor.width':1.25*scale})

fig, axes = plt.subplot_mosaic('ABCD;EFGH;IJKL', figsize=(11*scale, 8.5*scale*0.5))
fig.tight_layout(pad=1.0)

target_groups = [
    ('LRG4', c_lrg4), ('LRG3', c_lrg3), ('LRGL', c_lrgl),
    ('ELG',  c_elg),  ('TDSP',  c_tdsp),  ('QSO',  c_qso),
]

for (panel, ax), line, ll, (rlo, rhi) in zip(
        axes.items(), eml_lines, eml_labels, eml_ranges):
    if line not in df.columns or line+'_err' not in df.columns:
        ax.text(0.5, 0.5, f'{line} not in df', transform=ax.transAxes, ha='center')
        continue
    # Require positive flux (S/N is meaningless for non-detections stored as -999).
    cut = df[df[line] > 0.0001].copy()
    num_5 = len(cut[cut[line] / cut[line+'_err'] > 5])
    for target_class_str, color in target_groups:
        sub = cut[cut['TARGET_CLASS'].str.contains(target_class_str)]
        ax.hist(sub[line]/sub[line+'_err'], bins=100, range=(rlo,rhi),
                log=True, histtype='step', color=color, alpha=1, fill=False)
    # Grey total histogram underneath the target-class outlines.
    ax.hist(cut[line]/cut[line+'_err'], bins=100, range=(rlo,rhi),
            log=True, histtype='step', color=c_tot, alpha=0.10, fill=True, zorder=-1)
    ax.hist(cut[line]/cut[line+'_err'], bins=100, range=(rlo,rhi),
            log=True, histtype='step', color='black', alpha=1, fill=False, zorder=-1)
    ax.set_xlim(right=rhi)
    ax.annotate(f'{panel}: {ll}\n> 5 : {num_5:,}',
                xy=(1,1), xycoords='axes fraction',
                xytext=(-0.25,-0.25), textcoords='offset fontsize',
                fontsize='large', va='top', ha='right', fontfamily='serif',
                bbox=dict(facecolor='none', edgecolor='none', pad=3.0))

plt.subplots_adjust(wspace=0.2, hspace=0.2)
fig.supxlabel('(Signal-to-Noise Ratio)$_{\\mathrm{line}}$', y=-0.07, fontsize=48)
fig.supylabel('Number of Galaxies', x=-0.02, fontsize=48)
plt.savefig(os.path.join(OUTPUT_DIR,'eml_hists.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'eml_hists.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Figure 14: spind_hists ────────────────────────────────────────────────
# Error histograms for four spectral indices. Because index values can be
# positive or negative, S/N is poorly defined near zero — showing the error
# distribution directly is more informative. Where a quality cut exists
# (Dn4000 error < 0.03; HδA error < 0.8, per Spindler+2018), the count
# below the cut is annotated on the panel.
spind_lines  = ['Dn4000_err','HDeltaA_err','MGB_err','BalmerBreak_err']
spind_labels = [r'D$_{\mathrm{n}}$4000 error',
                r'Lick H$\delta_{\mathrm{A}}$ error',
                r'Lick Mg$b$ error',
                'Balmer Break error']
spind_cuts  = [0.03, 0.8, None, None]
spind_ranges= [(-0.001,1),(-0.001,10),(-0.001,10),(-0.001,1)]

scale = 3
plot_prettier(150, fontsize=24)
fig, axes = plt.subplot_mosaic('ABCD', figsize=(11*scale, 8.5*scale*0.25))
fig.tight_layout(pad=1.0)

for (panel, ax), line, ll, cut_val, (rlo, rhi) in zip(
        axes.items(), spind_lines, spind_labels, spind_cuts, spind_ranges):
    if line not in df.columns:
        continue
    cut = df[df[line] > 0.0].copy()
    num_cut = len(cut[cut[line] < cut_val]) if cut_val else len(cut)
    for target_class_str, color in target_groups:
        sub = cut[cut['TARGET_CLASS'].str.contains(target_class_str)]
        ax.hist(sub[line], bins=100, range=(rlo,rhi), log=True,
                histtype='step', color=color, fill=False)
    ax.hist(cut[line], bins=100, range=(rlo,rhi), log=True,
            histtype='step', color=c_tot, alpha=0.10, fill=True, zorder=-1)
    ax.hist(cut[line], bins=100, range=(rlo,rhi), log=True,
            histtype='step', color='black', alpha=1, fill=False, zorder=-1)
    ax.set_xlim(right=rhi)
    note = f'\n< cut: {num_cut:,}' if cut_val else ''
    ax.annotate(f'{panel}: {ll}{note}',
                xy=(1,1), xycoords='axes fraction',
                xytext=(-0.25,-0.25), textcoords='offset fontsize',
                fontsize='large', va='top', ha='right', fontfamily='serif',
                bbox=dict(facecolor='none', edgecolor='none', pad=3.0))

plt.subplots_adjust(wspace=0.2)
fig.supxlabel('Error$_{\\mathrm{line}}$', y=-0.07, fontsize=56)
fig.supylabel('Number of Galaxies', x=-0.04, fontsize=56)
plt.savefig(os.path.join(OUTPUT_DIR,'spind_hists.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'spind_hists.png'), dpi=150, bbox_inches='tight')
plt.show()

# Section 13 — Figure 15: Gaussian vs Integrated EW Comparison
### *Does the Gaussian fit actually describe the line, or is it just optimistic?*
*Compares equivalent widths from the best-fit Gaussian model against those from direct continuum-subtracted integration, as a function of S/N. Good agreement at high S/N is the expected result; deviations flag lines with substructure or broad components that the Gaussian misses. **Requires:** Sections 0 and 11.*

**Output:** `EW_comps.pdf` / `.png`

In [ ]:
# ── DIAGNOSTIC: EW_diff prevalence check ─────────────────────────────────
# For each line, computes the fraction of galaxies where EW_diff =
# (Gaussian EW - integrated EW) / Gaussian EW is near 1 (i.e., the
# integrated EW is near zero). A high fraction signals a data issue
# rather than a real physical effect — this was used to identify the
# H_beta_INT_EW regression in the v12 catalog.
print(f'df rows at this point: {len(df):,}')
print()
for _line in ['OII','Hb','OIII_5008','NII_6585','Ha','SII']:
    _ew, _int_ew, _ew_err = _line+'_ew', _line+'_int_ew', _line+'_ew_err'
    _dfw = df[[_ew, _int_ew, _ew_err]].copy()
    _dfw['EW_diff'] = (_dfw[_ew] - _dfw[_int_ew]) / _dfw[_ew]
    _dfw = _dfw[(_dfw[_ew] > 0) & (_dfw[_int_ew] > 0) & np.isfinite(_dfw['EW_diff'])]
    _near_one = _dfw[_dfw['EW_diff'] > 0.9]
    print(f'{_line}: total n={len(_dfw):,}, EW_diff>0.9 n={len(_near_one):,} '
          f'({100*len(_near_one)/len(_dfw):.2f}%)')

In [ ]:
# ── Compute binned EW_diff vs S/N for Fig 15 ─────────────────────────────
# For each line, bins the percent difference between Gaussian and integrated
# EW as a function of the Gaussian EW's own S/N. The 16th/84th percentile
# envelope and median are stored for the plotting cell.
# S/N here is the EW's own S/N (EW/EW_err), not the line flux S/N —
# this isolates the EW measurement quality from the underlying flux S/N.
ew_lines  = ['OII','Hb','OIII_5008','NII_6585','Ha','SII']
ew_labels = ['[O II]', r'H$\beta$', '[O III] 5008',
             '[N II] 6585', r'H$\alpha$', '[S II]']

binning = 26  # 25 bins of width 2, spanning S/N 0-52
x2      = np.linspace(0, 52, binning)

snrs_all = []; top_all = []; bot_all = []; mean_all = []

for line in ew_lines:
    ew_col      = line + '_ew'
    ew_int_col  = line + '_int_ew'
    ew_err_col  = line + '_ew_err'

    if ew_col not in df.columns or ew_int_col not in df.columns or ew_err_col not in df.columns:
        snrs_all.append(x2[1:]); top_all.append([np.nan]*(len(x2)-1))
        bot_all.append([np.nan]*(len(x2)-1)); mean_all.append([np.nan]*(len(x2)-1))
        continue

    dfw = df[[ew_col, ew_int_col, ew_err_col]].copy()
    dfw['S/N']     = dfw[ew_col] / dfw[ew_err_col]
    dfw['EW_diff'] = (dfw[ew_col] - dfw[ew_int_col]) / dfw[ew_col]
    dfw = dfw[(dfw[ew_col] > 0) & (dfw[ew_int_col] > 0) & np.isfinite(dfw['EW_diff'])]

    sn_l, tp_l, bt_l, mn_l = [], [], [], []
    for i, ix in enumerate(x2[1:]):
        sl = dfw[(dfw['S/N'] > x2[i]) & (dfw['S/N'] < ix)]['EW_diff']
        sn_l.append((ix + x2[i]) / 2)
        tp_l.append(np.nanpercentile(sl, 68.27+15.7) if len(sl) else np.nan)
        bt_l.append(np.nanpercentile(sl, 15.7) if len(sl) else np.nan)
        mn_l.append(np.nanmedian(sl)            if len(sl) else np.nan)

    snrs_all.append(sn_l); top_all.append(tp_l)
    bot_all.append(bt_l); mean_all.append(mn_l)

In [ ]:
# ── Figure 15: EW_comps ───────────────────────────────────────────────────
# Plots the binned EW_diff (solid line = median, dashed = 16th/84th pctile)
# against EW S/N. The grey lines show the theoretical 1-sigma scatter
# expected from random noise alone — deviations above this indicate
# systematic differences between the Gaussian and integrated EW estimators.
plot_prettier(100, fontsize=24)
plt.figure(figsize=[11, 8.5])

cls = ['C0','C1','C2','C3','C4','C5']
x_th = np.linspace(0.01, 51, binning)
y_th = np.sqrt(2) / x_th

for i, (line, ll) in enumerate(zip(ew_lines, ew_labels)):
    plt.plot(snrs_all[i], top_all[i],  color=cls[i], ls='--')
    plt.plot(snrs_all[i], mean_all[i], color=cls[i], label=ll)
    plt.plot(snrs_all[i], bot_all[i],  color=cls[i], ls='--')

plt.plot(x_th,  y_th, c='gray', lw=6, alpha=0.5, label='Theoretical 1$\\sigma$')
plt.plot(x_th, -y_th, c='gray', lw=6, alpha=0.5)
plt.legend(loc=1, fontsize=18, ncol=2)
plt.xlabel('Signal-to-Noise Ratio$_{\\mathrm{line}}$', fontsize=36)
plt.ylabel(r'EW$\left(\frac{\rm Gauss - Integrated}{\rm Gauss}\right)_{\rm line}$', fontsize=36)
plt.xlim(0.0, 50.0)
plt.ylim(-0.5, 0.5)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'EW_comps.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'EW_comps.png'), dpi=150, bbox_inches='tight')
plt.show()
del top_all, bot_all, mean_all, x_th, y_th

In [ ]:
# ── Section 13 cleanup ───────────────────────────────────────────────────
del snrs_all

# Section 14 — Figure 16: SDSS spZline Comparison
### *Checking our work against the pipeline we were built to replace.*
*Loads SDSS spZline fluxes from AUXDATA, applies the same Milky Way dust correction used by the eBOSS-DAP, then plots the ratio of eBOSS-DAP to spZline fluxes as a function of S/N for six strong lines. Agreement within the expected scatter validates the pipeline; any systematic offset in Hβ reflects the difference in stellar continuum treatment between the two codes. **Requires:** Sections 0 and 11.*

**Output:** `sdss_comp.pdf` / `.png`

In [ ]:
# ── Load spZline fluxes and apply Milky Way dust correction ───────────────
# spZline fluxes are stored in AUXDATA (piped in from the original spZline
# FITS files). They are NOT corrected for MW reddening by SDSS, so we apply
# the same correction used by the eBOSS-DAP before comparing. The correction
# is wavelength-specific (per-line, not per-spectrum).
aux = fits.getdata(PATH_AUXDATA, 1)

spz_pl  = aux['PLATE'].byteswap().newbyteorder()
spz_mj  = aux['MJD'].byteswap().newbyteorder()
spz_fb  = aux['FIBER'].byteswap().newbyteorder()

# SII is summed from the 6717 and 6731 components to match our SII column.
spz_oii_fl  = aux['SPZ_OII_3727_FLUX'].byteswap().newbyteorder()
spz_hb_fl   = aux['SPZ_H_beta_FLUX'].byteswap().newbyteorder()
spz_oiii_fl = aux['SPZ_OIII_5007_FLUX'].byteswap().newbyteorder()
spz_ha_fl   = aux['SPZ_H_alpha_FLUX'].byteswap().newbyteorder()
spz_nii_fl  = aux['SPZ_NII_6584_FLUX'].byteswap().newbyteorder()
spz_sii_fl  = (aux['SPZ_SII_6717_FLUX'].byteswap().newbyteorder() +
               aux['SPZ_SII_6731_FLUX'].byteswap().newbyteorder())
spz_tags = make_tag_from_pmf(spz_pl, spz_mj, spz_fb)
del aux, spz_pl, spz_mj, spz_fb

df_spz = pd.DataFrame({
    'tag':      spz_tags,
    'OII_sdss': spz_oii_fl, 'Hb_sdss':   spz_hb_fl,
    'OIII_sdss':spz_oiii_fl,'Ha_sdss':   spz_ha_fl,
    'NII_sdss': spz_nii_fl, 'SII_sdss':  spz_sii_fl,
})
del spz_tags, spz_oii_fl, spz_hb_fl, spz_oiii_fl, spz_ha_fl, spz_nii_fl, spz_sii_fl

# Apply per-spectrum MW E(B-V) correction using the same GCC09_MWAvg curve
# used by the eBOSS-DAP. Each line is corrected at its approximate rest
# wavelength (close enough given the narrow redshift range of the comparison).
df_ebv = df[['tag','ebv_mw']].copy()
df_spz = df_spz.merge(df_ebv, on='tag')
ext_model = dext()
for col in ['OII_sdss','Hb_sdss','OIII_sdss','Ha_sdss','NII_sdss','SII_sdss']:
    wl_map = {'OII_sdss':3728,'Hb_sdss':4861,'OIII_sdss':5007,
              'Ha_sdss':6563,'NII_sdss':6584,'SII_sdss':6725}
    wl_aa  = wl_map[col]
    corr   = ext_model.extinguish(wl_aa * u.AA, Ebv=df_spz['ebv_mw'])
    df_spz[col] = df_spz[col] / corr

# Merge eBOSS-DAP and dust-corrected spZline fluxes into one DataFrame for
# the trumpet-style comparison. comp_lines maps short names to column suffixes.
df_comp = df[['tag','OII','Hb','OIII_5008','Ha','NII_6585','SII',
               'OII_err','Hb_err','OIII_5008_err','Ha_err','NII_6585_err','SII_err']].copy()
df_comp = df_comp.merge(df_spz[['tag','OII_sdss','Hb_sdss','OIII_sdss',
                                  'Ha_sdss','NII_sdss','SII_sdss']], on='tag')
del df_spz, df_ebv

comp_lines = ['OII','Hb','OIII','Ha','NII','SII']
col_map    = {'OII':'OII','Hb':'Hb','OIII':'OIII_5008','Ha':'Ha','NII':'NII_6585','SII':'SII'}
for line in comp_lines:
    fl1 = df_comp[col_map[line]]
    fl2 = df_comp[f'{line}_sdss']
    er1 = df_comp[col_map[line]+'_err']
    df_comp[f'{line}_flux1']     = fl1
    df_comp[f'{line}_sdss']      = fl2
    df_comp[f'{line}_flux1_err'] = er1
    df_comp[f'{line}_S/N']       = fl1 / er1
    df_comp[f'{line}_diff']      = fl1 / fl2

In [ ]:
# ── Figure 16: sdss_comp ──────────────────────────────────────────────────
# Trumpet-style plot of log(F_eBOSSDAP / F_spZline) vs eBOSS-DAP S/N for
# six strong lines. The pink percentile dots should track the theoretical
# sqrt(2)/S/N orange lines if the two measurements are consistent. The
# median offset at S/N > 3 is annotated on each panel; for Hβ this is
# notably non-zero, reflecting the improved stellar continuum treatment in
# the eBOSS-DAP vs the spZline PCA approach.
scale   = 3
binning = 15 * 4
x2      = np.linspace(0, 26, binning)
x_th    = np.linspace(0.01, 26, binning)
y_th    = np.sqrt(2) / x_th
pointsz = 36 * scale

plot_prettier(150, fontsize=24)
plt.rcParams.update({'xtick.major.size':5*scale,'xtick.major.width':1.25*scale,
                     'xtick.minor.size':2.5*scale,'xtick.minor.width':1.25*scale,
                     'ytick.major.size':5*scale,'ytick.major.width':1.25*scale,
                     'ytick.minor.size':2.5*scale,'ytick.minor.width':1.25*scale})

fig, axes = plt.subplot_mosaic('ABC;DEF', figsize=(11*scale, 8.5*scale*2/3))
fig.tight_layout(pad=1.0)

comp_label_map = {
    'OII':  '[O II] Sum Flux',   'Hb':   r'H$\beta$ Flux',
    'OIII': '[O III] 5008 Flux', 'Ha':   r'H$\alpha$ Flux',
    'NII':  '[N II] 6585 Flux',  'SII':  '[S II] Sum Flux',
}

for (panel, ax), line in zip(axes.items(), comp_lines):
    d1 = df_comp[np.abs(df_comp[f'{line}_diff']) <= 50].copy()
    d2 = d1[np.abs(d1[f'{line}_S/N']) <= 50]
    d3 = d2[(d2[f'{line}_flux1'] != 0) & (d2[f'{line}_sdss'] != 0)]
    d4 = d3[(d3[f'{line}_flux1'] != -999) & (d3[f'{line}_sdss'] != -999)]
    if len(d4) == 0:
        continue

    log_diff = np.log(d4[f'{line}_diff'])
    snr_vals = d4[f'{line}_S/N']

    ax.hist2d(snr_vals, log_diff, bins=100, range=[[0,26],[-1.1,1.1]],
              cmin=1, cmap='gray_r')

    snrs_b, top_b, bot_b, mean_b = [], [], [], []
    for i, ix in enumerate(x2[1:]):
        sl = log_diff[(snr_vals > x2[i]) & (snr_vals < ix)]
        snrs_b.append((ix + x2[i]) / 2)
        top_b.append(np.nanpercentile(sl, 84.0) if len(sl) else np.nan)
        bot_b.append(np.nanpercentile(sl, 16.0) if len(sl) else np.nan)
        mean_b.append(np.nanmean(sl)            if len(sl) else np.nan)

    ax.scatter(snrs_b, top_b, c='C6', s=pointsz, label='16th/84th percentile', rasterized=True)
    ax.scatter(snrs_b, bot_b, c='C6', s=pointsz, rasterized=True)
    ax.plot(x_th,  y_th,  c='C1', lw=6, label='1$\\sigma$ theoretical')
    ax.plot(x_th, -y_th,  c='C1', lw=6)
    ax.plot(x_th,  2*y_th,c='C2', lw=6, label='2$\\sigma$ theoretical')
    ax.plot(x_th, -2*y_th,c='C2', lw=6)
    ax.hlines(0, 0, 50, color='C1')
    ax.set_xlim(0, 26); ax.set_ylim(-1.1, 1.1)

    # Median offset at S/N > 3 (ignoring the noisiest regime where both
    # methods are dominated by random error rather than systematic differences).
    offset = np.nanmedian(mean_b[7:])
    ax.annotate(f'{panel}: {comp_label_map[line]}\nOffset = {offset:.3f}',
                xy=(1,0), xycoords='axes fraction',
                xytext=(-0.25, +2.75), textcoords='offset fontsize',
                fontsize='large', va='top', ha='right', fontfamily='serif',
                bbox=dict(facecolor='none', edgecolor='none', pad=3.0))

plt.subplots_adjust(wspace=0.2, hspace=0.2)
fig.supxlabel('Signal-to-Noise Ratio$_{\\mathrm{eBOSS-DAP}}$', y=-0.07, fontsize=48)
fig.supylabel('log ( Flux$_{\\mathrm{eBOSS-DAP}}$ / Flux$_{\\mathrm{SDSS}}$ )',
              x=-0.05, fontsize=48)
axes['C'].legend(loc=1)
fig.patch.set_rasterized(True)
plt.savefig(os.path.join(OUTPUT_DIR,'sdss_comp.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'sdss_comp.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── sdss_comp summary table (released alongside the paper) ────────────────
# Writes per-S/N-bin percentile statistics for the sdss_comp figure to CSV.
# Released as a supplementary table per the Data Availability section.
binning_t = 15*4
x2_t = np.linspace(0, 26, binning_t)
tdict = {'SNR_bin': []}
for line in comp_lines:
    d1 = df_comp[np.abs(df_comp[f'{line}_diff'])<=50]
    d2 = d1[np.abs(d1[f'{line}_S/N'])<=50]
    d3 = d2[(d2[f'{line}_flux1']!=0)&(d2[f'{line}_sdss']!=0)]
    d4 = d3[(d3[f'{line}_flux1']!=-999)&(d3[f'{line}_sdss']!=-999)]
    ld = np.log(d4[f'{line}_diff']); sv = d4[f'{line}_S/N']
    snrs_t,meds_t,p16_t,p84_t,p023_t,p977_t = [],[],[],[],[],[]
    for i,ix in enumerate(x2_t[1:]):
        sl = ld[(sv>x2_t[i])&(sv<ix)]
        snrs_t.append((ix+x2_t[i])/2)
        meds_t.append(np.nanmedian(sl) if len(sl) else np.nan)
        p16_t.append(np.nanpercentile(sl,15.7) if len(sl) else np.nan)
        p84_t.append(np.nanpercentile(sl,83.97) if len(sl) else np.nan)
        p023_t.append(np.nanpercentile(sl,2.3) if len(sl) else np.nan)
        p977_t.append(np.nanpercentile(sl,97.7) if len(sl) else np.nan)
    if not tdict['SNR_bin']: tdict['SNR_bin']=snrs_t
    tdict[f'{line}_median']=meds_t; tdict[f'{line}_p16']=p16_t
    tdict[f'{line}_p84']=p84_t; tdict[f'{line}_p023']=p023_t; tdict[f'{line}_p977']=p977_t
tdf = pd.DataFrame(tdict)
tdf.to_csv(os.path.join(OUTPUT_DIR,'sdss_comp_table.csv'), index=False)
print(tdf.to_latex(index=False, float_format='%.4f'))
print('sdss_comp_table.csv saved.')

In [ ]:
# ── Section 14 cleanup ───────────────────────────────────────────────────
del df_comp

# Section 15 — Figures 17 & 18: Repeat Spectra Trumpets
### *How well do the error bars actually describe the scatter?*
*Uses `savedf` from Section 11 to compare line flux (and Hβ EW) measurements between repeat observations of the same galaxy. The "trumpet" shape — scatter narrowing with S/N — is the expected pattern if errors are correctly estimated; deviations at high S/N reveal the spectrophotometric noise floor that formal errors can't capture. Fig. 17 covers six strong optical lines; Fig. 18 zooms in on three intrinsically weak lines ([Ne V], [O III] 4364, He II) where the statistics are thinner. This section also fits and stores `scale_factors`, the per-line error inflation terms reported in Table 3. **Requires:** Sections 0 and 11.*

**Fig. 17** `trumpet_plots_strong_avg.pdf` — strong lines (OII, Hβ flux, Hβ EW, OIII, NII, Hα).

**Fig. 18** `trumpet_plots_weak_avg.pdf` — weak lines ([Ne V], [O III] 4364, He II).

In [ ]:
# ── Load matched repeat-spectra flux CSV ─────────────────────────────────

# DIAGNOSTIC: check for 'disp'-related columns (Fig 19 debug -- 'disp not
# available' in trumpet_panel means 'disp_x'/'disp_y' aren't in savedf.columns
# under that exact name; this prints what IS there so we can see whether it's
# e.g. 'SC_disp_x'/'SC_disp_y' instead).
_disp_cols = [c for c in savedf.columns if 'disp' in c.lower()]
print('disp-related columns in savedf:', _disp_cols)
del _disp_cols

# Derived ratio columns (R3, O23, N2, Ha/Hb) and their propagated errors,
# per eBOSS_error_analysis.ipynb -- these must exist before the trump_lines
# loop below, since R3/O23/N2/Ha/Hb's '_err_x'/'_err_y' are NOT raw FITS
# columns but error-propagated from the underlying line fluxes.
savedf['R3_x']  = np.divide(savedf['OIII_5008_x'], savedf['Hb_x'])
savedf['R3_y']  = np.divide(savedf['OIII_5008_y'], savedf['Hb_y'])
savedf['O23_x'] = np.divide(savedf['OII_x'],  savedf['OIII_5008_x'])
savedf['O23_y'] = np.divide(savedf['OII_y'],  savedf['OIII_5008_y'])
savedf['N2_x']  = np.divide(savedf['NII_6585_x'], savedf['Ha_x'])
savedf['N2_y']  = np.divide(savedf['NII_6585_y'], savedf['Ha_y'])
savedf['Ha/Hb_x'] = np.divide(savedf['Ha_x'], savedf['Hb_x'])
savedf['Ha/Hb_y'] = np.divide(savedf['Ha_y'], savedf['Hb_y'])

savedf['R3_err_x'] = savedf['R3_x'] * np.sqrt((savedf['OIII_5008_err_x']/savedf['OIII_5008_x'])**2 + (savedf['Hb_err_x']/savedf['Hb_x'])**2)
savedf['R3_err_y'] = savedf['R3_y'] * np.sqrt((savedf['OIII_5008_err_y']/savedf['OIII_5008_y'])**2 + (savedf['Hb_err_y']/savedf['Hb_y'])**2)
savedf['O23_err_x'] = savedf['O23_x'] * np.sqrt((savedf['OII_err_x']/savedf['OII_x'])**2 + (savedf['OIII_5008_err_x']/savedf['OIII_5008_x'])**2)
savedf['O23_err_y'] = savedf['O23_y'] * np.sqrt((savedf['OII_err_y']/savedf['OII_y'])**2 + (savedf['OIII_5008_err_x']/savedf['OIII_5008_x'])**2)
savedf['N2_err_x'] = savedf['N2_x'] * np.sqrt((savedf['NII_6585_err_x']/savedf['NII_6585_x'])**2 + (savedf['Ha_err_x']/savedf['Ha_x'])**2)
savedf['N2_err_y'] = savedf['N2_y'] * np.sqrt((savedf['NII_6585_err_y']/savedf['NII_6585_y'])**2 + (savedf['Ha_err_y']/savedf['Ha_y'])**2)
savedf['Ha/Hb_err_x'] = savedf['Ha/Hb_x'] * np.sqrt((savedf['Ha_err_x']/savedf['Ha_x'])**2 + (savedf['Hb_err_x']/savedf['Hb_x'])**2)
savedf['Ha/Hb_err_y'] = savedf['Ha/Hb_y'] * np.sqrt((savedf['Ha_err_y']/savedf['Ha_y'])**2 + (savedf['Hb_err_y']/savedf['Hb_y'])**2)

# Derive ratio/diff/S-N/flux/zspeed columns for every relevant line
# ── Instrumental-resolution-corrected stellar velocity dispersion (Fig 19) ──
# sigma_int = sqrt(sigma_measured^2 - sigma_correction^2), per Recommended
# Usage Guidelines (Stellar Velocity Dispersion row): SC_dispersion_int =
# sqrt(SC_dispersion^2 - SC_correction^2). Error propagated via
# d(sigma_int)/d(sigma) = sigma/sigma_int.
for suf in ('_x', '_y'):
    disp = savedf['SC_disp'+suf]
    corr = savedf['SC_correction'+suf]
    disp_err = savedf['SC_disp_err'+suf]
    diff_sq = disp**2 - corr**2
    disp_int = np.sqrt(np.where(diff_sq > 0, diff_sq, np.nan))
    savedf['SC_disp_int'+suf] = disp_int
    savedf['SC_disp_int_err'+suf] = disp_err * disp / disp_int

trump_lines = ['NeV','NeV_3347','NeV_3427','OII','NeIII_3869',
               'OIII_4364','OIII_4960','HeII_4687','HeI_5877','Hb','Hb_ew',
               'OIII_5008','OI_6302','NII_6585','Ha','SII_6718','SII_6732',
               'SII','SC_disp_int','R3','N2','O23','Ha/Hb']

for line in trump_lines:
    xc = line+'_x'; yc = line+'_y'
    if xc not in savedf.columns or yc not in savedf.columns:
        continue
    savedf[line+'_diff'] = np.divide(savedf[xc], savedf[yc])
    bool_arr = np.random.choice(2, len(savedf))
    savedf[line+'_diff_rand'] = np.where(
        bool_arr == 1,
        savedf[xc] / savedf[yc],
        savedf[yc] / savedf[xc])
    savedf[line+'_S/N1']  = savedf[xc] / savedf[line+'_err_x']
    savedf[line+'_S/N2']  = savedf[yc] / savedf[line+'_err_y']
    savedf[line+'_flux1'] = savedf[xc]
    savedf[line+'_flux2'] = savedf[yc]
    savedf[line+'_zspeed1'] = savedf['z_x'] * 299792
    savedf[line+'_zspeed2'] = savedf['z_y'] * 299792
print(f'Matched pairs loaded: {len(savedf):,}')


In [ ]:
def trumpet_panel(ax, df_t, line, snr_max=21, binning=45, dotcolor='C6', pointsz=78):
    """Draw a single trumpet-plot panel."""
    x2 = np.linspace(0, snr_max, binning)
    x  = np.linspace(0.01, snr_max, binning)
    y  = np.sqrt(2) / x

    if line+'_diff_rand' not in df_t.columns:
        ax.text(0.5, 0.5, f'{line} not available', transform=ax.transAxes, ha='center')
        return

    d1 = df_t[np.abs(df_t[line+'_diff_rand'] - 1) < 49].copy()
    d2 = d1[np.abs(d1[line+'_S/N1']) <= snr_max]
    d3 = d2[(d2[line+'_flux1'] != 0) & (d2[line+'_flux2'] != 0)]
    d4 = d3[(d3[line+'_flux1'] != -999) & (d3[line+'_flux2'] != -999)].copy()
    d4[line+'_S/N'] = (d4[line+'_S/N1'] + d4[line+'_S/N2']) / 2.0

    log_r  = np.log(np.abs(d4[line+'_diff_rand']))
    snr_v  = d4[line+'_S/N']

    ax.hist2d(snr_v, log_r, bins=100, range=[[0,snr_max],[-2.2,2.2]],
              cmin=1, cmap='gray_r')

    snrs_b, top_b, bot_b = [], [], []
    for i, ix in enumerate(x2[1:]):
        sl = log_r[(snr_v > x2[i]) & (snr_v < ix)]
        snrs_b.append((ix + x2[i]) / 2)
        top_b.append(np.nanpercentile(sl, 68.27 + 15.7) if len(sl) else np.nan)
        bot_b.append(np.nanpercentile(sl, 15.7)          if len(sl) else np.nan)

    ax.scatter(snrs_b, top_b, c=dotcolor, s=pointsz, rasterized=True)
    ax.scatter(snrs_b, bot_b, c=dotcolor, s=pointsz, rasterized=True,
               label='16th / 84th percentile')
    ax.plot(x,  y,  c='C1', lw=6, label='1$\\sigma$ theoretical')
    ax.plot(x, -y,  c='C1', lw=6)
    ax.plot(x,  2*y,c='C2', lw=6, label='2$\\sigma$ theoretical')
    ax.plot(x, -2*y,c='C2', lw=6)
    ax.hlines(0, 0, snr_max, color='C1')
    ax.set_xlim(0, snr_max)
    ax.set_ylim(-2.2, 2.2)


In [ ]:
# ── Figure 17: trumpet_plots_strong_avg ───────────────────────────────────
scale = 3
plot_prettier(150, fontsize=24)
plt.rcParams.update({'xtick.major.size':5*scale,'xtick.major.width':1.25*scale,
                     'xtick.minor.size':2.5*scale,'xtick.minor.width':1.25*scale,
                     'ytick.major.size':5*scale,'ytick.major.width':1.25*scale,
                     'ytick.minor.size':2.5*scale,'ytick.minor.width':1.25*scale})

strong_lines  = ['OII','Hb','Hb_ew','OIII_5008','NII_6585','Ha']
strong_labels = ['[O II] sum', r'H$\beta$ Flux', r'H$\beta$ EW',
                 '[O III] 5008 Flux', '[N II] 6585 Flux', r'H$\alpha$ Flux']

fig, axes = plt.subplot_mosaic('ABC;DEF', figsize=(11*scale, 8.5*scale*2/3))
fig.tight_layout(pad=1.0)

for (panel, ax), line, ll in zip(axes.items(), strong_lines, strong_labels):
    trumpet_panel(ax, savedf, line, snr_max=21,
                  binning=15*3, dotcolor='C6', pointsz=26*scale)
    ax.annotate(f'{panel}: {ll}',
                xy=(1,1), xycoords='axes fraction',
                xytext=(-0.25,-0.25), textcoords='offset fontsize',
                fontsize='large', va='top', ha='right', fontfamily='serif',
                bbox=dict(facecolor='none', edgecolor='none', pad=3.0))

plt.subplots_adjust(wspace=0.2, hspace=0.2)
fig.supxlabel('Signal-to-Noise Ratio', y=-0.07, fontsize=48)
fig.supylabel('log (Flux$_1$ / Flux$_2$)', x=-0.02, fontsize=48)
plt.savefig(os.path.join(OUTPUT_DIR,'trumpet_plots_strong_avg.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'trumpet_plots_strong_avg.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── trumpet_strong summary table (for Fig 17) ────────────────────────────
binning_t = 15*3
x2_t = np.linspace(0, 21, binning_t)
tdict = {'SNR_bin': []}
for line in strong_lines:
    if line+'_diff_rand' not in savedf.columns: continue
    d1 = savedf[np.abs(savedf[line+'_diff_rand']-1)<49].copy()
    d2 = d1[(d1[line+'_flux1']!=0)&(d1[line+'_flux2']!=0)]
    d3 = d2[(d2[line+'_flux1']!=-999)&(d2[line+'_flux2']!=-999)]
    lr = np.log(np.abs(d3[line+'_diff_rand'])); sv = d3[line+'_S/N1']
    snrs_t,meds_t,p16_t,p84_t,p023_t,p977_t = [],[],[],[],[],[]
    for i,ix in enumerate(x2_t[1:]):
        sl = lr[(sv>x2_t[i])&(sv<ix)]
        snrs_t.append((ix+x2_t[i])/2)
        meds_t.append(np.nanmedian(sl) if len(sl) else np.nan)
        p16_t.append(np.nanpercentile(sl,15.7) if len(sl) else np.nan)
        p84_t.append(np.nanpercentile(sl,83.97) if len(sl) else np.nan)
        p023_t.append(np.nanpercentile(sl,2.3) if len(sl) else np.nan)
        p977_t.append(np.nanpercentile(sl,97.7) if len(sl) else np.nan)
    if not tdict['SNR_bin']: tdict['SNR_bin']=snrs_t
    tdict[f'{line}_median']=meds_t; tdict[f'{line}_p16']=p16_t
    tdict[f'{line}_p84']=p84_t; tdict[f'{line}_p023']=p023_t; tdict[f'{line}_p977']=p977_t
tdf = pd.DataFrame(tdict)
tdf.to_csv(os.path.join(OUTPUT_DIR,'trumpet_plots_strong_avg_table.csv'), index=False)
print(tdf.to_latex(index=False, float_format='%.4f'))
print('trumpet_plots_strong_avg_table.csv saved.')


In [ ]:
# ── Figure 18: trumpet_plots_weak_avg ─────────────────────────────────────
# Average S/N = (S/N1+S/N2)/2; individual points below threshold; x-range 0-11
scale   = 3
binning = 15 * 2
x2      = np.linspace(0, 11, binning)
x_th    = np.linspace(0.01, 11, binning)
y_th    = np.sqrt(2) / x_th
pointsz = 32 * scale

plot_prettier(150, fontsize=24)
plt.rcParams.update({'xtick.major.size':5*scale,'xtick.major.width':1.25*scale,
                     'xtick.minor.size':2.5*scale,'xtick.minor.width':1.25*scale,
                     'ytick.major.size':5*scale,'ytick.major.width':1.25*scale,
                     'ytick.minor.size':2.5*scale,'ytick.minor.width':1.25*scale})

weak_lines     = ['NeV','OIII_4364','HeII_4687']
weak_labels    = ['[Ne V] Flux', '[O III] 4364 Flux', 'He II 4687 Flux']
weak_thresholds= [7, 5, 4]   # individual points plotted above these S/N values

fig, axes = plt.subplot_mosaic('ABC', figsize=(11*scale, 8.5*scale/3))
fig.tight_layout(pad=-1.0)

for (panel, ax), line, ll, pt_thr in zip(
        axes.items(), weak_lines, weak_labels, weak_thresholds):

    if line+'_diff_rand' not in savedf.columns:
        ax.text(0.5, 0.5, f'{line} not available', transform=ax.transAxes, ha='center')
        continue

    d1 = savedf[np.abs(savedf[line+'_diff_rand'] - 1) < 49].copy()
    d2 = d1[(d1[line+'_flux1'] != 0) & (d1[line+'_flux2'] != 0)]
    d3 = d2[(d2[line+'_flux1'] != -999) & (d2[line+'_flux2'] != -999)].copy()
    d3[line+'_S/N'] = (d3[line+'_S/N1'] + d3[line+'_S/N2']) / 2.0
    log_r = np.log(np.abs(d3[line+'_diff_rand']))
    snr_v = d3[line+'_S/N']

    ax.hist2d(snr_v, log_r, bins=100, range=[[0,11],[-2.2,2.2]], cmin=1, cmap='gray_r')

    # Individual points above S/N threshold (small black dots, alpha 0.25)
    dfmini = d3[d3[line+'_S/N'] >= pt_thr]
    ax.scatter(dfmini[line+'_S/N'],
               np.log(np.abs(dfmini[line+'_diff_rand'])),
               s=pointsz/8, c='k', alpha=0.25, rasterized=True)

    snrs_b, top_b, bot_b = [], [], []
    for i_b, ix in enumerate(x2[1:]):
        sl = log_r[(snr_v > x2[i_b]) & (snr_v < ix)]
        snrs_b.append((ix + x2[i_b]) / 2)
        top_b.append(np.nanpercentile(sl, 68.27+15.7) if len(sl) else np.nan)
        bot_b.append(np.nanpercentile(sl, 15.7)        if len(sl) else np.nan)

    ax.scatter(snrs_b, top_b, c='C6', s=pointsz, rasterized=True,
               label='16th/84th percentile')
    ax.scatter(snrs_b, bot_b, c='C6', s=pointsz, rasterized=True)
    ax.plot(x_th,  y_th,  c='C1', lw=6, label='1$\sigma$ theoretical')
    ax.plot(x_th, -y_th,  c='C1', lw=6)
    ax.plot(x_th,  2*y_th,c='C2', lw=6)
    ax.plot(x_th, -2*y_th,c='C2', lw=6)
    ax.hlines(0, 0, 11, color='C1')
    ax.set_xlim(0, 11); ax.set_ylim(-2.2, 2.2)
    ax.annotate(f'{panel}: {ll}',
                xy=(1,0), xycoords='axes fraction',
                xytext=(-0.25,+1.25), textcoords='offset fontsize',
                fontsize='large', va='bottom', ha='right', fontfamily='serif',
                bbox=dict(facecolor='white', edgecolor='none', pad=3.0))

axes['B'].set_yticklabels([])
axes['C'].set_yticklabels([])
axes['C'].legend(loc=1)
plt.subplots_adjust(wspace=0.03, hspace=0.05)
fig.supxlabel('Signal-to-Noise Ratio', y=-0.15, fontsize=48)
fig.supylabel('log (Flux$_1$ / Flux$_2$)', x=-0.05, fontsize=48)
fig.patch.set_rasterized(True)
plt.savefig(os.path.join(OUTPUT_DIR,'trumpet_plots_weak_avg.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'trumpet_plots_weak_avg.png'), dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── trumpet_weak summary table (for Fig 18) ──────────────────────────────
binning_tw = 15*2
x2_tw = np.linspace(0, 11, binning_tw)
tdict = {'SNR_bin': []}
for line in weak_lines:
    if line+'_diff_rand' not in savedf.columns: continue
    d1 = savedf[np.abs(savedf[line+'_diff_rand']-1)<49].copy()
    d2 = d1[(d1[line+'_flux1']!=0)&(d1[line+'_flux2']!=0)]
    d3 = d2[(d2[line+'_flux1']!=-999)&(d2[line+'_flux2']!=-999)].copy()
    d3[line+'_S/N'] = (d3[line+'_S/N1']+d3[line+'_S/N2'])/2.0
    lr = np.log(np.abs(d3[line+'_diff_rand'])); sv = d3[line+'_S/N']
    snrs_t,meds_t,p16_t,p84_t,p023_t,p977_t = [],[],[],[],[],[]
    for i,ix in enumerate(x2_tw[1:]):
        sl = lr[(sv>x2_tw[i])&(sv<ix)]
        snrs_t.append((ix+x2_tw[i])/2)
        meds_t.append(np.nanmedian(sl) if len(sl) else np.nan)
        p16_t.append(np.nanpercentile(sl,15.7) if len(sl) else np.nan)
        p84_t.append(np.nanpercentile(sl,83.97) if len(sl) else np.nan)
        p023_t.append(np.nanpercentile(sl,2.3) if len(sl) else np.nan)
        p977_t.append(np.nanpercentile(sl,97.7) if len(sl) else np.nan)
    if not tdict['SNR_bin']: tdict['SNR_bin']=snrs_t
    tdict[f'{line}_median']=meds_t; tdict[f'{line}_p16']=p16_t
    tdict[f'{line}_p84']=p84_t; tdict[f'{line}_p023']=p023_t; tdict[f'{line}_p977']=p977_t
tdf = pd.DataFrame(tdict)
tdf.to_csv(os.path.join(OUTPUT_DIR,'trumpet_plots_weak_avg_table.csv'), index=False)
print(tdf.to_latex(index=False, float_format='%.4f'))
print('trumpet_plots_weak_avg_table.csv saved.')


In [ ]:
# ── Paper Table 1: Uncertainty Scaling Factors ───────────────────────────
# Fits sigma_measured(S/N) = sqrt((sqrt(2)/S/N)^2 + (sqrt(2)*a)^2)
# to the binned 1-sigma scatter from the repeat-spectra trumpet plots,
# weighted by 1/sqrt(S/N), per eBOSS_error_analysis.ipynb's final
# fitting block (func_new / curve_fit). The fitted 'a' is the additive
# spectrophotometric floor reported in Table 1 of the paper:
#   sigma_corrected = sqrt(sigma_DAP^2 + (a * F)^2)
# For sigma_* the scaling is multiplicative:
#   sigma_*_corrected = sigma_*_DAP * a
# (fit via func_disp(x,a) = a*sqrt(2)/x instead of func_new)
from scipy.optimize import curve_fit

def _sigma_model(snr, a):
    """Additive floor model: sigma = sqrt((sqrt(2)/S/N)^2 + (sqrt(2)*a)^2)"""
    return np.sqrt((np.sqrt(2) / snr)**2 + (np.sqrt(2) * a)**2)

def _disp_model(snr, a):
    """Multiplicative scaling model for sigma_*: sigma = a*sqrt(2)/S/N"""
    return a * np.sqrt(2) / snr

# Lines to fit -- (savedf column prefix, paper label)
_scale_lines = [
    # Table 1 row order matches the paper exactly
    ('NeV_3347',   '[Ne V] 3347'),
    ('NeV_3427',   '[Ne V] 3427'),
    ('OII',        '[O II] 3727 + 3729'),
    ('NeIII_3869', '[Ne III] 3870'),
    ('OIII_4364',  '[O III] 4364'),
    ('HeII_4687',  'He II 4687'),
    ('Hb',         r'H$\beta$'),
    ('OIII_4960',  '[O III] 4960'),
    ('OIII_5008',  '[O III] 5008'),
    ('HeI_5877',   'He I 5877'),
    ('OI_6302',    '[O I] 6302'),
    ('Ha',         r'H$\alpha$'),
    ('NII_6585',   '[N II] 6585'),
    ('SII_6718',   '[S II] 6718'),
    ('SII_6732',   '[S II] 6732'),
    # Ratios -- per LaTeX macros
    ('O23',        'log (([O II] 3727 + 3729) / [O III] 5008)'),
    ('R3',         r'log ([O III] 5008 / H$\beta$)'),
    ('N2',         r'log ([N II] 6585 / H$\alpha$)'),
    ('Ha/Hb',      r'H$\alpha$ / H$\beta$'),
]

binning_fit = 45
x2_fit = np.linspace(0, 21, binning_fit)

scale_factors = {}   # line_key -> (a_value, paper_label)

for line, label in _scale_lines:
    col_diff  = line + '_diff_rand'
    col_snr1  = line + '_S/N1'
    col_snr2  = line + '_S/N2'
    col_flux1 = line + '_flux1'    # = line_x
    col_flux2 = line + '_flux2'    # = line_y
    col_err1  = line + '_err_x'    # error on flux1

    if col_diff not in savedf.columns:
        print(f'Skipping {line}: {col_diff} not in savedf')
        continue

    df1 = savedf[np.abs(savedf[col_diff]) <= 50].copy()
    df1['_snr_avg'] = (df1[col_snr1] + df1[col_snr2]) / 2.0
    df1 = df1[
        df1['_snr_avg'].between(3, 25) &
        (df1[col_flux1] > 0) & (df1[col_flux2] > 0) &
        (df1[col_flux1] != -999.0) & (df1[col_flux2] != -999.0)
    ]

    snrs, stds = [], []
    for i, ix in enumerate(x2_fit[1:]):
        dfslice = df1[(df1['_snr_avg'] >= x2_fit[i]) & (df1['_snr_avg'] < ix)]
        snrs.append((ix + x2_fit[i]) / 2)
        if len(dfslice) > 0:
            vals = np.log(dfslice[col_diff].to_numpy(dtype=float))
            vals = vals[np.isfinite(vals)]
            if len(vals) > 0:
                mean_val = np.mean(vals)
                p84 = np.nanpercentile(vals, 68.27 + 15.7)
                p16 = np.nanpercentile(vals, 15.7)
                scatter = (abs(p84 - mean_val) + abs(mean_val - p16)) / 2.0
                stds.append(scatter if scatter > 0 else np.nan)
            else:
                stds.append(np.nan)
        else:
            stds.append(np.nan)

    xdata = np.array(snrs)
    ydata = np.array(stds)
    weights = 1.0 / np.sqrt(xdata)
    valid = np.isfinite(ydata) & (xdata > 0)
    if valid.sum() < 3:
        print(f'{line}: insufficient valid bins for fitting')
        continue
    try:
        popt, _ = curve_fit(_sigma_model, xdata[valid], ydata[valid],
                            sigma=weights[valid], p0=[0.1])
        scale_factors[line] = (float(popt[0]), label)
        print(f'{label:35s}  a = {popt[0]:.3f}')
    except RuntimeError:
        print(f'{line}: curve_fit failed')

# Stellar velocity dispersion -- multiplicative correction
# sigma_* is itself a dispersion, so the correction is multiplicative:
#   sigma_*_corrected = sigma_*_DAP * a
# Model: scatter(S/N) = a * sqrt(2) / S/N
# Uses same percentile-based scatter as emission lines (not nanstd).
col_disp_diff  = 'SC_disp_int_diff_rand'
col_disp_snr1  = 'SC_disp_int_S/N1'
col_disp_snr2  = 'SC_disp_int_S/N2'
col_disp_flux1 = 'SC_disp_int_x'
col_disp_flux2 = 'SC_disp_int_y'
if col_disp_diff in savedf.columns:
    df_d = savedf[
        (savedf[col_disp_flux1] > 0) & (savedf[col_disp_flux2] > 0) &
        (savedf[col_disp_flux1] != -999.0)
    ].copy()
    df_d['_snr_avg'] = (df_d[col_disp_snr1] + df_d[col_disp_snr2]) / 2.0
    df_d = df_d[df_d['_snr_avg'].between(5, 25)]
    snrs_d, stds_d = [], []
    for i, ix in enumerate(x2_fit[1:]):
        dfslice = df_d[(df_d['_snr_avg'] >= x2_fit[i]) & (df_d['_snr_avg'] < ix)]
        snrs_d.append((ix + x2_fit[i]) / 2)
        if len(dfslice) > 0:
            vals = np.log(dfslice[col_disp_diff].to_numpy(dtype=float))
            vals = vals[np.isfinite(vals)]
            if len(vals) > 0:
                mean_val = np.mean(vals)
                p84 = np.nanpercentile(vals, 68.27 + 15.7)
                p16 = np.nanpercentile(vals, 15.7)
                scatter = (abs(p84 - mean_val) + abs(mean_val - p16)) / 2.0
                stds_d.append(scatter if scatter > 0 else np.nan)
            else:
                stds_d.append(np.nan)
        else:
            stds_d.append(np.nan)
    xdata_d = np.array(snrs_d)
    ydata_d = np.array(stds_d)
    valid_d = np.isfinite(ydata_d) & (xdata_d > 0)
    if valid_d.sum() >= 5:
        popt_d, _ = curve_fit(_disp_model, xdata_d[valid_d], ydata_d[valid_d],
                       sigma=1/np.sqrt(xdata_d[valid_d]), p0=[1.0])
        disp_scale = float(popt_d[0])
        scale_factors['SC_disp_int'] = (disp_scale, 'sigma_star')
        print(f'{"sigma_star (multiplicative)":35s}  a = {disp_scale:.3f}')
    else:
        disp_scale = 1.0
        print(f'sigma_star: insufficient bins, using fallback {disp_scale}')

# ── Paper Table 1: write CSV ─────────────────────────────────────────────
rows = ['Name,a']
for line, (a, label) in scale_factors.items():
    rows.append(f'{label},{a:.3f}')
rows.append(f'sigma_star,{disp_scale:.3f}')
txt = (
    '% Paper Table 1: Uncertainty Scaling (table:error_scale)\n'
    '% Fitted from repeat-spectra trumpet plots (Figs 17-19).\n'
    '% Additive floor: sigma_corrected = sqrt(sigma_DAP^2 + (a*F)^2)\n'
    '% Stellar dispersion: same additive model -- sigma_*_corrected = sqrt(sigma_*_DAP^2 + (a*sigma_*_DAP/S*N)^2)... see Table 1\n'
) + '\n'.join(rows) + '\n'
with open(os.path.join(OUTPUT_DIR, 'table_error_scale.txt'), 'w') as f:
    f.write(txt)
print('\nPaper Table 1 (table_error_scale.txt) written.')


In [ ]:
# ── Diagnostic: sigma_star multiplicative factor by S/N range ────────────
# Tests whether the dispersion correction is only needed at low S/N,
# per Tremonti's comment. Bins of width 2.5 starting from 0.
col_disp_diff  = 'SC_disp_int_diff_rand'
col_disp_snr1  = 'SC_disp_int_S/N1'
col_disp_snr2  = 'SC_disp_int_S/N2'
col_disp_flux1 = 'SC_disp_int_x'
col_disp_flux2 = 'SC_disp_int_y'

df_d = savedf[
    (savedf[col_disp_flux1] > 0) & (savedf[col_disp_flux2] > 0) &
    (savedf[col_disp_flux1] != -999.0)
].copy()
df_d['_snr_avg'] = (df_d[col_disp_snr1] + df_d[col_disp_snr2]) / 2.0

bin_edges = np.arange(0, 25.1, 2.5)
for lo, hi in zip(bin_edges[:-1], bin_edges[1:]):
    sub = df_d[df_d['_snr_avg'].between(lo, hi)]
    vals = np.log(sub[col_disp_diff].to_numpy(dtype=float))
    vals = vals[np.isfinite(vals)]
    if len(vals) == 0:
        print(f"S/N {lo:5.1f}-{hi:5.1f}: no data")
        continue
    mean_val = np.mean(vals)
    p84 = np.nanpercentile(vals, 68.27 + 15.7)
    p16 = np.nanpercentile(vals, 15.7)
    scatter = (abs(p84 - mean_val) + abs(mean_val - p16)) / 2.0
    snr_mid = sub['_snr_avg'].median()
    theory = np.sqrt(2) / snr_mid
    a_local = scatter / theory
    print(f"S/N {lo:5.1f}-{hi:5.1f}: n={len(vals):6,}  median S/N={snr_mid:6.2f}  "
          f"scatter={scatter:.4f}  sqrt(2)/S/N={theory:.4f}  a={a_local:.3f}")


In [ ]:
# ── Fit Hb_ew scale factor separately ────────────────────────────────────
# Hb_ew is needed for the corrected trumpet plots (Fig 17) but is not a
# Paper Table 1 entry since EW scaling is not reported there.
for line, label in [('Hb_ew', 'Hbeta EW')]:
    col_diff  = line + '_diff_rand'
    col_snr1  = line + '_S/N1'
    col_snr2  = line + '_S/N2'
    col_flux1 = line + '_flux1'
    col_flux2 = line + '_flux2'

    if col_diff not in savedf.columns:
        print(f'Skipping {line}: {col_diff} not in savedf')
        continue
    df1 = savedf[np.abs(savedf[col_diff]) <= 50].copy()
    df1['_snr_avg'] = (df1[col_snr1] + df1[col_snr2]) / 2.0
    df1 = df1[
        df1['_snr_avg'].between(3, 25) &
        (df1[col_flux1] > 0) & (df1[col_flux2] > 0) &
        (df1[col_flux1] != -999.0) & (df1[col_flux2] != -999.0)
    ]
    snrs, stds = [], []
    for i, ix in enumerate(x2_fit[1:]):
        dfslice = df1[(df1['_snr_avg'] >= x2_fit[i]) & (df1['_snr_avg'] < ix)]
        snrs.append((ix + x2_fit[i]) / 2)
        if len(dfslice) > 0:
            vals = np.log(dfslice[col_diff].to_numpy(dtype=float))
            vals = vals[np.isfinite(vals)]
            if len(vals) > 0:
                mean_val = np.mean(vals)
                p84 = np.nanpercentile(vals, 68.27 + 15.7)
                p16 = np.nanpercentile(vals, 15.7)
                scatter = (abs(p84 - mean_val) + abs(mean_val - p16)) / 2.0
                stds.append(scatter if scatter > 0 else np.nan)
            else:
                stds.append(np.nan)
        else:
            stds.append(np.nan)
    xdata = np.array(snrs)
    ydata = np.array(stds)
    weights = 1.0 / np.sqrt(xdata)
    valid = np.isfinite(ydata) & (xdata > 0)
    if valid.sum() >= 3:
        popt, _ = curve_fit(_sigma_model, xdata[valid], ydata[valid],
                            sigma=weights[valid], p0=[0.1])
        scale_factors[line] = (float(popt[0]), label)
        print(f'{label:35s}  a = {popt[0]:.3f}  (trumpet plot only, not in Table 1)')


In [ ]:
# ── Test of scale factors: trumpet plots with scaled 1-sigma orange lines ──
# Identical to Figs 17 & 18 (original trumpet_panel) but:
#   - orange lines show the model prediction sqrt(2/S/N^2 + 2a^2)
#     instead of the pure shot-noise sqrt(2)/S/N
#   - green 2-sigma lines removed
# If the scale factors are correct, the pink percentile dots should lie
# exactly on the scaled orange lines at all S/N.
# Uses avg S/N = (S/N1 + S/N2) / 2 on x-axis, matching trumpet_panel.

def trumpet_panel_scaled_lines(ax, df_t, line, scale_a,
                                snr_max=21, binning=45,
                                dotcolor='C6', pointsz=78):
    x2 = np.linspace(0, snr_max, binning)
    x  = np.linspace(0.01, snr_max, binning)

    # Scaled 1-sigma envelope: sqrt(2/S/N^2 + 2a^2)
    y_scaled = np.sqrt(2.0/x**2 + 2.0*scale_a**2)

    col_diff  = line + '_diff_rand'
    col_snr1  = line + '_S/N1'
    col_snr2  = line + '_S/N2'
    col_flux  = line + '_flux1'
    col_flux2 = line + '_flux2'

    if col_diff not in df_t.columns:
        ax.text(0.5, 0.5, f'{line} not available',
                transform=ax.transAxes, ha='center', va='center')
        return

    d1 = df_t[np.abs(df_t[col_diff] - 1) < 49].copy()
    d1 = d1[
        (d1[col_flux]  != 0) & (d1[col_flux]  != -999) &
        (d1[col_flux2] != 0) & (d1[col_flux2] != -999)
    ].copy()

    d1['_snr_avg'] = (d1[col_snr1] + d1[col_snr2]) / 2.0
    d1 = d1[np.abs(d1['_snr_avg']) <= snr_max]

    log_r = np.log(np.abs(d1[col_diff]))
    snr_v = d1['_snr_avg']

    ax.hist2d(snr_v, log_r, bins=100, range=[[0, snr_max], [-2.2, 2.2]],
              cmin=1, cmap='gray_r')

    snrs_b, top_b, bot_b = [], [], []
    for i, ix in enumerate(x2[1:]):
        sl = log_r[(snr_v > x2[i]) & (snr_v < ix)]
        snrs_b.append((ix + x2[i]) / 2)
        top_b.append(np.nanpercentile(sl, 68.27 + 15.7) if len(sl) else np.nan)
        bot_b.append(np.nanpercentile(sl, 15.7)          if len(sl) else np.nan)

    ax.scatter(snrs_b, top_b, c=dotcolor, s=pointsz, rasterized=True)
    ax.scatter(snrs_b, bot_b, c=dotcolor, s=pointsz, rasterized=True,
               label='16th / 84th percentile')
    ax.plot(x,  y_scaled, c='C1', lw=6, label=r'1$\sigma$ (corrected)')
    ax.plot(x, -y_scaled, c='C1', lw=6)
    ax.hlines(0, 0, snr_max, color='C1')
    ax.set_xlim(0, snr_max)
    ax.set_ylim(-2.2, 2.2)


plot_prettier(150, fontsize=24)
scale = 3
plt.rcParams.update({
    'xtick.major.size': 5*scale, 'xtick.major.width': 1.25*scale,
    'xtick.minor.size': 2.5*scale, 'xtick.minor.width': 1.25*scale,
    'ytick.direction': 'in',
    'ytick.major.size': 5*scale, 'ytick.major.width': 1.25*scale,
    'ytick.minor.size': 2.5*scale, 'ytick.minor.width': 1.25*scale,
})
pointsize = 26*scale
dotcolor = 'C6'

# ── Strong lines ───────────────────────────────────────────────────────────
strong_lines  = ['OII','Hb','Hb_ew','OIII_5008','NII_6585','Ha']
strong_labels = ['[O II] sum', r'H$\beta$ Flux', r'H$\beta$ EW',
                 '[O III] 5008', '[N II] 6585', r'H$\alpha$']

fig_s, axes_s = plt.subplot_mosaic('ABC;DEF', figsize=(11*scale, 8.5*scale*2/3))
fig_s.tight_layout(pad=-1.0)

for ax_key, line, label in zip(['A','B','C','D','E','F'], strong_lines, strong_labels):
    a = scale_factors.get(line, (0.0, ''))[0]
    trumpet_panel_scaled_lines(axes_s[ax_key], savedf, line, scale_a=a,
                                snr_max=21, binning=45,
                                dotcolor=dotcolor, pointsz=pointsize)
    axes_s[ax_key].annotate(
        label, xy=(1, 0), xycoords='axes fraction',
        xytext=(-0.5, 0.5), textcoords='offset fontsize',
        fontsize=22, va='bottom', ha='right', fontfamily='serif',
        bbox=dict(facecolor='white', edgecolor='black', pad=6.0), zorder=100
    )

for key in ['B','C','E','F']:
    axes_s[key].set_yticklabels([])
axes_s['A'].set_ylabel(r'log ($F_1$ / $F_2$)', fontsize=36)
fig_s.supxlabel('Signal-to-Noise Ratio', fontsize=48, y=-0.07)
fig_s.supylabel(r'log ($F_1$ / $F_2$)', fontsize=48, x=-0.02)
plt.subplots_adjust(wspace=0.03)
plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_plots_strong_scale_test.pdf'),
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_plots_strong_scale_test.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# ── Weak lines ─────────────────────────────────────────────────────────────
weak_lines  = ['NeV', 'OIII_4364', 'HeII_4687']
weak_labels = ['[Ne V] 3427', '[O III] 4364', 'He II 4687']

fig_w, axes_w = plt.subplot_mosaic('ABC', figsize=(11*scale, 8.5*scale/3))
fig_w.tight_layout(pad=-1.0)

for ax_key, line, label in zip(['A','B','C'], weak_lines, weak_labels):
    a = scale_factors.get(line, (0.0, ''))[0]
    trumpet_panel_scaled_lines(axes_w[ax_key], savedf, line, scale_a=a,
                                snr_max=21, binning=45,
                                dotcolor=dotcolor, pointsz=pointsize)
    axes_w[ax_key].annotate(
        label, xy=(1, 0), xycoords='axes fraction',
        xytext=(-0.5, 0.5), textcoords='offset fontsize',
        fontsize=22, va='bottom', ha='right', fontfamily='serif',
        bbox=dict(facecolor='white', edgecolor='black', pad=6.0), zorder=100
    )

for key in ['B','C']:
    axes_w[key].set_yticklabels([])
axes_w['A'].set_ylabel(r'log ($F_1$ / $F_2$)', fontsize=36)
fig_w.supxlabel('Signal-to-Noise Ratio', fontsize=48, y=-0.15)
fig_w.supylabel(r'log ($F_1$ / $F_2$)', fontsize=48, x=-0.05)
plt.subplots_adjust(wspace=0.03)
plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_plots_weak_scale_test.pdf'),
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_plots_weak_scale_test.png'),
            dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── DIAGNOSTIC: trumpet plots with error-corrected S/N on x-axis ─────────
# NOT a paper figure. Second validation step: re-draws Figs 17 & 18 using
# the corrected S/N (F / sigma_true) on the x-axis rather than the nominal
# S/N. If the scale factors are correct, the corrected plots should show the
# pink dots tracking the pure sqrt(2)/S/N orange line.
#   sigma_true = sqrt(sigma_DAP^2 + (a * F)^2)
#   S/N_true   = F / sigma_true
# Each observation's S/N is corrected independently before averaging.
# x-axis cutoff is applied on the ORIGINAL S/N (not corrected) to avoid
# a hard edge artifact from the cutoff interacting with the corrected values.

def trumpet_panel_corrected(ax, df_t, line, scale_a,
                             snr_max=21, binning=45,
                             dotcolor='C6', pointsz=78):
    """Identical to trumpet_panel but x-axis uses error-corrected S/N."""
    x2 = np.linspace(0, snr_max, binning)
    x  = np.linspace(0.01, snr_max, binning)
    y  = np.sqrt(2) / x

    col_diff  = line + '_diff_rand'
    col_flux  = line + '_flux1'
    col_err   = line + '_err_x'
    col_flux2 = line + '_flux2'
    col_err2  = line + '_err_y'
    col_snr1  = line + '_S/N1'
    col_snr2  = line + '_S/N2'

    if col_diff not in df_t.columns:
        ax.text(0.5, 0.5, f'{line} not available',
                transform=ax.transAxes, ha='center', va='center')
        return

    d1 = df_t[np.abs(df_t[col_diff] - 1) < 49].copy()
    d1 = d1[
        (d1[col_flux]  != 0)    & (d1[col_flux]  != -999) &
        (d1[col_flux2] != 0)    & (d1[col_flux2] != -999) &
        (d1[col_err]   >  0)    & (d1[col_err]   != -999) &
        (d1[col_err2]  >  0)    & (d1[col_err2]  != -999)
    ].copy()

    # Filter on original avg S/N, not corrected S/N.
    d1['_snr_orig'] = (d1[col_snr1] + d1[col_snr2]) / 2.0
    d1 = d1[np.abs(d1['_snr_orig']) <= snr_max]

    sigma_true1     = np.sqrt(d1[col_err]**2  + (scale_a * d1[col_flux])**2)
    sigma_true2     = np.sqrt(d1[col_err2]**2 + (scale_a * d1[col_flux2])**2)
    snr_true1       = d1[col_flux]  / sigma_true1
    snr_true2       = d1[col_flux2] / sigma_true2
    d1['_snr_true'] = (snr_true1 + snr_true2) / 2.0

    log_r = np.log(np.abs(d1[col_diff]))
    snr_v = d1['_snr_true']

    ax.hist2d(snr_v, log_r, bins=100, range=[[0, snr_max], [-2.2, 2.2]],
              cmin=1, cmap='gray_r')

    snrs_b, top_b, bot_b = [], [], []
    for i, ix in enumerate(x2[1:]):
        sl = log_r[(snr_v > x2[i]) & (snr_v < ix)]
        snrs_b.append((ix + x2[i]) / 2)
        top_b.append(np.nanpercentile(sl, 68.27 + 15.7) if len(sl) else np.nan)
        bot_b.append(np.nanpercentile(sl, 15.7)          if len(sl) else np.nan)

    ax.scatter(snrs_b, top_b, c=dotcolor, s=pointsz, rasterized=True)
    ax.scatter(snrs_b, bot_b, c=dotcolor, s=pointsz, rasterized=True,
               label='16th / 84th percentile')
    ax.plot(x,  y,  c='C1', lw=6, label='1$\\sigma$ theoretical')
    ax.plot(x, -y,  c='C1', lw=6)
    ax.plot(x,  2*y, c='C2', lw=6, label='2$\\sigma$ theoretical')
    ax.plot(x, -2*y, c='C2', lw=6)
    ax.hlines(0, 0, snr_max, color='C1')
    ax.set_xlim(0, snr_max)
    ax.set_ylim(-2.2, 2.2)


scale = 3
plot_prettier(150, fontsize=24)
plt.rcParams.update({
    'xtick.major.size': 5*scale, 'xtick.major.width': 1.25*scale,
    'xtick.minor.size': 2.5*scale, 'xtick.minor.width': 1.25*scale,
    'ytick.direction': 'in',
    'ytick.major.size': 5*scale, 'ytick.major.width': 1.25*scale,
    'ytick.minor.size': 2.5*scale, 'ytick.minor.width': 1.25*scale,
})
pointsize = 26*scale
dotcolor = 'C6'

strong_lines  = ['OII','Hb','Hb_ew','OIII_5008','NII_6585','Ha']
strong_labels = ['[O II] 3727+3729', r'H$\beta$', '[O III] 5008',
                 r'H$\alpha$', '[N II] 6585']

fig_s, axes_s = plt.subplot_mosaic('ABC;DEF', figsize=(11*scale, 8.5*scale*2/3))
fig_s.tight_layout(pad=-1.0)

for ax_key, line, label in zip(['A','B','C','D','E','F'], strong_lines, strong_labels):
    a = scale_factors.get(line, (0.0, ''))[0]
    trumpet_panel_corrected(axes_s[ax_key], savedf, line, scale_a=a,
                             snr_max=21, binning=45,
                             dotcolor=dotcolor, pointsz=pointsize)
    axes_s[ax_key].annotate(
        label, xy=(1, 0), xycoords='axes fraction',
        xytext=(-0.5, 0.5), textcoords='offset fontsize',
        fontsize=22, va='bottom', ha='right', fontfamily='serif',
        bbox=dict(facecolor='white', edgecolor='black', pad=6.0), zorder=100
    )

for key in ['B','C','E','F']:
    axes_s[key].set_yticklabels([])
axes_s['A'].set_ylabel(r'log ($F_1$ / $F_2$)', fontsize=36)
fig_s.supxlabel(r'Corrected S/N ($F\ /\ \sigma_{\rm true}$)', fontsize=48, y=-0.07)
fig_s.supylabel(r'log ($F_1$ / $F_2$)', fontsize=48, x=-0.02)
plt.subplots_adjust(wspace=0.03)
plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_plots_strong_corrected.pdf'),
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_plots_strong_corrected.png'),
            dpi=150, bbox_inches='tight')
plt.show()

weak_lines  = ['NeV', 'OIII_4364', 'HeII_4687']
weak_labels = ['[Ne V] 3427', '[O III] 4364', 'He II 4687']

fig_w, axes_w = plt.subplot_mosaic('ABC', figsize=(11*scale, 8.5*scale/3))
fig_w.tight_layout(pad=-1.0)

for ax_key, line, label in zip(['A','B','C'], weak_lines, weak_labels):
    a = scale_factors.get(line, (0.0, ''))[0]
    trumpet_panel_corrected(axes_w[ax_key], savedf, line, scale_a=a,
                             snr_max=21, binning=45,
                             dotcolor=dotcolor, pointsz=pointsize)
    axes_w[ax_key].annotate(
        label, xy=(1, 0), xycoords='axes fraction',
        xytext=(-0.5, 0.5), textcoords='offset fontsize',
        fontsize=22, va='bottom', ha='right', fontfamily='serif',
        bbox=dict(facecolor='white', edgecolor='black', pad=6.0), zorder=100
    )

for key in ['B','C']:
    axes_w[key].set_yticklabels([])
axes_w['A'].set_ylabel(r'log ($F_1$ / $F_2$)', fontsize=36)
fig_w.supxlabel(r'Corrected S/N ($F\ /\ \sigma_{\rm true}$)', fontsize=48, y=-0.15)
fig_w.supylabel(r'log ($F_1$ / $F_2$)', fontsize=48, x=-0.05)
plt.subplots_adjust(wspace=0.03)
plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_plots_weak_corrected.pdf'),
            dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_plots_weak_corrected.png'),
            dpi=150, bbox_inches='tight')
plt.show()

# Section 16 — Figure 19: Stellar Velocity Dispersion Repeat Spectra
### *Same test, different quantity: how reliable is σ★?*
*Applies the trumpet-plot comparison from Section 15 to the instrumental-resolution-corrected stellar velocity dispersion rather than emission line fluxes. The σ★ errors require only a small multiplicative rescaling to match the observed scatter — a cleaner result than the emission lines, which are also subject to spectrophotometric noise. Relies on `savedf` and the `trumpet_panel` function defined in Section 15, so that section must have been run first. **Requires:** Sections 0, 11, and 15.*

**Output:** `trumpet_plots_disp_avg.pdf` / `.png`

In [ ]:
# ── Figure 19: trumpet_plots_disp_avg ────────────────────────────────────
# Same trumpet structure as Fig 17, applied to the instrumental-resolution-
# corrected stellar velocity dispersion (SC_disp_int). Uses trumpet_panel
# defined in Section 15. Unlike the emission line plots, sigma_* errors
# only need a multiplicative rescaling to match the observed scatter —
# the corrected scale factor is in scale_factors['SC_disp_int'].
scale = 3
fig, ax = plt.subplots(1, 1, figsize=(11, 8.5))
trumpet_panel(ax, savedf, 'SC_disp_int', snr_max=21,
              binning=15*3, dotcolor='C6', pointsz=26*scale)
ax.set_xlabel(r'Signal-to-Noise Ratio$_{\sigma_*}$', fontsize=36)
ax.set_ylabel(r'log ($\sigma_1$ / $\sigma_2$)', fontsize=36)
ax.legend(loc=1, fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'trumpet_plots_disp_avg.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'trumpet_plots_disp_avg.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Sections 15 & 16 cleanup ─────────────────────────────────────────────
# savedf is used by both Sections 15 (Figs 17-18) and 16 (Fig 19), so the
# del is placed here, after both sections are done, rather than at the end
# of either individual section. Re-running any single figure cell won't
# hit a NameError on a second pass.
del savedf

# Section 17 — Figure 20: Mass–σ★ Diagram
### *Do the kinematics follow the mass, and do we agree with everyone else?*
*Plots stellar velocity dispersion against stellar mass for eBOSS galaxies with S/N > 5 σ★ detections, with the median relation overlaid against the SDSS-I and SHELS comparison samples from Zahid et al. Requires the Drake stellar-mass catalog (`HAS_SUMMARY`); skips gracefully if absent. Note that high-EW galaxies (ELGs in particular) show a systematic offset due to Balmer absorption infilling — see Section 22 for the EW-binned version. **Requires:** Sections 0 and 11. Needs `HAS_SUMMARY = True`.*

**Output:** `mass_vs_vdisp_kde.pdf` / `.png`

In [ ]:
if not HAS_SUMMARY:
    print('This section requires data from D. Miller et al. (in prep.), '
          'which will be released alongside the Drake et al. catalog paper. Skipping.')
else:
    # ── Quality cuts for Fig 20 ───────────────────────────────────────────────
    # Require S/N > 5 on the velocity dispersion measurement and exclude the
    # pipeline's hard upper limit (450 km/s) which indicates a failed fit.
    cut_a = df[df['SC_disp'] > df['SC_disp_err'] * 5]
    cut_b = cut_a[cut_a['SC_disp'] > 0.001]
    cut_c = cut_b[cut_b['SC_disp_err'] > 0.001]
    cut   = cut_c[cut_c['SC_disp'] < 449.999].copy()
    del cut_a, cut_b, cut_c

    # Instrumental-resolution correction: subtract SC_CORRECTION in quadrature.
    # Negatives (where correction exceeds measurement) are clipped to 0.
    cut['SC_disp_intrinsic'] = np.sqrt(
        np.clip(cut['SC_disp']**2 - cut['SC_correction']**2, 0, None))

    # Median sigma_* in 0.2-dex mass bins for the white median line.
    masses_bins = np.arange(8.0, 12.6, 0.2)
    masses_plot = (masses_bins[:-1] + masses_bins[1:]) / 2
    averages = [np.nanmedian(cut[(cut['LOGM'] > masses_bins[i]) &
                                  (cut['LOGM'] < masses_bins[i+1])]['SC_disp_intrinsic'])
                for i in range(len(masses_plot))]

    # ── Literature comparison: Zahid+2016 broken power law (SDSS-I and SHELS) ─
    sigb = 10**2.073
    a1, a2 = 0.403, 0.293
    mb_sdss = 10.26
    x1_sdss = np.arange(9.0, mb_sdss, 0.01)
    x2_sdss = np.arange(mb_sdss, 11.6, 0.01)
    y1_sdss = sigb * ((10**x1_sdss) / (10**10.26))**a1
    y2_sdss = sigb * ((10**x2_sdss) / (10**10.26))**a2

    sigb3 = 10**2.24
    a3 = 0.32
    mb3 = 11
    x3_shels = np.arange(9.0, 11.6, 0.01)
    y3_shels = sigb3 * ((10**x3_shels) / (10**mb3))**a3

In [ ]:
if not HAS_SUMMARY:
    print('This section requires data from D. Miller et al. (in prep.), '
          'which will be released alongside the Drake et al. catalog paper. Skipping.')
else:
    if not HAS_SUMMARY:
        print('This section requires data from D. Miller et al. (in prep.), '
              'which will be released alongside the Drake et al. catalog paper. Skipping.')
    else:
        # ── Figure 20: mass_vs_vdisp_kde ────────────────────────────────────
        plot_prettier(150, fontsize=18)
        plt.figure(figsize=[11, 8.5])
        handles = []

        from matplotlib.colors import LogNorm
        plt.hist2d(cut['LOGM'], cut['SC_disp_intrinsic'], bins=50,
                   range=[[8,12.5],[0.001,449.99]], cmin=3,
                   cmap='gray_r', zorder=0, norm=LogNorm())

        _target_classes = [
            ('LRG4', c_lrg4, 'eBOSS LRG'), ('LRG3', c_lrg3, 'BOSS LRG'),
            ('LRGL', c_lrgl, 'BOSS LOWZ LRG'), ('ELG', c_elg, 'ELG'),
            ('TDSP',  c_tdsp, 'TDSS/SPIDERS'), ('QSO', c_qso,  'QSO'),
        ]
        for _idx, (target_class_str, color, label) in enumerate(_target_classes):
            grp = cut[cut['TARGET_CLASS'].str.contains(target_class_str)]
            n_done = sum(len(cut[cut['TARGET_CLASS'].str.contains(c)]) for c,_,_ in _target_classes[:_idx+1])
            print(f'Starting [{n_done:,}/{len(cut):,} = {100*n_done//len(cut)}%] {label}...')
            if len(grp) < 10:
                continue
            sns.kdeplot(x=grp['LOGM'], y=grp['SC_disp_intrinsic'],
                        fill=False, thresh=0.5, levels=4, cut=10, gridsize=KDE_GRIDSIZE,
                        color=color, linewidths=3,
                        clip=((8,12.5),(0.001,449.99)))
            print('Finished.')
            handles.append(mlines.Line2D([], [], color=color, label=label))

        plt.plot(masses_plot, averages, color='w', zorder=100, lw=5,
                 path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()])
        handles.append(mlines.Line2D([], [], label='Median', color='w', lw=5,
            path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()]))

        plt.plot(x1_sdss, y1_sdss, color="xkcd:robin's egg blue", zorder=50, lw=5, ls='--',
                 path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()])
        plt.plot(x2_sdss, y2_sdss, color="xkcd:robin's egg blue", zorder=50, lw=5, ls='--',
                 path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()])
        handles.append(mlines.Line2D([], [], label='SDSS',
            color="xkcd:robin's egg blue", ls='--',
            path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()]))

        plt.plot(x3_shels, y3_shels, color='xkcd:steel grey', zorder=50, lw=5, ls='--',
                 path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()])
        handles.append(mlines.Line2D([], [], label='SHELS (0.6 < z < 0.7)',
            color='xkcd:steel grey', ls='--',
            path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()]))

        plt.xlabel(r'log$\left(\frac{M_*}{M_\odot}\right)$', fontsize=36)
        plt.ylabel('Stellar Continuum \n Velocity Dispersion (km s$^{-1}$)', fontsize=30)
        plt.xlim(8, 12.5); plt.ylim(0.001, 449.99)
        leg = plt.legend(handles=handles, loc=2, fontsize=12)
        leg.set_zorder(100)
        for lh in getattr(leg, 'legend_handles', getattr(leg, 'legendHandles', [])): lh.set_alpha(1)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR,'mass_vs_vdisp_kde.pdf'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(OUTPUT_DIR,'mass_vs_vdisp_kde.png'), dpi=150, bbox_inches='tight')
        plt.show()

# Section 18 — Figure 21: BPT Diagram (Two Redshift Bins)
### *Sorting 1.9 million galaxies by what's lighting them up.*
*Applies S/N > 3 cuts on all four BPT lines and restricts to z < 0.4963 (the redshift where Hα leaves the window), then splits into low- and high-redshift panels using the Law et al. 2021 demarcation lines. The contours by target class tell a consistent story: ELGs and QSO-targeted galaxies dominate the star-forming locus, LRGs spread into the AGN and LIER regions. Uses `df` and `AGN` from Section 11. **Requires:** Sections 0 and 11.*

**Output:** `BPT_kde_two.pdf` / `.png`

In [ ]:
# ── BPT demarcation lines and quality cuts for Fig 21 ────────────────────
# Law et al. 2021 demarcation lines: bptline1 is the Kewley+01-style curve
# separating AGN from composites; bptline2 is the star-forming sequence
# boundary; bptline3 is the LIER separator.
bptline1x = np.linspace(-2.0, -0.1, 100)
bptline1y = 0.438 / (bptline1x + 0.023) + 1.222
bptline2y = np.linspace(-0.65, 0.9, 100)
bptline2x = (-0.39*bptline2y**4 - 0.582*bptline2y**3
             - 0.637*bptline2y**2 - 0.048*bptline2y - 0.119)
bptline3x = np.linspace(-0.24, 0.5, 100)
bptline3y = 0.95 * bptline3x + 0.56

# S/N >= 3 on all four BPT lines. The z < 0.4963 cut prevents systematics
# from the Hα/Hβ tying switch that occurs at that redshift.
c1 = df[df['Hb']       / df['Hb_err']       >= 3]
c2 = c1[c1['Ha']       / c1['Ha_err']       >= 3]
c3 = c2[c2['NII_6585'] / c2['NII_6585_err'] >= 3]
c4 = c3[c3['OIII_5008']/ c3['OIII_5008_err']>= 3]
bpt_df = c4[(c4['z'] > 0.0005) & (c4['z'] < 0.4963)].copy()
del c1, c2, c3, c4

bpt_lo = bpt_df[bpt_df['z'] <= 0.25]
bpt_hi = bpt_df[(bpt_df['z'] > 0.25) & (bpt_df['z'] <= 0.4963)]
print(f'BPT sample  lo: {len(bpt_lo):,}   hi: {len(bpt_hi):,}')

In [ ]:
# ── Figure 21: BPT_kde_two ───────────────────────────────────────────────
# Two-panel BPT diagram split at z = 0.25. The per-class KDE contours show
# where each targeting category sits relative to the Law+2021 demarcation
# lines. thresh=0.5 means contours begin at the outermost 50% of each
# class's density — less crowded than starting from the tails.
plot_prettier(150, fontsize=18)

fig, axes = plt.subplot_mosaic("A;B", figsize=(11, 17), sharex=True)
fig.tight_layout(pad=-1.0)
plt.subplots_adjust(hspace=0.075)

linelabels = [r'$0.0005 < z  < 0.25$', r'$0.25 < z  < 0.4963$']

for ax, bpt_sub, panel in [(axes['A'], bpt_lo, 'A'), (axes['B'], bpt_hi, 'B')]:
    bpt_sub = bpt_sub[(bpt_sub['N2'] > -2.0) & (bpt_sub['N2'] < 0.5) &
                      (bpt_sub['R3'] > -1.0) & (bpt_sub['R3'] < 1.5)]

    ax.plot(bptline1x, bptline1y, c='k')
    ax.plot(bptline2x, bptline2y, c='k')
    ax.plot(bptline3x, bptline3y, c='k')

    handles = []
    _target_classes = [
        ('LRG4', c_lrg4, 'eBOSS LRG',      3,  10),
        ('LRG3', c_lrg3, 'BOSS LRG',       3,   3),
        ('LRGL', c_lrgl, 'BOSS LOWZ LRG',  3,   5),
        ('TDSP',  c_tdsp, 'TDSS/SPIDERS',   3,  20),
        ('QSO',  c_qso,  'QSO',           10,   5),
        ('ELG',  c_elg,  'ELG',            3,  35),
    ]
    for _idx, (target_class_str, color, label, cut_val, zorder) in enumerate(_target_classes):
        grp = bpt_sub[bpt_sub['TARGET_CLASS'].str.contains(target_class_str)]
        n_done = sum(len(bpt_sub[bpt_sub['TARGET_CLASS'].str.contains(c)]) for c,_,_,_,_ in _target_classes[:_idx+1])
        print(f'Starting panel {panel} [{n_done:,}/{len(bpt_sub):,} = {100*n_done//len(bpt_sub)}%] {label}...')
        if len(grp) < 10:
            continue
        sns.kdeplot(x=grp['N2'], y=grp['R3'],
                    fill=False, thresh=0.5, levels=5, cut=cut_val, gridsize=KDE_GRIDSIZE,
                    color=color, linewidths=3, zorder=zorder,
                    clip=((-2.0,0.5),(-1.0,1.5)), ax=ax)
        print('Finished.')
        if panel == 'A':
            handles.append(mlines.Line2D([], [], color=color, label=label))

    ax.hist2d(bpt_sub['N2'], bpt_sub['R3'],
              bins=50, range=[[-2.0,0.5],[-1.0,1.5]], cmin=2, cmap='gray_r',
              zorder=0, norm=LogNorm())

    ax.set_ylim(-1.0, 1.5)
    ax.set_xlim(-2.0, 0.5)
    ax.set_ylabel(r'Log([O III] / H$\beta$)', fontsize=36)

    if panel == 'A':
        leg = ax.legend(handles=handles, loc=3, fontsize=12)
        leg.set_zorder(100)
        for lh in getattr(leg, 'legend_handles', getattr(leg, 'legendHandles', [])):
            lh.set_alpha(1)

axes['B'].set_xlabel(r'Log([N II] / H$\alpha$)', fontsize=36)

for i, (label, ax) in enumerate(axes.items()):
    ax.annotate(
        label+': '+linelabels[i],
        xy=(1, 0), xycoords='axes fraction',
        xytext=(-0.25, +1.25), textcoords='offset fontsize',
        fontsize=20, va='bottom', ha='right', fontfamily='serif',
        bbox=dict(facecolor='none', edgecolor='none', pad=3.0))

# Region labels — AGN, SFGs, LIERs. No 'Comp' label, matching the paper.
for ax in [axes['A'], axes['B']]:
    ax.text(0.3, 1.3, 'AGN', fontsize=24, color='k',
            bbox=dict(facecolor='none', edgecolor='none', alpha=0.0), ha='center')
    ax.text(-1.75, -0.30, 'SFGs', fontsize=24, color='k',
            bbox=dict(facecolor='none', edgecolor='none', alpha=0.0), ha='center')
    ax.text(0.3, -0.65, 'LIERs', fontsize=24, color='k',
            bbox=dict(facecolor='none', edgecolor='none', alpha=0.0), ha='center')

plt.savefig(os.path.join(OUTPUT_DIR,'BPT_kde_two.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'BPT_kde_two.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 18 cleanup ───────────────────────────────────────────────────
del bpt_lo, bpt_hi, bpt_df

# Section 19 — Figure 22: Star-Formation Rate vs Stellar Mass
### *The main sequence, as seen by a survey that wasn't designed to find it.*
*Applies S/N > 5 cuts on Hβ and Hα, S/N > 3 on [O III] and [N II], restricts to BPT star-forming galaxies below z = 0.4963, then computes extinction-corrected Hα SFRs with an aperture correction for the missing fiber light. Overlays the star-forming main sequence from Schreiber et al. at z = 0.1 and z = 1 for context. Requires the Drake stellar-mass catalog (`HAS_SUMMARY`); skips if absent. **Requires:** Sections 0 and 11. Needs `HAS_SUMMARY = True`.*

**Output:** `mass_vs_sfr_kde.pdf` / `.png`

In [ ]:
if not HAS_SUMMARY:
    print('This section requires data from D. Miller et al. (in prep.), '
          'which will be released alongside the Drake et al. catalog paper. Skipping.')
else:
    # ── S/N + BPT cuts for Fig 22 ────────────────────────────────────────────
    # Hβ and Hα require S/N > 5 because they're used in the Balmer-decrement
    # dust correction; errors propagate directly into the SFR. [O III] and
    # [N II] only need S/N > 3 since they're used just for the BPT cut.
    s1 = df[df['Hb']       / df['Hb_err']       > 5]
    s2 = s1[s1['Ha']       / s1['Ha_err']       > 5]
    s3 = s2[s2['OIII_5008']/ s2['OIII_5008_err']> 3]
    s4 = s3[s3['NII_6585'] / s3['NII_6585_err'] > 3]
    s5 = s4[(s4['z'] > 0.0005) & (s4['z'] < 0.4963)]
    del s1, s2, s3, s4

    # BPT SFG selection using the Law+2021 SF boundary.
    sfg_mask = (0.438 / (s5['N2'] + 0.023) + 1.222) > s5['R3']
    sfg_mask &= s5['N2'] < -0.1
    sfg_df   = s5[sfg_mask].copy()
    del s5
    print(f'BPT SFGs for SFR diagram: {len(sfg_df):,}')

    # ── Balmer-decrement dust correction + SFR ────────────────────────────────
    # Case B recombination gives Hα/Hβ = 2.86 for typical HII region conditions.
    # The deviation from 2.86 gives E(B-V), which we use to correct Hα.
    # The corrected Hα flux is then converted to luminosity, corrected for
    # fiber aperture (infiber fraction), and converted to SFR using the
    # Kennicutt+1998 calibration.
    R_theo = 2.86
    R_obs  = sfg_df['Ha'] / sfg_df['Hb']
    sfg_df['ebv_sfr'] = 1.97 * np.log10(np.abs(R_obs / R_theo))

    ext_model = dext()
    sfg_df['Ha_corr'] = (sfg_df['Ha'] /
                         ext_model.extinguish(6562.8 * u.AA, Ebv=sfg_df['ebv_sfr'])) * 1e-17
    sfg_df = sfg_df[np.isfinite(sfg_df['Ha_corr'])].copy()

    distances_sfr    = cosmo.luminosity_distance(sfg_df['z'].values).to(u.cm).value
    sfg_df['Lum_Ha'] = (4 * np.pi * distances_sfr**2 * sfg_df['Ha_corr'] / sfg_df['infiber'])
    sfg_df['LOGSFR'] = np.log10(sfg_df['Lum_Ha']) - 41.27
    del distances_sfr

    # ── Star-forming main sequence (Speagle+2014) ─────────────────────────────
    # ages_sfr: [z~1, z~0.1] ages in Gyr, used to bracket the eBOSS redshift range.
    ages_sfr = [5.903, 12.410]
    Mstars   = 10**np.arange(8.25, 11.75, 0.05)
    a0, a1, a2, a3, a4 = 2.693, -0.186, 10.85, -0.0729, 0.99

In [ ]:
if not HAS_SUMMARY:
    print('This section requires data from D. Miller et al. (in prep.), '
          'which will be released alongside the Drake et al. catalog paper. Skipping.')
else:
    if not HAS_SUMMARY:
        print('This section requires data from D. Miller et al. (in prep.), '
              'which will be released alongside the Drake et al. catalog paper. Skipping.')
    else:
        # ── Figure 22: mass_vs_sfr_kde ────────────────────────────────────────
        plot_prettier(150, fontsize=18)
        plt.figure(figsize=[11, 8.5])
        handles = []

        plt.hist2d(sfg_df['LOGM'], sfg_df['LOGSFR'], bins=50,
                   range=[[8,12.5],[-3,3]], cmin=2, cmap='gray_r', zorder=0, norm=LogNorm())

        _kde_params = {
            'LRG4': dict(clip=((8,12.5),(-3,3)), cut=3, zorder=10),
            'LRG3': dict(clip=((8,12.5),(-3,3)), cut=3, zorder=3),
            'LRGL': dict(clip=((8,12.5),(-3,3)), cut=3, zorder=5),
            'ELG':  dict(clip=((8,12.5),(-3,3)), cut=3, zorder=15),
            'TDSP':  dict(clip=((3,12.5),(-3,3)), cut=3, zorder=20),
            'QSO':  dict(clip=((7.5,12.5),(-3,3)), cut=10, zorder=25),
        }

        _target_classes = [
            ('LRG4',c_lrg4,'eBOSS LRG'), ('LRG3',c_lrg3,'BOSS LRG'),
            ('LRGL',c_lrgl,'BOSS LOWZ LRG'), ('ELG',c_elg,'ELG'),
            ('TDSP', c_tdsp,'TDSS/SPIDERS'), ('QSO',c_qso,'QSO'),
        ]
        for _idx, (target_class_str, color, label) in enumerate(_target_classes):
            grp = sfg_df[sfg_df['TARGET_CLASS'].str.contains(target_class_str)]
            n_done = sum(len(sfg_df[sfg_df['TARGET_CLASS'].str.contains(c)]) for c,_,_ in _target_classes[:_idx+1])
            print(f'Starting [{n_done:,}/{len(sfg_df):,} = {100*n_done//len(sfg_df)}%] {label}...')
            if len(grp) < 10:
                continue
            p = _kde_params[target_class_str]
            sns.kdeplot(x=grp['LOGM'], y=grp['LOGSFR'],
                        fill=False, thresh=0.5, levels=4, cut=p['cut'], gridsize=KDE_GRIDSIZE,
                        color=color, linewidths=3, clip=p['clip'], zorder=p['zorder'])
            print('Finished.')
            handles.append(mlines.Line2D([], [], color=color, label=label))

        # Median line in 0.2-dex mass bins.
        mb = np.arange(8.5, 11.6, 0.2); mp = (mb[:-1]+mb[1:])/2
        avgs = [np.nanmedian(sfg_df[(sfg_df['LOGM']>mb[i])&(sfg_df['LOGM']<mb[i+1])]['LOGSFR'])
                for i in range(len(mp))]
        plt.plot(mp, avgs, color='w', lw=5, zorder=50,
                 path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()])
        handles.append(mlines.Line2D([], [], label='Median', color='w', lw=5,
            path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()]))

        # Speagle+2014 SFMS at two ages, bracketing the eBOSS redshift range.
        _sfms_style = [
            ('xkcd:steel grey',       f'Age {ages_sfr[0]} Gyr'),
            ("xkcd:robin's egg blue", f'Age {ages_sfr[1]} Gyr'),
        ]
        for t, (col, lbl) in zip(ages_sfr, _sfms_style):
            y_ms = a0 + a1*t - np.log10(1 + (Mstars/10**(a2+a3*t))**(-a4))
            plt.plot(np.log10(Mstars), y_ms, color=col, zorder=50, lw=5, ls='--',
                     path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()])
            handles.append(mlines.Line2D([], [], label=lbl, color=col, ls='--', lw=5,
                path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()]))

        leg = plt.legend(handles=handles, loc=4, fontsize=12)
        leg.set_zorder(100)
        for lh in getattr(leg, 'legend_handles', getattr(leg, 'legendHandles', [])): lh.set_alpha(1)
        plt.ylabel(r'Log(SFR) [M$_\odot$ / yr]', fontsize=36)
        plt.xlabel(r'Log$\left( \frac{\mathrm{M}_{*}} {\mathrm{M}_{\odot}} \right)$', fontsize=36)
        plt.xlim(8, 12); plt.ylim(-2.5, 2.5)
        plt.tight_layout()
        plt.savefig(os.path.join(OUTPUT_DIR,'mass_vs_sfr_kde.pdf'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(OUTPUT_DIR,'mass_vs_sfr_kde.png'), dpi=150, bbox_inches='tight')
        plt.show()

In [ ]:
# ── Section 19 cleanup ───────────────────────────────────────────────────
del sfg_df

# Section 20 — Figure 23: Mass–Metallicity Relation
### *The tightest scaling relation in galaxy evolution, now with more galaxies and more caveats.*
*Applies BPT AGN removal and S/N > 3 cuts on [O II], Hβ, [O III], Hα, and [N II], then computes gas-phase metallicities using the Nakajima et al. 2022 R23 calibration with O32 branch disambiguation. Shown in two panels: the full low-z sample (which includes some deblending artifacts at very low redshift) and a cleaner higher-z slice that limits redshift evolution. The LRG vs ELG split is doing real scientific work here — the two populations trace genuinely different MZRs. Reads from `zach_df` built in Section 11; caches the measured metallicities to `mzr_measured.csv` to avoid recomputing. Requires the Drake catalog. **Requires:** Sections 0 and 11. Needs `HAS_SUMMARY = True`.*

**Output:** `MZR_kde_two.pdf` / `.png`

In [ ]:
if not HAS_SUMMARY:
    print('This section requires data from D. Miller et al. (in prep.), '
          'which will be released alongside the Drake et al. catalog paper. Skipping.')
else:
    # ── Nakajima+2022 R23 metallicity calibration helpers ─────────────────────
    # R23 = log10((OII + OIII) / Hβ); metallicity inferred by inverting a
    # polynomial calibration. R23 is double-valued (upper/lower branch), so
    # O32 = log10(OIII/OII) is used to break the degeneracy via get_branch.
    oh_sol   = 8.69
    oh_all   = np.linspace(7.6, 9.0, 10_000)
    x_cal    = oh_all - oh_sol
    r23_cal  = np.poly1d(np.array([0.527,-1.569,-1.652,-0.421])[::-1])(x_cal)
    r23_max  = np.nanmax(r23_cal[np.isfinite(r23_cal)])
    r23_min  = np.nanmin(r23_cal[np.isfinite(r23_cal)])
    r23_turn = 8.07  # 12+log(O/H) at the R23 turnaround point

    def get_branch(o32):
        x_turn   = r23_turn - oh_sol
        o32_split= np.poly1d(np.array([-0.691,-2.944,-1.308])[::-1])(x_turn)
        return 'lower' if o32 > o32_split else 'upper'

    def get_r23_metal(r23, branch):
        """Invert the R23 calibration numerically on the specified branch."""
        if branch == 'lower':
            oh_th = np.linspace(7.6, r23_turn, 10_000)
        else:
            oh_th = np.linspace(r23_turn, 9.0, 10_000)
        x = oh_th - oh_sol
        r23_th = np.poly1d(np.array([0.527,-1.569,-1.652,-0.421])[::-1])(x)
        return oh_th[np.argmin(np.abs(r23_th - r23))]

    # ── Load or compute mzr_measured.csv ─────────────────────────────────────
    # Metallicity computation is slow (~several minutes for the full sample),
    # so results are cached to disk. If the CSV exists from a previous run,
    # load it directly; otherwise compute and save.
    if os.path.exists(PATH_MZR_MEASURED):
        df_mzr = pd.read_csv(PATH_MZR_MEASURED)
        print(f'Loaded mzr_measured.csv  ({len(df_mzr):,} rows)')
    else:
        print('Computing metallicities from mzr data (in memory) — this may take a few minutes.')
        mzr_raw = zach_df.copy()

        lines_mzr = ['OII','OIII','Hbeta','Halpha','NII']
        for ln in lines_mzr:
            mzr_raw = mzr_raw[(mzr_raw[ln] > 0) & (mzr_raw[ln] < 10000)
                              & (mzr_raw[ln] / mzr_raw[f'{ln}_err'] > 3)]

        mzr_raw['R3_bpt'] = np.log10(mzr_raw['OIII']   / mzr_raw['Hbeta'])
        mzr_raw['N2_bpt'] = np.log10(mzr_raw['NII']    / mzr_raw['Halpha'])
        # BPT SFG cut: same Law+2021 boundary as Sections 18 and 19.
        sfg_m  = (0.438 / (mzr_raw['N2_bpt'] + 0.023) + 1.222) > mzr_raw['R3_bpt']
        sfg_m &= mzr_raw['N2_bpt'] < -0.1
        mzr_sfg = mzr_raw[sfg_m].copy()
        mzr_sfg = mzr_sfg[(mzr_sfg['redshift'] > 0.0005) & (mzr_sfg['redshift'] < 0.4963)]

        mzr_sfg['r23'] = np.log10((mzr_sfg['OII'] + mzr_sfg['OIII']) / mzr_sfg['Hbeta'])
        mzr_sfg['o32'] = np.log10(mzr_sfg['OIII'] / mzr_sfg['OII'])
        # Clip to calibration range.
        mzr_sfg = mzr_sfg[(mzr_sfg['r23'] < r23_max) & (mzr_sfg['r23'] > r23_min)].copy()
        mzr_sfg['metal'] = np.nan

        for ii in tqdm(range(len(mzr_sfg)), desc='Metallicities'):
            r23v = mzr_sfg['r23'].iat[ii]
            o32v = mzr_sfg['o32'].iat[ii]
            mzr_sfg.iat[ii, mzr_sfg.columns.get_loc('metal')] = get_r23_metal(r23v, get_branch(o32v))

        mzr_sfg.to_csv(PATH_MZR_MEASURED, index=False)
        df_mzr = mzr_sfg
        del mzr_raw, mzr_sfg
        print(f'Computed and saved mzr_measured.csv  ({len(df_mzr):,} rows)')

In [ ]:
if not HAS_SUMMARY:
    print('This section requires data from D. Miller et al. (in prep.), '
          'which will be released alongside the Drake et al. catalog paper. Skipping.')
else:
    if not HAS_SUMMARY:
        print('This section requires data from D. Miller et al. (in prep.), '
              'which will be released alongside the Drake et al. catalog paper. Skipping.')
    else:
        # ── Figure 23: MZR_kde_two ────────────────────────────────────────────────
        # Two-panel MZR: Panel A shows the full low-z sample; Panel B restricts
        # to a narrower redshift range to limit redshift evolution and avoid the
        # low-z deblending artifacts. Both panels show ELG/QSO and LRG/TDSP
        # median tracks and per-class KDE contours.
        plot_prettier(150, fontsize=18)
        fig, axes = plt.subplot_mosaic("A;B", figsize=(11, 17), sharex=True)
        fig.tight_layout(pad=-1.0)
        plt.subplots_adjust(hspace=0.075)

        linelabels = [r'$0.0005 < z  < 0.4963$', r'$0.3 < z  < 0.4963$']

        def _class_medians(sub, masses, masses_plot):
            """Median metallicity per mass bin for ELG/QSO and LRG/TDSP subsets.
            Bins with fewer than 1% of the subset's galaxies are excluded
            to avoid noisy edge-bin medians dominating the trend line.
            """
            elg_like = sub[sub['TARGET_CLASS'].str.contains('ELG|QSO', na=False)]
            elg_like = elg_like[(elg_like['LOGM']>8)&(elg_like['LOGM']<12.5)&
                                 (elg_like['metal']>8)&(elg_like['metal']<9)]
            lrg_like = sub[sub['TARGET_CLASS'].str.contains('LRG|TDSP', na=False)]
            lrg_like = lrg_like[(lrg_like['LOGM']>8)&(lrg_like['LOGM']<12.5)&
                                 (lrg_like['metal']>8)&(lrg_like['metal']<9)]

            avgs_elg, avgs_lrg = [], []
            for i, mass in enumerate(masses[1:]):
                topdf = elg_like[elg_like['LOGM'] > masses[i]]
                minidf = topdf[topdf['LOGM'] < mass]
                avgs_elg.append(np.nanmedian(minidf['metal']) if len(minidf) >= 0.01*len(elg_like) else np.nan)

                topdf = lrg_like[lrg_like['LOGM'] > masses[i]]
                minidf = topdf[topdf['LOGM'] < mass]
                avgs_lrg.append(np.nanmedian(minidf['metal']) if len(minidf) >= 0.01*len(lrg_like) else np.nan)
            return np.array(avgs_elg), np.array(avgs_lrg)

        # ── Panel A: full z < 0.4963 sample ─────────────────────────────────────
        subA = df_mzr[(df_mzr['redshift'] > 0.0005) & (df_mzr['redshift'] < 0.4963)
                      & (df_mzr['metal'] > 8) & (df_mzr['metal'] < 9)].copy()

        handlesA = []
        _target_classes = [
            ('LRG4', c_lrg4, 'eBOSS LRG',      3,  10),
            ('LRG3', c_lrg3, 'BOSS LRG',       3,   3),
            ('LRGL', c_lrgl, 'BOSS LOWZ LRG',  3,   5),
            ('TDSP',  c_tdsp, 'TDSS/SPIDERS',   3,  20),
            ('QSO',  c_qso,  'QSO',           10,   5),
            ('ELG',  c_elg,  'ELG',            3,  35),
        ]
        for _idx, (target_class_str, color, label, cut_val, zorder) in enumerate(_target_classes):
            grp = subA[subA['TARGET_CLASS'].str.contains(target_class_str)]
            n_done = sum(len(subA[subA['TARGET_CLASS'].str.contains(c)]) for c,_,_,_,_ in _target_classes[:_idx+1])
            print(f'Starting [{n_done:,}/{len(subA):,} = {100*n_done//len(subA)}%] {target_class_str}...')
            if len(grp) < 10:
                continue
            sns.kdeplot(x=grp['LOGM'], y=grp['metal'], ax=axes['A'],
                        fill=False, thresh=0.5, levels=4, cut=cut_val, gridsize=KDE_GRIDSIZE,
                        color=color, linewidths=3, zorder=zorder,
                        clip=((8,12.5),(8,9)))
            print('Finished.')
            handlesA.append(mlines.Line2D([], [], color=color, label=label))

        masses_A = np.arange(8.2, 11.5, 0.2)
        masses_plot_A = np.arange(8.3, 11.5, 0.2)
        avgs_elg_A, avgs_lrg_A = _class_medians(subA, masses_A, masses_plot_A)

        axes['A'].plot(masses_plot_A, avgs_elg_A, color="xkcd:robin's egg blue", zorder=50, lw=5,
                       path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()])
        handlesA.append(mlines.Line2D([], [], label='ELG Median', color="xkcd:robin's egg blue",
            zorder=50, lw=5, path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()]))
        axes['A'].plot(masses_plot_A, avgs_lrg_A, color='xkcd:rust', zorder=50, lw=5,
                       path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()])
        handlesA.append(mlines.Line2D([], [], label='LRG Median', color='xkcd:rust',
            zorder=50, lw=5, path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()]))

        axes['A'].hist2d(subA['LOGM'], subA['metal'], bins=50,
                         range=[[8,12.5],[8,9]], cmin=5, cmap='gray_r', zorder=0, norm=LogNorm())

        legA = axes['A'].legend(handles=handlesA, loc=2, fontsize=12)
        legA.set_zorder(100)
        for lh in getattr(legA, 'legend_handles', getattr(legA, 'legendHandles', [])):
            lh.set_alpha(1)

        # ── Panel B: z = 0.3-0.5 to limit redshift evolution and deblending issues ─
        subB = df_mzr[(df_mzr['redshift'] > 0.3) & (df_mzr['redshift'] < 0.4963)
                      & (df_mzr['metal'] > 8) & (df_mzr['metal'] < 9)].copy()

        masses_B = np.arange(9.2, 11.5, 0.2)
        masses_plot_B = np.arange(9.3, 11.5, 0.2)
        avgs_elg_B, avgs_lrg_B = _class_medians(subB, masses_B, masses_plot_B)

        # Literature MZR fits for comparison: Nakajima+2022 functional form
        # evaluated at the LEGA-C (z~0.7) and SDSS-I (z~0.1) parameters.
        Z0, M0, gamma, beta   = 8.74, 10.13, 0.3, 1.99
        legac_metal = Z0 - (gamma/beta) * np.log10(1 + (10**masses_plot_B / 10**M0)**-beta)

        Z00, M00, gamma0, beta0 = 8.793, 10.02, 0.28, 1.2
        sdss_metal  = Z00 - (gamma0/beta0) * np.log10(1 + (10**masses_plot_B / 10**M00)**-beta0)

        handlesB = []
        axes['B'].plot(masses_plot_B, avgs_elg_B, color="xkcd:robin's egg blue", zorder=50, lw=5,
                       path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()])
        handlesB.append(mlines.Line2D([], [], label='ELG Median', color="xkcd:robin's egg blue",
            zorder=50, lw=5, path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()]))
        axes['B'].plot(masses_plot_B, avgs_lrg_B, color='xkcd:rust', zorder=50, lw=5,
                       path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()])
        handlesB.append(mlines.Line2D([], [], label='LRG Median', color='xkcd:rust',
            zorder=50, lw=5, path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()]))

        axes['B'].plot(masses_plot_B, legac_metal, color='xkcd:steel grey', zorder=50, lw=5, ls='--',
                       path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()])
        handlesB.append(mlines.Line2D([], [], label='LEGA-C ($z = 0.7$)', color='xkcd:steel grey',
            ls='--', zorder=50, lw=5, path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()]))
        axes['B'].plot(masses_plot_B, sdss_metal, color='xkcd:mint', zorder=50, lw=5, ls='--',
                       path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()])
        handlesB.append(mlines.Line2D([], [], label='SDSS ($z \\simeq 0.1$)', color='xkcd:mint',
            ls='--', zorder=50, lw=5, path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()]))

        axes['B'].hist2d(subB['LOGM'], subB['metal'], bins=50,
                         range=[[8,12.5],[8,9]], cmin=5, cmap='gray_r', zorder=0, norm=LogNorm())

        legB = axes['B'].legend(handles=handlesB, loc=2, fontsize=12)
        legB.set_zorder(100)
        for lh in getattr(legB, 'legend_handles', getattr(legB, 'legendHandles', [])):
            lh.set_alpha(1)

        for ax in [axes['A'], axes['B']]:
            ax.set_ylim(8.3, 9)
            ax.set_xlim(8, 12)
            ax.set_ylabel(r'12+Log$\left( \frac{\mathrm{O}}{\mathrm{H}} \right)$', fontsize=36)
        axes['B'].set_xlabel(r'Log$\left( \frac{\mathrm{M}_{*}}{\mathrm{M}_{\odot}} \right)$', fontsize=36)

        for i, (label, ax) in enumerate(axes.items()):
            ax.annotate(
                label+': '+linelabels[i],
                xy=(1, 0), xycoords='axes fraction',
                xytext=(-0.25, +1.25), textcoords='offset fontsize',
                fontsize=18, verticalalignment='bottom', horizontalalignment='right', fontfamily='serif',
                bbox=dict(facecolor='none', edgecolor='none', pad=3.0))

        plt.savefig(os.path.join(OUTPUT_DIR,'MZR_kde_two.pdf'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(OUTPUT_DIR,'MZR_kde_two.png'), dpi=150, bbox_inches='tight')
        plt.show()
        del subA, subB

In [ ]:
if not HAS_SUMMARY:
    print('This section requires data from D. Miller et al. (in prep.), '
          'which will be released alongside the Drake et al. catalog paper. Skipping.')
else:
    # ── Section 20 cleanup ───────────────────────────────────────────────────
    del df_mzr

# Section 21 — Figure 24: Dn4000 vs HδA
### *Stellar population age, the slow way.*
*Applies quality cuts from Spindler et al. (Dn4000 error < 0.03, HδA error < 0.8) and a redshift window of 0.05 < z < 0.8, then plots the two indices against each other with C3K SSP and continuous star-formation model tracks overlaid. Only a small fraction of the full sample passes the S/N threshold for both indices — this is a bright-subsample figure. The separation between post-starburst galaxies near the SSP track and the main population along the continuous model is the point. **Requires:** Sections 0 and 11.*

**Output:** `d4000vsHdelA_mass.pdf` / `.png`

In [ ]:
# ── Quality cuts and SSP/continuous track data for Fig 24 ────────────────
# The Spindler+2018 cuts (Dn4000 error < 0.03; HδA error < 0.8) select
# only the brightest spectra with reliable continuum measurements. The
# redshift window (0.05-0.8) avoids the very lowest-z deblending artifacts
# and the high-z regime where both indices are difficult to measure.
df_d4 = df[
    (df['Dn4000_err']  < 0.03) &
    (df['HDeltaA_err'] < 0.8)  &
    (df['z'] > 0.05) & (df['z'] < 0.8) &
    (df['HDeltaA'] > -10) & (df['Dn4000'] > -10)
].copy()
print(f'Dn4000/HDeltaA quality sample: {len(df_d4):,}')

# ── Load pre-computed SSP and continuous model tracks ────────────────────
# These FITS files contain Dn4000 and HδA values for the C3K SSPs and a
# continuous star-formation model, pre-measured at the relevant ages.
# The index range [20:103] and [66:99] selects the age range used in the
# paper — wider ranges include very young ages where the indices behave
# non-monotonically due to nebular contamination.
ssp_fits  = fits.open(PATH_SSPS_D4HDA)
cont_fits = fits.open(PATH_CONT_D4HDA)
our_fits  = fits.open(PATH_OUR_SPIND)

ssp_dn4000  = ssp_fits[1].data['D4000_N'][20:103]
ssp_hda     = ssp_fits[1].data['LICK_HD_A'][20:103]
cont_dn4000 = cont_fits[1].data['D4000_N'][20:103]
cont_hda    = cont_fits[1].data['LICK_HD_A'][20:103]
# This work's SSPs: the subset used for the scatter plot markers.
us_dn4000   = our_fits[1].data['D4000_N'][66:99]
us_hda      = our_fits[1].data['LICK_HD_A'][66:99]

ssp_fits.close(); cont_fits.close(); our_fits.close()

In [ ]:
# ── Figure 24: d4000vsHdelA_mass ─────────────────────────────────────────
# 2D log-scaled histogram of the quality-cut sample, with SSP and continuous
# model tracks overlaid. The SSP track goes from young (high HδA, low Dn4000)
# to old (low HδA, high Dn4000). The continuous model is a better match to
# the bulk of the data; galaxies near the SSP track are likely post-starburst.
plot_prettier(150, fontsize=18)
fig = plt.figure(figsize=(12, 8))
fig.tight_layout(pad=-1.0)

plt.hist2d(df_d4['Dn4000'], df_d4['HDeltaA'],
           bins=75, range=[[0.9,2.15],[-5.5,12.5]],
           cmin=2, cmap='gray_r', zorder=0, norm=LogNorm())

plt.plot(ssp_dn4000,  ssp_hda,  c='xkcd:dark red',
         label='SSPs', lw=4, zorder=5)
plt.plot(cont_dn4000, cont_hda, c='xkcd:dark yellow',
         label='Continuous Model', lw=4, zorder=6)
plt.scatter(us_dn4000, us_hda, c='xkcd:light gray',
            label="This Work's SSPs", s=64, zorder=10,
            marker='v', edgecolors='xkcd:dark red')

plt.ylim(-5.5, 12.5); plt.xlim(0.9, 2.15)
plt.xlabel(r'D$_{\rm n}$4000', fontsize=36)
plt.ylabel(r'Lick H$\delta_{\rm A}$', fontsize=36)
plt.legend(fontsize=14)
plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_DIR,'d4000vsHdelA_mass.pdf'), dpi=300, bbox_inches='tight')
plt.savefig(os.path.join(OUTPUT_DIR,'d4000vsHdelA_mass.png'), dpi=150, bbox_inches='tight')
plt.show()

In [ ]:
# ── Section 21 cleanup ───────────────────────────────────────────────────
del df_d4, ssp_dn4000, ssp_hda, cont_dn4000, cont_hda, us_dn4000, us_hda

# Section 22 — Appendix Figure A1: Mass–σ★ by Hβ EW Bin
### *Why the ELG kinematics look wrong, and why that's expected.*
*Reproduces the Section 17 mass–σ★ diagram but bins by Hβ EW instead of target class, making explicit the systematic offset in σ★ for high-EW galaxies caused by Balmer absorption infilling. This is the appendix figure that explains the caveat flagged in Section 17. Requires the Drake catalog. **Requires:** Sections 0 and 11. Needs `HAS_SUMMARY = True`.*

**Output:** `mass_vs_vdisp_kde_ew.pdf` / `.png`

In [ ]:
if not HAS_SUMMARY:
    print('This section requires data from D. Miller et al. (in prep.), '
          'which will be released alongside the Drake et al. catalog paper. Skipping.')
else:
    # ── Quality cuts and EW binning for Fig A1 ────────────────────────────────
    # Same sigma_* quality cuts as Section 17 (Fig 20), but now binning by
    # Hβ EW instead of target class. The purpose is to show explicitly that
    # high-EW galaxies (mostly ELGs) have systematically lower apparent σ★
    # due to Balmer absorption infilling.
    c_a = df[df['SC_disp'] > df['SC_disp_err'] * 5]
    c_b = c_a[c_a['SC_disp'] > 0.001]
    c_c = c_b[c_b['SC_disp_err'] > 0.001]
    cut_base = c_c[c_c['SC_disp'] < 449.999].copy()
    del c_a, c_b, c_c

    cut_base['SC_disp_intrinsic'] = np.sqrt(
        np.clip(cut_base['SC_disp']**2 - cut_base['SC_correction']**2, 0, None))

    cut_ew = cut_base[(cut_base['Hb_ew'] > 0) & np.isfinite(cut_base['Hb_ew'])].copy()
    cut_ew = cut_ew.dropna(subset=['LOGM','SC_disp_intrinsic'])

    # Six linear EW bins from 0 to 25 Å, with the last bin open-ended (> 25 Å).
    N_BINS = 6
    EW_MAX = 25

    cut_ew_pos = cut_ew[(cut_ew['Hb_ew'] > 0) & (~cut_ew['Hb_ew'].isna())].copy()
    cut_ew_pos = cut_ew_pos.dropna(subset=['LOGM','SC_disp_intrinsic'])

    ew_edges = np.linspace(0, EW_MAX, N_BINS)
    ew_edges = list(ew_edges) + [np.inf]

    cut_ew_pos['EW_bin'] = np.digitize(cut_ew_pos['Hb_ew'], ew_edges) - 1
    hb_bins = [cut_ew_pos[cut_ew_pos['EW_bin'] == i] for i in range(N_BINS)]

    ew_colors = [
        "magenta", "xkcd:dark red", "xkcd:burnt orange", "green",
        "xkcd:dark yellow", "blue", "xkcd:purple", "xkcd:teal",
        "xkcd:sky blue", "xkcd:forest green"
    ][:N_BINS]

    ew_labels = []
    for i in range(N_BINS):
        lo, hi = ew_edges[i], ew_edges[i+1]
        if np.isinf(hi):
            ew_labels.append(rf"$\mathrm{{EW}}_{{\mathrm{{H}}\beta}} > {lo:.1f}$")
        else:
            ew_labels.append(rf"${lo:.1f} < \mathrm{{EW}}_{{\mathrm{{H}}\beta}} \leq {hi:.1f}$")

In [ ]:
if not HAS_SUMMARY:
    print('This section requires data from D. Miller et al. (in prep.), '
          'which will be released alongside the Drake et al. catalog paper. Skipping.')
else:
    if not HAS_SUMMARY:
        print('This section requires data from D. Miller et al. (in prep.), '
              'which will be released alongside the Drake et al. catalog paper. Skipping.')
    else:
        # ── Figure A1: mass_vs_vdisp_kde_ew ──────────────────────────────────────
        # Same layout as Fig 20, but contours represent EW bins rather than
        # target classes. The median line (white, from cell 79's averages) and
        # SDSS/SHELS comparison lines are the same as Fig 20.
        plot_prettier(150, fontsize=18)
        fig = plt.figure(figsize=(12, 8))
        fig.tight_layout(pad=-1.0)

        handles = []

        for i in range(N_BINS):
            df_bin = hb_bins[i].dropna(subset=['LOGM','SC_disp_intrinsic'])
            if len(df_bin) < 5:
                print(f'{i+1}/{N_BINS} KDE bin skipped (insufficient data: {len(df_bin)} galaxies)')
                continue
            print(f'{i+1}/{N_BINS} KDE bin plotted ({len(df_bin)} galaxies)')
            sns.kdeplot(x=df_bin['LOGM'], y=df_bin['SC_disp_intrinsic'],
                        fill=False, thresh=0.5, levels=4, cut=3, gridsize=KDE_GRIDSIZE,
                        color=ew_colors[i], zorder=10+i,
                        clip=((8,12.5),(0.001,449.99)), linewidths=3)
            handles.append(mlines.Line2D([], [], color=ew_colors[i], label=ew_labels[i]))

        # Reuse the median relation from Section 17 (masses_plot, averages).
        plt.plot(masses_plot, averages, color='w', zorder=100, lw=5,
                 path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()])
        handles.append(mlines.Line2D([], [], label='Median', color='w', lw=5,
            path_effects=[pe.Stroke(linewidth=8, foreground='k'), pe.Normal()]))

        plt.hist2d(cut_ew_pos['LOGM'], cut_ew_pos['SC_disp_intrinsic'], bins=50,
                   range=[[8,12.5],[0.001,449.99]], cmin=3, cmap='gray_r', zorder=0, norm=LogNorm())

        # Reuse SDSS/SHELS comparison lines from Section 17.
        plt.plot(x1_sdss, y1_sdss, color="xkcd:robin's egg blue", lw=5, ls='--',
                 zorder=50, path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()])
        plt.plot(x2_sdss, y2_sdss, color="xkcd:robin's egg blue", lw=5, ls='--',
                 zorder=50, path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()])
        handles.append(mlines.Line2D([], [], label='SDSS', color="xkcd:robin's egg blue", ls='--', lw=5))

        plt.plot(x3_shels, y3_shels, color='xkcd:steel grey', lw=5, ls='--',
                 zorder=50, path_effects=[pe.Stroke(linewidth=8, foreground='w'), pe.Normal()])
        handles.append(mlines.Line2D([], [], label='SHELS (0.6 < z < 0.7)', color='xkcd:steel grey', ls='--', lw=5))

        plt.xlabel(r'log$\left( \frac{\mathrm{M}_{*}}{\mathrm{M}_{\odot}} \right)$', fontsize=36)
        plt.ylabel('Stellar Continuum \n Velocity Dispersion (km s$^{-1}$)', fontsize=30)
        plt.xlim(8, 12.5)
        plt.ylim(0.001, 449.99)

        leg = plt.legend(handles=handles, loc=2, fontsize=12)
        leg.set_zorder(100)
        for lh in getattr(leg, 'legend_handles', getattr(leg, 'legendHandles', [])):
            lh.set_alpha(1)

        plt.savefig(os.path.join(OUTPUT_DIR,'mass_vs_vdisp_kde_ew.pdf'), dpi=300, bbox_inches='tight')
        plt.savefig(os.path.join(OUTPUT_DIR,'mass_vs_vdisp_kde_ew.png'), dpi=150, bbox_inches='tight')
        plt.show()
        del cut_ew_pos, hb_bins, handles

In [ ]:
# ── Section 22 cleanup ───────────────────────────────────────────────────
del cut_ew, cut_base
# df and AGN are no longer needed after this point — Sections 23-25 are
# self-contained or use pre-computed files.
del df, AGN

# Section 23 — Appendix Figures A3 & A4: Survey-Choice Error Analysis
### *Does it matter whether we used the full line list or the reference line we tied to?*
*Trumpet-plot comparisons run on a subsample of spectra near the EW boundary (9 < EWHβ < 11 Å) to verify that neither the high/low-EW line-set split nor the Hα/Hβ tying switch introduces systematic flux offsets. The answer in both cases is no — differences are consistent with noise. Reads four pre-computed DAP output FITS files (`PATH_HIHI`, `PATH_HILO`, `PATH_LOHI`, `PATH_LOLO`); skips gracefully if any are missing. **Self-contained:** needs only Section 0, but requires the four quality-checker FITS files on disk.*

**App. Fig. A3** `trumpet_EW.pdf` — full vs reduced line-set comparison.

**App. Fig. A4** `trumpet_z.pdf` — Hα-tied vs Hβ-tied comparison.

In [ ]:
# ── Load quality-checker FITS files and generate Figs A3 & A4 ────────────
# Each FITS file contains emission-line fluxes for a subsample of spectra
# run under a specific pipeline configuration:
#   HIHI: high-EW line list  + high-z (Hβ-tied) configuration
#   HILO: low-EW  line list  + high-z (Hβ-tied) configuration
#   LOHI: low-EW  line list  + low-z  (Hα-tied) configuration
#   LOLO: low-EW  line list  + low-z  (Hα-tied) — i.e. the same objects as
#         LOHI but run with the Hβ tying prescription instead
#
# Fig A3 (trumpet_EW): compares HIHI vs HILO for the same objects to isolate
#   the effect of using the full vs reduced line list.
# Fig A4 (trumpet_z): compares LOHI vs LOLO for the same objects to isolate
#   the effect of switching the reference (tying) line from Hα to Hβ.
#
# All four files will be available on the eBOSS-DAP GitHub repository.
PATH_HIHI = os.path.join(DATA_DIR, 'Bin_hbew_approx_10_hihi_fits_eml_summary.fits')
PATH_HILO = os.path.join(DATA_DIR, 'Bin_hbew_approx_10_hilo_fits_eml_summary.fits')
PATH_LOHI = os.path.join(DATA_DIR, 'Bin_hbew_approx_10_lohi_fits_eml_summary.fits')
PATH_LOLO = os.path.join(DATA_DIR, 'Bin_hbew_approx_10_lolo_fits_eml_summary.fits')

required = [PATH_HIHI, PATH_HILO, PATH_LOHI, PATH_LOLO]
if all(os.path.exists(p) for p in required):
    t_hihi = fits.getdata(PATH_HIHI, 1)
    t_hilo = fits.getdata(PATH_HILO, 1)
    t_lohi = fits.getdata(PATH_LOHI, 1)
    t_lolo = fits.getdata(PATH_LOLO, 1)

    comp_lines_ew = ['OII_3727','H_beta','OIII_5008','H_alpha','NII_6585','SII_6718']
    comp_labels_ew = ['[O II] 3727+3729 Flux', r'H$\beta$ Flux', '[O III] 5008 Flux',
                      r'H$\alpha$ Flux', '[N II] 6585 Flux', '[S II] 6718+6733 Flux']

    def get_flux_and_err(t, line):
        fl  = t[line + '_FLUX'].byteswap().newbyteorder()
        err = t[line + '_FLUX_ERR'].byteswap().newbyteorder()
        return fl, err

    plot_prettier(100, fontsize=24)

    scale = 3
    plt.rcParams.update({
        'xtick.major.size': 5*scale,
        'xtick.major.width': 1.25*scale,
        'xtick.minor.size': 2.5*scale,
        'xtick.minor.width': 1.25*scale,
        'ytick.direction': 'in',
        'ytick.major.size': 5*scale,
        'ytick.major.width': 1.25*scale,
        'ytick.minor.size': 2.5*scale,
        'ytick.minor.width': 1.25*scale
    })

    binning = 15 * 3
    pointsize = 26 * scale
    dotcolor = 'C6'

    x = np.linspace(0, 21, binning)
    y = np.sqrt(2) / x
    x2 = np.linspace(0, 21, binning)

    def compute_stats(snr, ratio):
        """Bin log(ratio) vs S/N and return per-bin 16th/84th percentiles."""
        stats_df = pd.DataFrame({"snr": snr, "ratio": ratio}).replace([np.inf, -np.inf], np.nan).dropna()

        snrs, top, bot = [], [], []
        for i, ix in enumerate(x2[1:]):
            dfless  = stats_df[stats_df["snr"] < ix]
            dfslice = dfless[dfless["snr"] > x2[i]]

            snrs.append((ix + x2[i]) / 2)
            if len(dfslice) > 0:
                vals = np.log(dfslice["ratio"])
                top.append(np.nanpercentile(vals, 84.0))
                bot.append(np.nanpercentile(vals, 16.0))
            else:
                top.append(np.nan)
                bot.append(np.nan)

        return snrs, top, bot

    def draw_panel(ax, snr_all, ratio_all):
        """Draw individual points and 1-sigma theoretical line on ax."""
        ax.scatter(snr_all, np.log(ratio_all), c='k', s=pointsize/2, rasterized=True)

        ax.plot(x,  y,  c='C1', lw=6)
        ax.plot(x, -y,  c='C1', lw=6)

        ax.set_xlim(0, 21)
        ax.set_ylim(-0.25, 0.25)
        ax.hlines(0, 0, 50, color='C1')

    # ── Figure A3: trumpet_EW — full vs reduced line list ─────────────────────
    # y-axis: log(F_LowEW / F_HighEW) — the ratio of the same object's flux
    # measured with the reduced vs full line list, using Hβ tying throughout.
    fig1, axes1 = plt.subplot_mosaic("ABC;DEF", figsize=(11*scale, 8.5*scale*2/3))
    fig1.tight_layout(pad=-1.0)

    for ax_key, line in zip(["A","B","C","D","E","F"], comp_lines_ew):
        hiEW_flux, hiEW_err = get_flux_and_err(t_hihi, line)
        loEW_flux, loEW_err = get_flux_and_err(t_hilo, line)

        snr_all   = hiEW_flux / hiEW_err
        ratio_all = loEW_flux / hiEW_flux

        draw_panel(axes1[ax_key], snr_all, ratio_all)

    fig1.supxlabel('Signal-to-noise ratio', y=-0.07, fontsize=48)
    fig1.supylabel(r'$\log\!\left(~\mathrm{F}_{\mathrm{Low\,EW}} ~/~ \mathrm{F}_{\mathrm{High\,EW}}\right)$',
                   x=-0.05, fontsize=48)

    for i, (label, ax) in enumerate(axes1.items()):
        ax.annotate(
            label + ': ' + comp_labels_ew[i],
            xy=(1, 0), xycoords='axes fraction',
            xytext=(-0.25, +1.25), textcoords='offset fontsize',
            fontsize='large', verticalalignment='bottom',
            horizontalalignment='right', fontfamily='serif',
            bbox=dict(facecolor='none', edgecolor='none', pad=3.0)
        )

    for key, ax in axes1.items():
        if key not in ["D","E","F"]:
            ax.set_xticklabels([])
        if key not in ["A","D"]:
            ax.set_yticklabels([])

    plt.subplots_adjust(wspace=0.03, hspace=0.05)
    fig1.patch.set_rasterized(True)
    plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_EW.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_EW.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()

    # ── Figure A4: trumpet_z — Hα-tied vs Hβ-tied ────────────────────────────
    # y-axis: log(F_Lowz / F_Highz) — same objects measured with low-z (Hα)
    # vs high-z (Hβ) tying prescription, using the reduced line list.
    fig2, axes2 = plt.subplot_mosaic("ABC;DEF", figsize=(11*scale, 8.5*scale*2/3))
    fig2.tight_layout(pad=-1.0)

    for ax_key, line in zip(["A","B","C","D","E","F"], comp_lines_ew):
        hiZ_flux, hiZ_err = get_flux_and_err(t_lohi, line)
        loZ_flux, loZ_err = get_flux_and_err(t_lolo, line)

        snr_all   = hiZ_flux / hiZ_err
        ratio_all = loZ_flux / hiZ_flux

        draw_panel(axes2[ax_key], snr_all, ratio_all)

    fig2.supxlabel('Signal-to-noise ratio', y=-0.07, fontsize=48)
    fig2.supylabel(r'$\log\!\left(~\mathrm{F}_{\mathrm{Low\,z}} ~/~ \mathrm{F}_{\mathrm{High\,z}}\right)$',
                   x=-0.05, fontsize=48)

    for i, (label, ax) in enumerate(axes2.items()):
        ax.annotate(
            label + ': ' + comp_labels_ew[i],
            xy=(1, 0), xycoords='axes fraction',
            xytext=(-0.25, +1.25), textcoords='offset fontsize',
            fontsize='large', verticalalignment='bottom',
            horizontalalignment='right', fontfamily='serif',
            bbox=dict(facecolor='none', edgecolor='none', pad=3.0)
        )

    for key, ax in axes2.items():
        if key not in ["D","E","F"]:
            ax.set_xticklabels([])
        if key not in ["A","D"]:
            ax.set_yticklabels([])

    plt.subplots_adjust(wspace=0.03, hspace=0.05)
    fig2.patch.set_rasterized(True)
    plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_z.pdf'), bbox_inches='tight')
    plt.savefig(os.path.join(OUTPUT_DIR, 'trumpet_z.png'), dpi=150, bbox_inches='tight')
    plt.show()
    plt.close()
else:
    missing = [os.path.basename(p) for p in required if not os.path.exists(p)]
    print('WARNING: missing quality-checker FITS files:')
    for m in missing: print(f'  {m}')
    print(f'Download all four to: {DATA_DIR}')

In [ ]:
# ── Per-galaxy percent-difference report for Figs A3 & A4 ─────────────────
# Quantifies the systematic offset and scatter between the two pipeline
# configurations for each line. The mean percent difference measures bias;
# the sigma-ratio (normalised by measurement error) checks whether any
# offset is consistent with noise. These numbers go into Tables A1 and A2.
if all(os.path.exists(p) for p in required):
    report_rows = []

    for comparison_name, (t1, t2, label1, label2) in [
        ('EW (High-EW vs Low-EW, high-z)', (t_hihi, t_hilo, 'High-EW', 'Low-EW')),
        ('z (H\u03b1-tied vs H\u03b2-tied, low-EW)', (t_lohi, t_lolo, 'H\u03b1-tied', 'H\u03b2-tied')),
    ]:
        for line, ll in zip(comp_lines_ew, comp_labels_ew):
            flux1, err1 = get_flux_and_err(t1, line)
            flux2, err2 = get_flux_and_err(t2, line)

            valid = (np.isfinite(flux1) & np.isfinite(flux2) &
                     np.isfinite(err1)  & np.isfinite(err2) &
                     (err1 > 0) & (err2 > 0) &
                     ((flux1 + flux2) != 0))

            f1, f2, e1, e2 = flux1[valid], flux2[valid], err1[valid], err2[valid]

            # Symmetric percent difference: (f1-f2)/mean(f1,f2) * 100.
            pct_diff = 100.0 * (f1 - f2) / ((f1 + f2) / 2.0)
            err_diff = 100.0 * np.sqrt(e1**2 + e2**2) / np.abs((f1 + f2) / 2.0)
            sigma_ratio = pct_diff / err_diff

            report_rows.append({
                'comparison': comparison_name,
                'line': ll.replace('\n', ' '),
                'n_gal': valid.sum(),
                'mean_pct_diff': np.nanmean(pct_diff),
                'std_pct_diff': np.nanstd(pct_diff),
                'mean_sigma_ratio': np.nanmean(sigma_ratio),
            })

    report_df = pd.DataFrame(report_rows)
    print(report_df.to_string(index=False, float_format=lambda x: f'{x:.4f}'))

    table_note = (
        "\nTable note:\n"
        "  comparison      -- which two pipeline configurations are being compared\n"
        "  line            -- emission line or line ratio\n"
        "  n_gal           -- number of galaxies in the comparison\n"
        "  mean_pct_diff   -- average percent difference (f1-f2)/((f1+f2)/2)*100 per\n"
        "                     galaxy; far from 0 indicates a SYSTEMATIC offset\n"
        "  std_pct_diff    -- scatter in that per-galaxy percent difference (RANDOM\n"
        "                     variation between the two configurations)\n"
        "  mean_sigma_ratio-- mean_pct_diff normalized by the propagated measurement\n"
        "                     error; near 0 indicates the offset is consistent with\n"
        "                     measurement noise alone\n"
    )
    print(table_note)

    report_df.to_csv(os.path.join(OUTPUT_DIR, 'survey_choice_pct_diff_report.csv'), index=False)
    with open(os.path.join(OUTPUT_DIR, 'survey_choice_pct_diff_report_note.txt'), 'w') as f:
        f.write(table_note)
else:
    print('Skipping percent-difference report: quality-checker FITS files not found.')

In [ ]:
# ── Section 23 cleanup ───────────────────────────────────────────────────
if all(os.path.exists(p) for p in required):
    del t_hihi, t_hilo, t_lohi, t_lolo

# Section 24 — Figures 12, A1, A2: Spectrum Fit Plots (External Tool)
### *Three representative fits, generated elsewhere.*
*These multi-panel fit plots are produced by the eBOSS-DAP spectrum plotter, not by this notebook. The cell just prints the relevant spectrum tags as a reminder. **Self-contained:** run any time.*

**See:** https://github.com/owenmatthewsa/ebossdap/

| Figure | Tag | Output file |
|--------|-----|-------------|
| 12 (AGN) | `4037-55631-0920` | `NeV_Combined_Plots.pdf` |
| A1 (SFG) | `7386-56769-0560` | `SFG_Combined_Plots.pdf` |
| A2 (Red Dead) | `6138-56598-0432` | `RedDead_Combined_Plots.pdf` |

In [ ]:
# ── Spectrum fit plot tags (external tool) ────────────────────────────────
# These figures are generated by the eBOSS-DAP spectrum plotter, not here.
# Run the plotter against the tags below and move the output PDFs to the
# figs/ directory. See: https://github.com/owenmatthewsa/ebossdap/
print('Spectrum fit figures (Figs 12, A1, A2) are generated externally.')
print('Use: https://github.com/owenmatthewsa/ebossdap/')
print()
print('Spectrum tags:')
print('  AGN (Fig 12):        4037-55631-0920')
print('  SFG (App Fig A1):    7386-56769-0560')
print('  Red Dead (App A2):   6138-56598-0432')

# Summary — All Figures Produced

| Fig | File | Section |
|-----|------|---------|
| 1 | `sample_makeup_unified.pdf` | 3 |
| 2 | `sample_SNR.pdf` | 4 |
| 3 | `HB_EW_vs_z_kde.pdf` | 5 |
| 4 | `fluxdownfiberratio_kde.pdf` | 6 |
| 5 | `pair_hist_all_gauss.pdf` | 7 |
| 6 | `bowtie_plot_all.pdf` | 7 |
| 7 | `abs_cal_plot_5400_colorcut.pdf` | 8 |
| 8 | `cutout.png` *(image, not generated)* | — |
| 9 | `cutoff_spec.pdf` | 9 |
| 10 | `ssps_plots.pdf` | 10 |
| 11 | `stackfitmew1.pdf` | 11 |
| 12 | `NeV_Combined_Plots.pdf` *(external)* | 24 |
| 13 | `eml_hists.pdf` | 12 |
| 14 | `spind_hists.pdf` | 12 |
| 15 | `EW_comps.pdf` | 13 |
| 16 | `sdss_comp.pdf` | 14 |
| 17 | `trumpet_plots_strong_avg.pdf` | 15 |
| 18 | `trumpet_plots_weak_avg.pdf` | 15 |
| 19 | `trumpet_plots_disp_avg.pdf` | 16 |
| 20 | `mass_vs_vdisp_kde.pdf` | 17 |
| 21 | `BPT_kde_two.pdf` | 18 |
| 22 | `mass_vs_sfr_kde.pdf` | 19 |
| 23 | `MZR_kde_two.pdf` | 20 |
| 24 | `d4000vsHdelA_mass.pdf` | 21 |
| A1 | `SFG_Combined_Plots.pdf` *(external)* | 24 |
| A2 | `RedDead_Combined_Plots.pdf` *(external)* | 24 |
| A3 | `trumpet_EW.pdf` | 23 |
| A4 | `trumpet_z.pdf` | 23 |
| A5 | `mass_vs_vdisp_kde_ew.pdf` | 22 |


# Section 25 — Static Paper Tables
### *The tables that don't come from data.*
*Writes the catalog column description tables (emission lines, spectral indices, AUXDATA) and the recommended usage guidelines table to `OUTPUT_DIR` as LaTeX-formatted `.txt` files. Nothing in memory is required — these are hardcoded. **Self-contained:** needs only Section 0.*

In [ ]:
# ── Paper Tables 2-4: catalog column descriptions ────────────────────────
# These tables are written to text files and copy-pasted into the LaTeX
# source. Nothing from memory is required.

# ── Paper Table 2: Emission Line Catalog ─────────────────────────────────
eml_rows = [
    ('PMF_STRING',        '-',                    'STRING', 'Combined Plate-MJD-Fiber'),
    ('PLATE',             '-',                    'STRING', 'The Plate'),
    ('MJD',               '-',                    'STRING', 'Modified Julian Date of observation'),
    ('FIBER',             '-',                    'STRING', 'The Fiber'),
    ('Z',                 '-',                    'DOUBLE', 'The redshift'),
    ('SC_VELOCITY',       'km/s',                 'DOUBLE', 'Stellar continuum velocity'),
    ('SC_VELOCITY_ERR',   'km/s',                 'DOUBLE', 'Error in stellar continuum velocity'),
    ('SC_DISPERSION',     'km/s',                 'DOUBLE', 'Stellar continuum velocity dispersion'),
    ('SC_DISPERSION_ERR', 'km/s',                 'DOUBLE', 'Error in dispersion'),
    ('SC_CORRECTION',     'km/s',                 'DOUBLE', 'Stellar continuum dispersion correction'),
    ('SNR',               '-',                    'DOUBLE', 'Median S/N of spectrum'),
    ('LINEID_FLUX',       '1e-17 erg/s/cm2',      'DOUBLE', 'Flux of line LINEID'),
    ('LINEID_FLUX_ERR',   '1e-17 erg/s/cm2',      'DOUBLE', 'Error in flux of line LINEID'),
    ('LINEID_EW',         'Angstrom',              'DOUBLE', 'Restframe Gaussian EW of line LINEID'),
    ('LINEID_EW_ERR',     'Angstrom',              'DOUBLE', 'Error in Gaussian EW'),
    ('LINEID_INT_EW',     'Angstrom',              'DOUBLE', 'Restframe integrated EW'),
    ('LINEID_INT_EW_ERR', 'Angstrom',              'DOUBLE', 'Error in integrated EW'),
    ('LINEID_SIGMA',      'km/s',                  'DOUBLE', 'Sigma of line LINEID'),
    ('LINEID_SIGMA_ERR',  'km/s',                  'DOUBLE', 'Error in sigma'),
]
eml_df = pd.DataFrame(eml_rows, columns=['Column Name','Unit','Format','Description'])
eml_df.to_csv(os.path.join(OUTPUT_DIR,'table_eml_catalog_descriptions.txt'), index=False)
print('Paper Table 2: table_eml_catalog_descriptions.txt')
print(eml_df.to_string(index=False))

# ── Paper Table 3: Spectral Index Catalog ────────────────────────────────
spind_rows = [
    ('PMF_STRING', 'STRING', 'Combined Plate-MJD-Fiber'),
    ('PLATE',      'STRING', 'The Plate'),
    ('MJD',        'STRING', 'Modified Julian Date'),
    ('FIBER',      'STRING', 'The Fiber'),
    ('Z',          'DOUBLE', 'The redshift'),
    ('SNR',        'DOUBLE', 'Median S/N of spectrum'),
    ('SPIND',      'DOUBLE', 'Spectral index value (see MaNGA-DAP Table 4)'),
    ('SPIND_ERR',  'DOUBLE', 'Error in spectral index'),
]
spind_df = pd.DataFrame(spind_rows, columns=['Column Name','Format','Description'])
spind_df.to_csv(os.path.join(OUTPUT_DIR,'table_spind_catalog_descriptions.txt'), index=False)
print('Paper Table 3: table_spind_catalog_descriptions.txt')
print(spind_df.to_string(index=False))

# ── Paper Table 4: AUXDATA Catalog ───────────────────────────────────────
auxdata_rows = [
    ('PMF_STRING',              'STRING',  'Template Weight Columns',  'Combined Plate-MJD-Fiber'),
    ('PLATE',                   'INT',     'Template Weight Columns',  'The Plate'),
    ('MJD',                     'INT',     'Template Weight Columns',  'Modified Julian Date'),
    ('FIBER',                   'INT',     'Template Weight Columns',  'The Fiber'),
    ('E(B-V)',                  'DOUBLE',  'Template Weight Columns',  'Milky Way E(B-V) at spectrum position (input to eBOSS-DAP)'),
    ('RCHI2',                   'DOUBLE',  'Template Weight Columns',  'Reduced chi^2 of best-fit stellar continuum model'),
    ('FIBERMASKED',             'BOOLEAN', 'Template Weight Columns',  'Full fiber mask flag (see masking section)'),
    ('AgeXX_XXXXMetY_Y',       'DOUBLE',  'Template Weight Columns',  'Weight for SSP template age XX.XXXX Gyr, metallicity Y.Y Zsun'),
    ('AgeC_XXXXMetY_Y+nebcont','DOUBLE',  'Template Weight Columns',  'Weight for continuous template with nebular continuum'),
    ('multicoeff_N',            'DOUBLE',  'Template Weight Columns',  'Nth-order multiplicative polynomial coefficient'),
    ('SPECPRIMARY',             'BYTE',    'spAll',  'Flag set to 1 for the primary observation among repeat/duplicate spectra; use to remove duplicates'),
    ('SN_MEDIAN_G',             'FLOAT',   'spAll',  'Median S/N per pixel in g band'),
    ('SN_MEDIAN_R',             'FLOAT',   'spAll',  'Median S/N per pixel in r band'),
    ('SN_MEDIAN_I',             'FLOAT',   'spAll',  'Median S/N per pixel in i band'),
    ('SN_MEDIAN_ALL',           'FLOAT',   'spAll',  'Median S/N per pixel across all bands'),
    ('MODELFLUX_G',             'FLOAT',   'spAll',  'Best-fit model flux in g band'),
    ('MODELFLUX_R',             'FLOAT',   'spAll',  'Best-fit model flux in r band'),
    ('MODELFLUX_I',             'FLOAT',   'spAll',  'Best-fit model flux in i band'),
    ('MODELFLUX_IVAR_G',        'FLOAT',   'spAll',  'Inverse variance of MODELFLUX_G'),
    ('MODELFLUX_IVAR_R',        'FLOAT',   'spAll',  'Inverse variance of MODELFLUX_R'),
    ('MODELFLUX_IVAR_I',        'FLOAT',   'spAll',  'Inverse variance of MODELFLUX_I'),
    ('SPECTROFLUX_G',           'FLOAT',   'spAll',  'Spectroscopic flux in g band'),
    ('SPECTROFLUX_R',           'FLOAT',   'spAll',  'Spectroscopic flux in r band'),
    ('SPECTROFLUX_I',           'FLOAT',   'spAll',  'Spectroscopic flux in i band'),
    ('PETROTH90_I',             'FLOAT',   'spAll',  'Petrosian 90% light radius in i band (arcsec)'),
    ('PETROTH90ERR_I',          'FLOAT',   'spAll',  'Uncertainty on PETROTH90_I (arcsec)'),
    ('LAMBDA_EFF',              'FLOAT',   'spAll',  'Effective wavelength of fiber offset correction (Angstrom)'),
    ('RA',                      'DOUBLE',  'spAll',  'Right Ascension (J2000, degrees)'),
    ('DEC',                     'DOUBLE',  'spAll',  'Declination (J2000, degrees)'),
    ('SPZ_OII_3727_FLUX',       'FLOAT',   'spZline', '[O II] 3727 flux (1e-17 erg/s/cm2)'),
    ('SPZ_H_beta_FLUX',         'FLOAT',   'spZline', 'H-beta flux (1e-17 erg/s/cm2)'),
    ('SPZ_OIII_5007_FLUX',      'FLOAT',   'spZline', '[O III] 5007 flux (1e-17 erg/s/cm2)'),
    ('SPZ_H_alpha_FLUX',        'FLOAT',   'spZline', 'H-alpha flux (1e-17 erg/s/cm2)'),
    ('SPZ_NII_6584_FLUX',       'FLOAT',   'spZline', '[N II] 6584 flux (1e-17 erg/s/cm2)'),
    ('SPZ_SII_6717_FLUX',       'FLOAT',   'spZline', '[S II] 6717 flux (1e-17 erg/s/cm2)'),
    ('SPZ_SII_6731_FLUX',       'FLOAT',   'spZline', '[S II] 6731 flux (1e-17 erg/s/cm2)'),
    ('INFIBER',                 'FLOAT',   'targettext', 'Fraction of galaxy light within 2-arcsec fiber aperture (i band)'),
    ('INFIBER_SN',              'FLOAT',   'targettext', 'Signal to noise of INFIBER'),
    ('TARGET_TEXT',             'STRING',  'targettext', 'Full SDSS target type string'),
    ('TARGET_CLASS',            'STRING',  'targettext', 'Target class (QSO, LRG3, LRG4, LRGL, ELG, TDSP, ANCI, MISC)'),
]
auxdata_df = pd.DataFrame(auxdata_rows, columns=['Column Name','Format','Source','Description'])
auxdata_df.to_csv(os.path.join(OUTPUT_DIR,'table_auxdata_catalog_descriptions.txt'), index=False)
print('Paper Table 4: table_auxdata_catalog_descriptions.txt')
print(auxdata_df.to_string(index=False))

In [ ]:
# ── Paper Table 5: Emission Line List ────────────────────────────────────
# Parsed directly from the .par files in eBOSSDAP_DIR. The line list table
# in the paper is generated from these files rather than maintained by hand,
# so it always reflects the actual pipeline configuration. Lines new to the
# eBOSS-DAP (not in MaNGA) are marked with *, lines not in the SDSS spZline
# catalog with †.

import re as _re
import numpy as _np

EBOSSDAP_DIR = os.path.join(WORKING_DIR, 'eBOSSDAP')
ELP_HEW_HZ   = os.path.join(EBOSSDAP_DIR, 'elpedr2_hew_hz.par')
ELB_HEW      = os.path.join(EBOSSDAP_DIR, 'elbedr2_hew.par')
ELP_LEW_HZ   = os.path.join(EBOSSDAP_DIR, 'elpedr2_lew_hz.par')

def _parse_elp(path):
    lines = []
    with open(path, 'rb') as f:
        for raw in f.read().decode('utf-8', errors='replace').split('\n'):
            raw = raw.strip()
            if not raw.startswith('DAPEML'): continue
            parts = raw.split()
            idx = int(parts[1]); name = parts[2]; restwave = float(parts[3])
            m = _re.findall(r'\{([^}]+)\}', raw)
            def _pt(s, _parts=parts):
                p = s.split()
                return (None, None) if p[0] == 'None' else (int(p[0]), p[1])
            lines.append({'idx': idx, 'name': name, 'restwave': restwave,
                          'tf': _pt(m[0]), 'tv': _pt(m[1]), 'ts': _pt(m[2])})
    return lines

def _parse_elb(path):
    entries = []
    with open(path, 'rb') as f:
        for raw in f.read().decode('utf-8', errors='replace').split('\n'):
            raw = raw.strip()
            if not raw.startswith('DAPELB'): continue
            lam = float(raw.split()[3])
            m = _re.findall(r'\{([^}]+)\}', raw)
            prim = m[0].split()
            entries.append({'lam': lam, 'lo': float(prim[0]), 'hi': float(prim[1])})
    return entries

elp     = _parse_elp(ELP_HEW_HZ)
elb     = _parse_elb(ELB_HEW)
lew_set = {e['name'] for e in _parse_elp(ELP_LEW_HZ)}

idx_to_elp = {e['idx']: e for e in elp}
elb_lams   = _np.array([e['lam'] for e in elb])

_OLD_MANGA = {
    'OII_3727','OII_3729','H12','H11','H_theta','H_eta','NeIII_3870',
    'HeI_3890','H_zeta','NeIII_3969','H_epsilon','H_delta','H_gamma',
    'HeII_4687','H_beta','OIII_4960','OIII_5008','NI_5199','NI_5202',
    'HeI_5877','OI_6302','OI_6366','NII_6550','H_alpha','NII_6585',
    'SII_6718','SII_6733','HeI_7068','ArIII_7138','ArIII_7753',
    'P_eta','SIII_9071','P_zeta','SIII_9533','P_epsilon',
}
_OLD_SDSS = {
    'CIII_1909','MGII_2796','OII_3727','OII_3729','NeIII_3870','NeIII_3969',
    'H_epsilon','H_delta','H_gamma','OIII_4364','HeII_4687','H_beta',
    'OIII_4960','OIII_5008','HeI_5877','OI_6302','SIII_6314','OI_6366',
    'NII_6550','H_alpha','NII_6585','SII_6718','SII_6733','ArIII_7138',
}

# LaTeX display names for each line.
_LN = {
    'CIII_1906':      r'[C\,{\sc iii}] 1906',
    'CIII_1909':      r'[C\,{\sc iii}] 1909',
    'CII_2325C':      r'[C\,{\sc ii}] 2325C',
    'CII_2325D':      r'[C\,{\sc ii}] 2325D',
    'CII_2328':       r'[C\,{\sc ii}] 2328',
    'FEIISTAR_2365':  r'[Fe\,{\sc ii}*] 2365',
    'FEIISTAR_2396':  r'[Fe\,{\sc ii}*] 2396',
    'NEIV_2423':      r'[Ne\,{\sc iv}] 2423',
    'NEIV_2425':      r'[Ne\,{\sc iv}] 2425',
    'OII_2470':       r'[O\,{\sc ii}] 2470',
    'FEIISTAR_2613':  r'[Fe\,{\sc ii}*] 2613',
    'FEIISTAR_2626':  r'[Fe\,{\sc ii}*] 2626',
    'FEIISTAR_2632':  r'[Fe\,{\sc ii}*] 2632',
    'HeII_2734':      r'He\,{\sc ii} 2734',
    'MgV_2784':       r'[Mg\,{\sc v}] 2784',
    'MGII_2796':      r'[Mg\,{\sc ii}] 2796',
    'MGII_2803':      r'[Mg\,{\sc ii}] 2803',
    'FeIV_2830':      r'[Fe\,{\sc iv}] 2830',
    'FeIV_2836':      r'[Fe\,{\sc iv}] 2836',
    'HeI_2945':       r'He\,{\sc i} 2945',
    'OIII_3134':      r'[O\,{\sc iii}] 3134',
    'HeI_3189':       r'He\,{\sc i} 3189',
    'HeII_3204':      r'He\,{\sc ii} 3204',
    'NeV_3347':       r'[Ne\,{\sc v}] 3347',
    'NeV_3427':       r'[Ne\,{\sc v}] 3427',
    'FeVII_3586':     r'[Fe\,{\sc vii}] 3586',
    'OII_3727':       r'[O\,{\sc ii}] 3727',
    'OII_3729':       r'[O\,{\sc ii}] 3729',
    'H12':            'H12',
    'FeVII_3760':     r'[Fe\,{\sc vii}] 3760',
    'H11':            'H11',
    'H_theta':        r'H$\theta$',
    'H_eta':          r'H$\eta$',
    'NeIII_3870':     r'[Ne\,{\sc iii}] 3870',
    'HeI_3890':       r'He\,{\sc i} 3890',
    'H_zeta':         r'H$\zeta$',
    'NeIII_3969':     r'[Ne\,{\sc iii}] 3969',
    'H_epsilon':      r'H$\epsilon$',
    'HeI_4027':       r'He\,{\sc i} 4027',
    'SII_4070':       r'[S\,{\sc ii}] 4070',
    'H_delta':        r'H$\delta$',
    'H_gamma':        r'H$\gamma$',
    'OIII_4364':      r'[O\,{\sc iii}] 4364',
    'HeI_4473':       r'He\,{\sc i} 4473',
    'FeIII_4659':     r'[Fe\,{\sc iii}] 4659',
    'HeII_4687':      r'He\,{\sc ii} 4687',
    'Ar_IV+HeI_4713': r'[Ar\,{\sc iv}] + He\,{\sc i} 4713',
    'FeIII_4742':     r'[Fe\,{\sc iii}] 4742',
    'H_beta':         r'H$\beta$',
    'OIII_4960':      r'[O\,{\sc iii}] 4960',
    'HeI_4924':       r'He\,{\sc i} 4924',
    'FeIII_4988':     r'[Fe\,{\sc iii}] 4988',
    'OIII_5008':      r'[O\,{\sc iii}] 5008',
    'FeII_5018':      r'[Fe\,{\sc ii}] 5018',
    'NI_5199':        r'[N\,{\sc i}] 5199',
    'NI_5202':        r'[N\,{\sc i}] 5202',
    'FeVII_5723':     r'[Fe\,{\sc vii}] 5723',
    'NII_5757':       r'[N\,{\sc ii}] 5757',
    'HeI_5877':       r'He\,{\sc i} 5877',
    'FeVII_6086':     r'[Fe\,{\sc vii}] 6086',
    'OI_6302':        r'[O\,{\sc i}] 6302',
    'SIII_6314':      r'[S\,{\sc iii}] 6314',
    'OI_6366':        r'[O\,{\sc i}] 6366',
    'FeX_6374':       r'[Fe\,{\sc x}] 6374',
    'NII_6550':       r'[N\,{\sc ii}] 6550',
    'H_alpha':        r'H$\alpha$',
    'NII_6585':       r'[N\,{\sc ii}] 6585',
    'HeI_6680':       r'He\,{\sc i} 6680',
    'SII_6718':       r'[S\,{\sc ii}] 6718',
    'SII_6733':       r'[S\,{\sc ii}] 6733',
    'HeI_7068':       r'He\,{\sc i} 7068',
    'ArIII_7138':     r'[Ar\,{\sc iii}] 7138',
    'OII_7321':       r'[O\,{\sc ii}] 7321',
    'OII_7332':       r'[O\,{\sc ii}] 7332',
    'ArIII_7753':     r'[Ar\,{\sc iii}] 7753',
    'P_eta':          r'P$\eta$',
    'SIII_9071':      r'[S\,{\sc iii}] 9071',
    'P_zeta':         r'P$\zeta$',
    'SIII_9533':      r'[S\,{\sc iii}] 9533',
    'P_epsilon':      r'P$\epsilon$',
}

_RANGE_TIE = r'[0.57--1.75] $({\rm H}\alpha|{\rm H}\beta)$'

def _fmt(raw):
    return _LN.get(raw, raw)

rows = []
for ep in elp:
    raw  = ep['name']
    name = _fmt(raw)
    flag = ''
    if raw not in _OLD_MANGA: flag += r'\textsuperscript{*}'
    if raw not in _OLD_SDSS:  flag += r'\textsuperscript{\dag}'
    fi, fp = ep['tf']
    if fi is None:
        ftie = r'\ldots'
    else:
        coeff = fp.strip('=').strip()
        ftie  = (_fmt(idx_to_elp[fi]['name']) if fp == '='
                 else coeff + ' $\\times$ ' + _fmt(idx_to_elp[fi]['name']))
    best_i   = int(_np.argmin(_np.abs(elb_lams - ep['restwave'])))
    eb       = elb[best_i]
    intrange = (r'\ldots' if eb['lo'] == -1.0
                else f"{eb['lo']:.1f}--{eb['hi']:.1f}")
    si, sp = ep['ts']
    if si is None:
        dtie = r'\ldots'
    else:
        dtie = (_RANGE_TIE if sp == '1.75'
                else _fmt(idx_to_elp[si]['name']))
    in_lew = 'Yes' if raw in lew_set else 'No'
    rows.append({
        'Name':              name + flag,
        'Vacuum Wavelength': f"{ep['restwave']:.3f}",
        'Flux Tie':          ftie,
        'Integration Range': intrange,
        'Dispersion Tie':    dtie,
        'In Low-EW Sample':  in_lew,
    })

ll_df = pd.DataFrame(rows)
ll_df.to_csv(os.path.join(OUTPUT_DIR, 'table_line_list.txt'), index=False)
print(f"table_line_list.txt: {len(ll_df)} lines, "
      f"{ll_df['In Low-EW Sample'].eq('Yes').sum()} in low-EW sample.")
print()
print(ll_df.to_latex(index=False, escape=False, column_format='lccccr'))

In [ ]:
# ── Paper Table 6: Recommended Usage Guidelines ──────────────────────────
# Hardcoded summary of the best-practice caveats discussed throughout the
# paper. Written to CSV for archiving; formatted as a LaTeX longtable in
# the paper itself.
usage_rows = [
    ('Wavelength Coverage','3800-10200 Ang obs-frame',
     'Spectrophotometry better than 25% in this range. Errors increase toward edges.'),
    ('Emission Line Fluxes','lambda_obs>4500: S/N>=3; lambda_obs<4500: S/N>=5',
     'Add Table 1 scaling in quadrature. Check fibermasked flag.'),
    ('Emission Line Luminosities','Fiber fraction>=25% for global quantities',
     'Correct aperture bias using INFIBER column from eboss_targettext_v3.fit.'),
    ('High EW Lines','EW_Hbeta<100 Ang or sigma_Hbeta>=100 km/s',
     'Compare [O III] 5007/4959 integrated EWs for clipping. Check Ha/Hb ratio.'),
    ('Emission Line Velocity Dispersions','sigma<450 km/s; apply corrections',
     'Max dispersion 450 km/s. Corrections provided but not applied.'),
    ('Stellar Velocity Dispersion','sigma<450 km/s; S/N>5; EW_Hbeta<15 Ang',
     'High EW causes infilling errors. See Fig A5.'),
    ('Absorption Line Indices','Apply kinematic corrections; not Lick system',
     'See MaNGA-DAP Table 4 for definitions.'),
    ('BPT Diagnostics','All BPT lines S/N>=3; z<0.4963',
     'Ha near high spectrophotometric error at z=0.4963; tying line switches here.'),
    ('Metallicity Indicators','Hbeta/OIII/NII/SII S/N>=5',
     'Correct for reddening; keep lines in 10% spectrophotometric error range. Remove AGN.'),
    ('SFR Indicators','Ha and Hb S/N>=5',
     'Correct for reddening and fiber aperture.'),
    ('AGN Coronal Lines','High-ionization lines S/N>=5',
     'Avoid lambda_obs<4000 Ang.'),
    ('Photometry','z>0.1',
     'Deblending unreliable at low z. Visual inspection often required.'),
    ('Redshift','z>0.1',
     'Deblending problems and strong aperture bias at low z.'),
]
usage_df = pd.DataFrame(usage_rows, columns=['Category','Recommended Regime','Notes'])
usage_df.to_csv(os.path.join(OUTPUT_DIR,'table_recommended_usage.txt'), index=False)
print('table_recommended_usage.txt')
print(usage_df.to_string(index=False))
